In [ ]:
from google.colab import drive
drive.mount('/content/drive')

from pathlib import Path
import pandas as pd

ROOT = Path('/content/drive/MyDrive/Logika Ui')

rows = []
for p in ROOT.rglob('*'):
    if p.is_file():
        rows.append({
            'folder': p.relative_to(ROOT).parts[0],
            'file': p.name,
            'ext': p.suffix.lower(),
            'mb': round(p.stat().st_size / 1e6, 2),
        })

inv = pd.DataFrame(rows)

ringkas = (
    inv.groupby(['folder', 'ext'])
    .agg(jumlah_file=('file', 'count'), total_mb=('mb', 'sum'))
    .round(1)
)
print(ringkas)

Mounted at /content/drive
                                 jumlah_file  total_mb
folder                 ext                            
Data diperlukan.gsheet .gsheet             1       0.0
GADM 2                 .cpg                5       0.0
                       .dbf                5      18.3
                       .prj                5       0.0
                       .shp                5     404.3
                       .shx                5       0.7
Habatusauda_Esai.gdoc  .gdoc               1       0.0
Himawari               .csv               67       6.3
                       .npz                1       1.1
Himawari_nasional      .csv               68      36.1
Himawari_v2.ipynb      .ipynb              1       0.0
IMERG.ipynb            .ipynb              1       0.0
KTI_KARHUTLA           .csv             1310    2403.4
                       .geojson            1      31.9
                       .gsheet             1       0.0
                       .nc             

In [ ]:
!pip -q install netCDF4 rioxarray regionmask

import geopandas as gpd
import xarray as xr

berkas_gadm = [
    p for p in (ROOT / 'GADM 2').rglob('*')
    if p.suffix.lower() in ('.shp', '.gpkg', '.json', '.geojson')
]

gadm2 = gpd.read_file(ROOT / 'GADM 2' / 'gadm41_IDN_2.shp')
print(gadm2[['GID_2', 'NAME_2']].head())
print(len(gadm2))

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.6/10.6 MB 82.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.4/72.4 kB 4.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.3/72.3 kB 4.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 31.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.7/1.7 MB 45.3 MB/s eta 0:00:00
       GID_2           NAME_2
0  IDN.1.2_1       Aceh Barat
1  IDN.1.1_1  Aceh Barat Daya
2  IDN.1.3_1       Aceh Besar
3  IDN.1.4_1        Aceh Jaya
4  IDN.1.5_1     Aceh Selatan
502


In [ ]:
import numpy as np
import pandas as pd
import xarray as xr
import geopandas as gpd

def contoh(folder, ekstensi, n=3):
    berkas = sorted((ROOT / folder).rglob(f'*{ekstensi}'))
    print(f'{folder} {ekstensi}: {len(berkas)} file')
    print('  awal :', [str(p.relative_to(ROOT)) for p in berkas[:n]])
    print('  akhir:', [str(p.relative_to(ROOT)) for p in berkas[-n:]])
    return berkas

for folder in ['Himawari', 'MODIS', 'imerg']:
    berkas = contoh(folder, '.npz')
    z = np.load(berkas[0])
    for k in z.files:
        print('   isi:', k, z[k].shape, z[k].dtype)

for folder in ['Himawari', 'MODIS', 'imerg', 'KTI_KARHUTLA']:
    berkas = contoh(folder, '.csv')
    df = pd.read_csv(berkas[0], nrows=5)
    print('   kolom:', df.columns.tolist())
    print(df.head(3))

nc = contoh('KTI_KARHUTLA', '.nc')
print(xr.open_dataset(nc[0]))

shp = sorted((ROOT / 'GADM 2').glob('*.shp'))
print([p.name for p in shp])
gadm2 = gpd.read_file([p for p in shp if p.stem.endswith('_2')][0])
print(len(gadm2), gadm2.crs)
print(gadm2[['GID_2', 'NAME_2']].head())

Himawari .npz: 1 file
  awal : ['Himawari/prov_mask.npz']
  akhir: ['Himawari/prov_mask.npz']
   isi: mask (427, 280) int8
   isi: lat (427, 280) float32
   isi: lon (427, 280) float32
MODIS .npz: 2090 file
  awal : ['MODIS/potongan/MOD06_L2.A2019152.0300.061.2019152132730.npz', 'MODIS/potongan/MOD06_L2.A2019152.0305.061.2019152132221.npz', 'MODIS/potongan/MOD06_L2.A2019153.0345.061.2019153133423.npz']
  akhir: ['MODIS/potongan/MYD06_L2.A2026241.0800.061.2026243010404.npz', 'MODIS/potongan/MYD06_L2.A2026244.0810.061.2026245000303.npz', 'MODIS/potongan/MYD06_L2.A2026244.0815.061.2026245000639.npz']
   isi: lat (146, 62) float64
   isi: lon (146, 62) float64
   isi: r0 () int64
   isi: c0 () int64
   isi: Cloud_Top_Temperature (146, 62) float64
   isi: Cloud_Top_Height (146, 62) float64
   isi: Cloud_Phase_Infrared (146, 62) int8
   isi: Cloud_Effective_Radius_37_mean (146, 62) float64
   isi: Cloud_Effective_Radius_37_valid (146, 62) float64
   isi: Cloud_Effective_Radius_mean (146, 62)

In [ ]:
import re

imerg = pd.DataFrame(
    [(p.name[0], p.stem[2:]) for p in (ROOT / 'imerg' / 'potongan').glob('*.npz')],
    columns=['run', 'tanggal'],
)
imerg['tanggal'] = pd.to_datetime(imerg['tanggal'])
print(imerg.groupby('run')['tanggal'].agg(['min', 'max', 'count']))
print(pd.crosstab(imerg['tanggal'].dt.year, imerg['tanggal'].dt.month))

pola = re.compile(r'(MOD|MYD)06_L2\.A(\d{4})(\d{3})\.(\d{4})')
modis = []
for p in (ROOT / 'MODIS' / 'potongan').glob('*.npz'):
    m = pola.match(p.name)
    tgl = pd.Timestamp(int(m.group(2)), 1, 1) + pd.Timedelta(days=int(m.group(3)) - 1)
    modis.append((m.group(1), tgl))
modis = pd.DataFrame(modis, columns=['sensor', 'tanggal'])
print(pd.crosstab(modis['tanggal'].dt.year, modis['tanggal'].dt.month))

himawari = pd.concat(
    [pd.read_csv(p) for p in sorted((ROOT / 'Himawari').glob('himawari_*.csv'))],
    ignore_index=True,
)
himawari['tanggal'] = pd.to_datetime(himawari['tanggal'])
print(himawari['provinsi'].unique())
print(himawari.groupby('satelit')['tanggal'].agg(['min', 'max', 'count']))
print(pd.crosstab(himawari['tanggal'].dt.year, himawari['tanggal'].dt.month))

pola_era = re.compile(r'^(.*?)_?(\d{4})_(\d{2})\.nc$')
era = []
for p in (ROOT / 'KTI_KARHUTLA').rglob('*.nc'):
    m = pola_era.match(p.name)
    era.append((p.parent.name, m.group(1), int(m.group(2))))
era = pd.DataFrame(era, columns=['folder', 'jenis', 'tahun'])
print(pd.crosstab([era['folder'], era['jenis']], era['tahun']))

kti = []
for p in (ROOT / 'KTI_KARHUTLA').rglob('*'):
    if p.is_file():
        bagian = p.relative_to(ROOT / 'KTI_KARHUTLA').parts
        kti.append((bagian[0] if len(bagian) > 1 else '(akar)', p.suffix.lower()))
print(pd.DataFrame(kti, columns=['subfolder', 'ext']).value_counts())

gsod = pd.read_csv(ROOT / 'KTI_KARHUTLA' / 'gsod_gabungan_2015_2026.csv')
print(gsod.shape)
print(gsod.columns.tolist())
print(gsod.head(3))

prov = himawari['provinsi'].unique().tolist()
print(gadm2[gadm2['NAME_1'].isin(prov)].groupby('NAME_1').size())

           min        max  count
run                             
F   2020-06-01 2025-09-30    887
L   2025-10-01 2026-10-14    167
tanggal  6   7   8   9   10
tanggal                    
2020     30  31  31  30  31
2021     30  31  31  30  31
2022     30  31  31  30  31
2023     30  31  31  30  31
2024     30  31  31  30  31
2025     30  31  31  30  31
2026     30  31  31  30  14
tanggal   6    7    8    9    10
tanggal                         
2019     106  109  109  104  107
2020     105  105   81  100  107
2021     105  106  105  105  107
2022      50   49   16    0    0
2025      70   73   68   51   57
2026      31   28   27    9    0
['Riau' 'Jambi' 'Sumatera Selatan']
                 min        max  count
satelit                               
himawari8 2020-06-01 2022-10-31   9543
himawari9 2023-08-17 2026-10-14  10395
tanggal   6    7    8    9    10
tanggal                         
2020     630  651  648  609  636
2021     630  648  651  630  651
2022     630  651  648  582 

In [ ]:
import regionmask

PROV = ['Riau', 'Jambi', 'Sumatera Selatan']
studi = gadm2[gadm2['NAME_1'].isin(PROV)].reset_index(drop=True)
regions = regionmask.from_geopandas(studi, names='NAME_2')

berkas_imerg = sorted((ROOT / 'imerg' / 'potongan').glob('*.npz'))
z0 = np.load(berkas_imerg[0])
print(np.nanmin(z0['hujan']), np.nanmax(z0['hujan']))

peta = regions.mask(z0['lon'], z0['lat']).values
kode = np.unique(peta[~np.isnan(peta)]).astype(int)
print(len(studi), len(kode))
print(studi.loc[sorted(set(range(len(studi))) - set(kode)), 'NAME_2'].tolist())

baris = []
for p in berkas_imerg:
    z = np.load(p)
    run, tgl = p.stem.split('_')
    h = z['hujan']
    for k in kode:
        v = h[:, peta == k]
        v = v[~np.isnan(v)]
        if v.size == 0:
            continue
        baris.append({
            'tanggal': tgl,
            'run': run,
            'GID_2': studi.loc[k, 'GID_2'],
            'hujan_rata': v.mean(),
            'hujan_maks': v.max(),
            'frac_hujan': (v >= 0.1).mean(),
        })

imerg_kab = pd.DataFrame(baris)
imerg_kab['tanggal'] = pd.to_datetime(imerg_kab['tanggal'])

HASIL = ROOT / 'KTI_KARHUTLA' / 'hasil'
HASIL.mkdir(exist_ok=True)
imerg_kab.to_csv(HASIL / 'imerg_kab_harian.csv', index=False)

print(imerg_kab.shape)
print(imerg_kab.groupby('run')['hujan_rata'].describe())

0.0 46.96
38 38
[]
(40052, 6)
       count      mean       std  min       25%       50%       75%       max
run                                                                          
F    33706.0  0.215591  0.428141  0.0  0.005417  0.049896  0.225154  6.542349
L     6346.0  0.160865  0.359228  0.0  0.000000  0.022586  0.152509  6.343933


In [ ]:
import numpy as np
import pandas as pd
import xarray as xr

akar = ROOT / 'KTI_KARHUTLA'

katalog = pd.read_csv(ROOT / 'MODIS' / 'granul_katalog_2019_2026.csv')
selesai = {p.stem for p in (ROOT / 'MODIS' / 'potongan').glob('*.npz')}
katalog['stem'] = katalog['file'].str.replace('.hdf', '', regex=False)
belum = katalog[katalog['dalam_jendela'] & ~katalog['stem'].isin(selesai)]
print(katalog['dalam_jendela'].value_counts())
print(len(belum))
print(pd.crosstab(belum['tahun'], belum['bulan']))

z = np.load(ROOT / 'imerg' / 'potongan' / 'F_2020-06-01.npz')
print(z['waktu'])

for nama in ['sl_2020_06.nc', 'plx_2020_06.nc', 'slx_2020_06.nc']:
    d = xr.open_dataset(next(akar.rglob(nama)))
    print(nama, list(d.data_vars), dict(d.sizes))
    print(float(d.latitude.min()), float(d.latitude.max()),
          float(d.longitude.min()), float(d.longitude.max()))

for sub in ['firms', 'firms_gadm']:
    berkas = sorted((akar / sub).glob('*.csv'))
    print(sub, len(berkas), [p.name for p in berkas][:3])
    df = pd.read_csv(berkas[0], nrows=3)
    print(df.columns.tolist())
    print(df.head(2))

lolos = pd.read_csv(akar / 'gsod_stasiun_lolos_per_provinsi.csv')
print(lolos.shape)
print(lolos.head())

dalam_jendela
False    5538
True     2817
Name: count, dtype: int64
1143
bulan  6   7   8   9   10
tahun                    
2019    2   0   0   2   3
2020    0   0   0   0   3
2021    0   0   2   1   1
2022   54  54  81  90  79
2023   82  87  84  83  88
2024   70  67  64  62  66
2025    0   1   3   1   2
2026    2   1   0   5   3
['2020-06-01 03:00' '2020-06-01 03:30' '2020-06-01 04:00'
 '2020-06-01 04:30' '2020-06-01 05:00' '2020-06-01 05:30'
 '2020-06-01 06:00' '2020-06-01 06:30' '2020-06-01 07:00'
 '2020-06-01 07:30' '2020-06-01 08:00' '2020-06-01 08:30']
sl_2020_06.nc ['cape', 't2m', 'd2m'] {'valid_time': 120, 'latitude': 39, 'longitude': 95}
-12.0 7.0 95.0 142.0
plx_2020_06.nc ['w', 'q'] {'valid_time': 120, 'pressure_level': 3, 'latitude': 39, 'longitude': 95}
-12.0 7.0 95.0 142.0
slx_2020_06.nc ['tcwv', 'tclw', 'tcc', 'sp'] {'valid_time': 120, 'latitude': 39, 'longitude': 95}
-12.0 7.0 95.0 142.0
firms 12 ['viirs_2015.csv', 'viirs_2016.csv', 'viirs_2017.csv']
['latitude', 'longi

In [ ]:
import numpy as np
import pandas as pd
import xarray as xr

akar = ROOT / 'KTI_KARHUTLA'
HASIL = akar / 'hasil'

prioritas = belum.assign(urut=belum['tahun'].map({2023: 0, 2024: 1, 2022: 2}).fillna(3))
prioritas = prioritas.sort_values(['urut', 'tanggal', 'jam', 'menit'])
prioritas[['produk', 'tanggal', 'file', 'tahun', 'bulan']].to_csv(
    HASIL / 'modis_belum_diekstrak.csv', index=False)
print(prioritas.groupby('tahun').size())

for nama in ['pl_2020_06.nc', 'sl_2020_06.nc', 'plx_2020_06.nc']:
    d = xr.open_dataset(next(akar.rglob(nama)))
    print(nama, list(d.data_vars), dict(d.sizes))
    print(sorted(set(d.valid_time.dt.hour.values.tolist())))
    if 'pressure_level' in d.coords:
        print(d.pressure_level.values)

lolos = pd.read_csv(akar / 'gsod_stasiun_lolos_per_provinsi.csv')
print(lolos[lolos['NAME_1'].isin(['Riau', 'Jambi', 'Sumatera Selatan'])])

kelengkapan = pd.read_csv(akar / 'gsod_kelengkapan_per_stasiun.csv')
print(kelengkapan.shape)
print(kelengkapan.head())

tahun
2019      7
2020      3
2021      4
2022    358
2023    424
2024    329
2025      7
2026     11
dtype: int64
pl_2020_06.nc ['t', 'r', 'u', 'v'] {'valid_time': 120, 'pressure_level': 3, 'latitude': 39, 'longitude': 95}
[0, 6, 12, 18]
[850. 700. 500.]
sl_2020_06.nc ['cape', 't2m', 'd2m'] {'valid_time': 120, 'latitude': 39, 'longitude': 95}
[0, 6, 12, 18]
plx_2020_06.nc ['w', 'q'] {'valid_time': 120, 'pressure_level': 3, 'latitude': 39, 'longitude': 95}
[0, 6, 12, 18]
[850. 700. 500.]
              NAME_1  2015  2016  2017  2018  2019  2020  2021  2022  2023  \
7              Jambi     1     1     1     1     1     1     1     1     1   
24              Riau     2     2     2     2     2     2     2     2     2   
30  Sumatera Selatan     1     1     1     1     1     1     1     1     1   

    2024  
7      1  
24     2  
30     1  
(1208, 9)
       STATION  tahun  hari_valid  hari_target     persen  lolos           ID  \
0  96001099999   2015         151          153  98.692810  

In [ ]:
gid_studi = set(gadm2.loc[gadm2['NAME_1'].isin(['Riau', 'Jambi', 'Sumatera Selatan']), 'GID_2'])

firms = pd.concat(
    [pd.read_csv(p, usecols=['acq_date', 'confidence', 'frp', 'type', 'GID_2'])
     for p in sorted((akar / 'firms_gadm').glob('viirs_*.csv'))],
    ignore_index=True,
)
firms = firms[
    firms['GID_2'].isin(gid_studi)
    & (firms['type'] == 0)
    & (firms['confidence'].astype(str).str.lower() != 'l')
].copy()
firms['tanggal'] = pd.to_datetime(firms['acq_date'])

titik = (
    firms.groupby(['GID_2', 'tanggal'])
    .agg(titik_panas=('frp', 'size'), frp_total=('frp', 'sum'), frp_maks=('frp', 'max'))
    .reset_index()
)
titik.to_csv(HASIL / 'firms_kab_harian.csv', index=False)

print(titik.shape)
print(titik['tanggal'].dt.year.value_counts().sort_index())
print(titik.describe())

(31617, 5)
tanggal
2015    5699
2016    2868
2017    2286
2018    2983
2019    4409
2020    1927
2021    1693
2022    1610
2023    2902
2024    2302
2025    2064
2026     874
Name: count, dtype: int64
                             tanggal   titik_panas     frp_total      frp_maks
count                          31617  31617.000000  31617.000000  31617.000000
mean   2019-12-09 18:48:39.214346752     14.937122    120.167127     12.944845
min              2015-01-03 00:00:00      1.000000      0.120000      0.120000
25%              2016-09-09 00:00:00      1.000000      2.810000      2.190000
50%              2019-08-10 00:00:00      2.000000      8.550000      5.270000
75%              2023-03-23 00:00:00      6.000000     32.550000     11.440000
max              2026-06-30 00:00:00   4264.000000  52032.590000   1951.300000
std                              NaN     88.766866   1014.363398     34.109454


In [ ]:
folder_him = ROOT / 'Himawari'

him = pd.concat(
    [pd.read_csv(p) for p in sorted(folder_him.glob('himawari_*.csv'))],
    ignore_index=True,
)
him['tanggal'] = pd.to_datetime(him['tanggal'])
him = him.drop_duplicates(['tanggal', 'waktu', 'provinsi'])

kolom_fase = [f'fase{k}' for k in range(6)]
for k in range(6):
    him[f'f{k}'] = him[f'fase{k}'] / him['n_piksel']
valid = him['n_suhu_valid'].where(him['n_suhu_valid'] > 0)
him['lt273'] = him['n_suhu_lt_273'] / valid
him['lt253'] = him['n_suhu_lt_253'] / valid
him['lt233'] = him['n_suhu_lt_233'] / valid

print(him[[f'f{k}' for k in range(6)]].mean().round(3))
print(him[[f'f{k}' for k in range(6)] + ['suhu_mean']].corr()['suhu_mean'].round(2))
print(((him[kolom_fase].sum(axis=1) + him['fase_nan']) / him['n_piksel']).describe())

agg = {
    'satelit': ('satelit', 'first'),
    'him_suhu_mean': ('suhu_mean', 'mean'),
    'him_suhu_p10': ('suhu_p10', 'mean'),
    'him_suhu_min': ('suhu_min', 'min'),
    'him_lt273': ('lt273', 'mean'),
    'him_lt253': ('lt253', 'mean'),
    'him_lt233': ('lt233', 'mean'),
    'him_n': ('suhu_mean', 'size'),
}
for k in range(6):
    agg[f'him_f{k}'] = (f'f{k}', 'mean')

him_harian = him.groupby(['tanggal', 'provinsi']).agg(**agg).reset_index()
him_harian.to_csv(HASIL / 'himawari_prov_harian.csv', index=False)

print(him_harian.shape)
print(him_harian['him_n'].value_counts().head())

log = pd.concat(
    [pd.read_csv(p) for p in sorted(folder_him.glob('log_*.csv'))],
    ignore_index=True,
)
print(log['status'].value_counts())

f0    0.237
f1    0.119
f2    0.007
f3    0.012
f4    0.625
f5    0.000
dtype: float64
f0           0.69
f1           0.78
f2           0.14
f3           0.07
f4          -0.87
f5            NaN
suhu_mean    1.00
Name: suhu_mean, dtype: float64
count    19938.0
mean         1.0
std          0.0
min          1.0
25%          1.0
50%          1.0
75%          1.0
max          1.0
dtype: float64
(2865, 16)
him_n
7    2787
6      69
1       6
3       3
Name: count, dtype: int64
status
ok                6646
hari tidak ada     140
slot tidak ada      60
Name: count, dtype: int64


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

from pathlib import Path
import numpy as np
import pandas as pd
import geopandas as gpd

ROOT = Path('/content/drive/MyDrive/Logika Ui')
akar = ROOT / 'KTI_KARHUTLA'
HASIL = akar / 'hasil'

gadm2 = gpd.read_file(ROOT / 'GADM 2' / 'gadm41_IDN_2.shp')

for p in sorted(HASIL.glob('*')):
    print(p.name, round(p.stat().st_size / 1e6, 2), 'MB')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
era5_kab_harian.csv 67.24 MB
era5_nasional_harian.csv 869.48 MB
firms_kab_harian.csv 1.14 MB
firms_nasional_harian.csv 9.02 MB
himawari_prov_harian.csv 0.7 MB
imerg_kab_harian.csv 2.26 MB
imerg_kab_harian_v2.csv 2.26 MB
imerg_nasional_harian.csv 33.62 MB
modis_belum_diekstrak.csv 0.08 MB
prediksi_kelas_v1.csv 5.57 MB
prediksi_kelas_v2.csv 6.22 MB
prediksi_oof_v1.csv 1.59 MB
tabel_gabungan_nasional.csv 538.14 MB
tabel_gabungan_v1.csv 44.9 MB
tabel_gabungan_v2.csv 48.78 MB


In [ ]:
tabel = pd.read_csv(HASIL / 'tabel_gabungan_v1.csv', parse_dates=['tanggal'])
tabel['hujan'] = ((tabel['hujan_rata'] * 6) >= 1).astype(int)

print(tabel.shape)
cakupan = tabel.groupby('tahun').agg(
    hari=('tanggal', 'nunique'),
    era5=('t2m_06', lambda s: s.notna().mean()),
    himawari=('him_suhu_mean', lambda s: s.notna().mean()),
    firms=('titik_panas', lambda s: s.notna().mean()),
    hari_hujan=('hujan', 'mean'),
).round(2)
print(cakupan)
print(tabel['hujan'].mean().round(3))
print(tabel.groupby('run')['hujan'].mean().round(3))
print(tabel.groupby('bulan')['hujan'].mean().round(3))

dmi = pd.read_csv(akar / 'ONI IOD' / 'dmi.had.long.csv', skiprows=1, names=['tgl', 'dmi'])
dmi['tgl'] = pd.to_datetime(dmi['tgl'])
dmi['dmi'] = dmi['dmi'].where(dmi['dmi'] > -999)
print(dmi.tail(3))

for p in sorted((akar / 'ONI IOD').glob('*.csv')):
    d = pd.read_csv(p)
    print(p.name, d.shape)
    print(d.head(3))

(36328, 78)
       hari  era5  himawari  firms  hari_hujan
tahun                                         
2020    153  1.00      1.00   1.00        0.34
2021    153  1.00      1.00   1.00        0.32
2022    153  1.00      0.99   1.00        0.39
2023     76  1.00      1.00   1.00        0.18
2024    153  1.00      0.99   1.00        0.25
2025    133  1.00      1.00   1.00        0.30
2026    135  0.94      1.00   0.22        0.21
0.295
run
F    0.308
L    0.223
Name: hujan, dtype: float64
bulan
6     0.315
7     0.213
8     0.267
9     0.327
10    0.352
Name: hujan, dtype: float64
            tgl  dmi
1881 2026-10-01  NaN
1882 2026-11-01  NaN
1883 2026-12-01  NaN
dmi.had.long.csv (1884, 2)
         Date  \
0  1870-01-01   
1  1870-02-01   
2  1870-03-01   

    DMI HadISST1.1  missing value -9999 https://psl.noaa.gov/data/timeseries/month/  
0                                             -0.438                                 
1                                             -0.337       

In [ ]:
import numpy as np
import pandas as pd
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, HistGradientBoostingClassifier
from sklearn.metrics import brier_score_loss, log_loss, roc_auc_score

PROV = ['Riau', 'Jambi', 'Sumatera Selatan']
tabel = pd.read_csv(HASIL / 'tabel_gabungan_v1.csv', parse_dates=['tanggal'])
tabel['hujan'] = ((tabel['hujan_rata'] * 6) >= 1).astype(int)

wilayah = gadm2[gadm2['NAME_1'].isin(PROV)]
pusat = wilayah.to_crs('EPSG:6933').geometry.centroid.to_crs('EPSG:4326')
koor = pd.DataFrame({'GID_2': wilayah['GID_2'].values, 'lat': pusat.y.values, 'lon': pusat.x.values})
tabel = tabel.merge(koor, on='GID_2', how='left')

hari = tabel['tanggal'].dt.dayofyear
tabel['doy_sin'] = np.sin(2 * np.pi * hari / 365.25)
tabel['doy_cos'] = np.cos(2 * np.pi * hari / 365.25)

fitur_dasar = ['lat', 'lon', 'doy_sin', 'doy_cos']
fitur_era = [c for c in tabel.columns if c.endswith(('_00', '_06'))]
fitur_him = [c for c in tabel.columns
             if c.startswith('him_') and c not in ('him_n', 'him_f5')]
set_fitur = {
    'A prakiraan': fitur_dasar + fitur_era,
    'B prakiraan+awan': fitur_dasar + fitur_era + fitur_him,
}

def buat_model():
    return {
        'logistik': make_pipeline(SimpleImputer(strategy='median'), StandardScaler(),
                                  LogisticRegression(C=0.1, max_iter=2000)),
        'random forest': make_pipeline(SimpleImputer(strategy='median'),
                                       RandomForestClassifier(n_estimators=200, min_samples_leaf=20,
                                                              max_features='sqrt', n_jobs=-1,
                                                              random_state=1)),
        'gradient boosting': HistGradientBoostingClassifier(max_depth=4, learning_rate=0.05,
                                                            max_iter=250, l2_regularization=1.0,
                                                            early_stopping=False, random_state=1),
    }

def klimatologi(latih, uji):
    peta = latih.groupby(['GID_2', 'bulan'])['hujan'].mean().rename('p').reset_index()
    p = uji[['GID_2', 'bulan']].merge(peta, on=['GID_2', 'bulan'], how='left')['p']
    return p.fillna(latih['hujan'].mean()).to_numpy()

n = len(tabel)
y = tabel['hujan'].to_numpy()
tahun = tabel['tahun'].to_numpy()

oof = {'klimatologi': np.zeros(n)}
for th in np.unique(tahun):
    oof['klimatologi'][tahun == th] = klimatologi(tabel[tahun != th], tabel[tahun == th])

for nama_set, kolom in set_fitur.items():
    for nama, model in buat_model().items():
        kunci = f'{nama_set} | {nama}'
        oof[kunci] = np.zeros(n)
        for th in np.unique(tahun):
            model.fit(tabel.loc[tahun != th, kolom], y[tahun != th])
            oof[kunci][tahun == th] = model.predict_proba(tabel.loc[tahun == th, kolom])[:, 1]
        print('selesai', kunci)

brier_klim = brier_score_loss(y, oof['klimatologi'])
baris = []
for kunci, p in oof.items():
    p = np.clip(p, 1e-6, 1 - 1e-6)
    b = brier_score_loss(y, p)
    baris.append({
        'model': kunci,
        'brier': b,
        'skill_vs_klimatologi': 1 - b / brier_klim,
        'log_loss': log_loss(y, p),
        'auc': roc_auc_score(y, p),
    })
ringkas = pd.DataFrame(baris).set_index('model').round(4).sort_values('brier')
print(ringkas.to_string())

terbaik = [k for k in ringkas.index if k != 'klimatologi'][0]
p = oof[terbaik]
print(terbaik)
print(pd.DataFrame({'tahun': tahun, 'brier_model': (p - y) ** 2,
                    'brier_klim': (oof['klimatologi'] - y) ** 2}).groupby('tahun').mean().round(4))

bin_p = pd.cut(p, bins=np.linspace(0, 1, 11), include_lowest=True)
print(pd.DataFrame({'bin': bin_p, 'prediksi': p, 'amatan': y}).groupby('bin', observed=True).agg(
    jumlah=('amatan', 'size'), prediksi=('prediksi', 'mean'), amatan=('amatan', 'mean')).round(3))

tabel[['tanggal', 'GID_2', 'hujan']].assign(p_terbaik=p).to_csv(HASIL / 'prediksi_oof_v1.csv', index=False)

selesai A prakiraan | logistik
selesai A prakiraan | random forest
selesai A prakiraan | gradient boosting
selesai B prakiraan+awan | logistik
selesai B prakiraan+awan | random forest
selesai B prakiraan+awan | gradient boosting
                                       brier  skill_vs_klimatologi  log_loss     auc
model                                                                               
B prakiraan+awan | gradient boosting  0.1323                0.3595    0.4077  0.8654
B prakiraan+awan | random forest      0.1364                0.3401    0.4196  0.8595
B prakiraan+awan | logistik           0.1408                0.3188    0.4326  0.8461
A prakiraan | gradient boosting       0.1413                0.3164    0.4328  0.8451
A prakiraan | random forest           0.1435                0.3054    0.4398  0.8421
A prakiraan | logistik                0.1451                0.2978    0.4446  0.8356
klimatologi                           0.2066                0.0000    0.6036  0.5702
B prak

In [ ]:
import numpy as np
import pandas as pd
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.metrics import brier_score_loss, log_loss, roc_auc_score
from sklearn.inspection import permutation_importance

PROV = ['Riau', 'Jambi', 'Sumatera Selatan']
tabel = pd.read_csv(HASIL / 'tabel_gabungan_v1.csv', parse_dates=['tanggal'])
tabel['hujan'] = ((tabel['hujan_rata'] * 6) >= 1).astype(int)

wilayah = gadm2[gadm2['NAME_1'].isin(PROV)]
pusat = wilayah.to_crs('EPSG:6933').geometry.centroid.to_crs('EPSG:4326')
koor = pd.DataFrame({'GID_2': wilayah['GID_2'].values, 'lat': pusat.y.values, 'lon': pusat.x.values})
tabel = tabel.merge(koor, on='GID_2', how='left')

hari = tabel['tanggal'].dt.dayofyear
tabel['doy_sin'] = np.sin(2 * np.pi * hari / 365.25)
tabel['doy_cos'] = np.cos(2 * np.pi * hari / 365.25)

fitur_dasar = ['lat', 'lon', 'doy_sin', 'doy_cos']
fitur_era = [c for c in tabel.columns if c.endswith(('_00', '_06'))]
fitur_era_00 = [c for c in fitur_era if c.endswith('_00')]
fitur_him = [c for c in tabel.columns
             if c.startswith('him_') and c not in ('him_n', 'him_f5')]
set_fitur = {
    'A 00+06 UTC': fitur_dasar + fitur_era,
    'B 00+06 UTC + awan': fitur_dasar + fitur_era + fitur_him,
    'C 00 UTC saja': fitur_dasar + fitur_era_00,
    'D 00 UTC + awan': fitur_dasar + fitur_era_00 + fitur_him,
}

def model_gb():
    return HistGradientBoostingClassifier(max_depth=4, learning_rate=0.05, max_iter=250,
                                          l2_regularization=1.0, early_stopping=False,
                                          random_state=1)

def klimatologi(latih, uji):
    peta = latih.groupby(['GID_2', 'bulan'])['hujan'].mean().rename('p').reset_index()
    p = uji[['GID_2', 'bulan']].merge(peta, on=['GID_2', 'bulan'], how='left')['p']
    return p.fillna(latih['hujan'].mean()).to_numpy()

n = len(tabel)
y = tabel['hujan'].to_numpy()
tahun = tabel['tahun'].to_numpy()

oof = {'klimatologi': np.zeros(n)}
for th in np.unique(tahun):
    oof['klimatologi'][tahun == th] = klimatologi(tabel[tahun != th], tabel[tahun == th])

for nama_set, kolom in set_fitur.items():
    oof[nama_set] = np.zeros(n)
    for th in np.unique(tahun):
        m = model_gb()
        m.fit(tabel.loc[tahun != th, kolom], y[tahun != th])
        oof[nama_set][tahun == th] = m.predict_proba(tabel.loc[tahun == th, kolom])[:, 1]
    print('selesai', nama_set)

brier_klim = brier_score_loss(y, oof['klimatologi'])
baris = []
for kunci, p in oof.items():
    p = np.clip(p, 1e-6, 1 - 1e-6)
    b = brier_score_loss(y, p)
    baris.append({'model': kunci, 'brier': b, 'skill_vs_klimatologi': 1 - b / brier_klim,
                  'log_loss': log_loss(y, p), 'auc': roc_auc_score(y, p)})
print(pd.DataFrame(baris).set_index('model').round(4).sort_values('brier').to_string())

th = 2024
kolom = set_fitur['B 00+06 UTC + awan']
m = model_gb()
m.fit(tabel.loc[tahun != th, kolom], y[tahun != th])
pi = permutation_importance(m, tabel.loc[tahun == th, kolom], y[tahun == th],
                            scoring='neg_brier_score', n_repeats=5, random_state=1)
print(pd.Series(pi.importances_mean, index=kolom).sort_values(ascending=False).head(15).round(4))

print(tabel['hujan_maks'].describe().round(2))
curah = tabel['hujan_rata'] * 6
for ambang in [2, 5, 10]:
    kelas = np.where(curah >= 1, 'C2 hujan merata',
                     np.where(tabel['hujan_maks'] >= ambang, 'C1 konvektif lokal', 'C0 tidak hujan'))
    print('ambang', ambang, pd.Series(kelas).value_counts(normalize=True).round(3).to_dict())
kelas5 = np.where(curah >= 1, 'C2 hujan merata',
                  np.where(tabel['hujan_maks'] >= 5, 'C1 konvektif lokal', 'C0 tidak hujan'))
print(pd.crosstab(tabel['tahun'], kelas5, normalize='index').round(3))

selesai A 00+06 UTC
selesai B 00+06 UTC + awan
selesai C 00 UTC saja
selesai D 00 UTC + awan
                     brier  skill_vs_klimatologi  log_loss     auc
model                                                             
B 00+06 UTC + awan  0.1323                0.3595    0.4077  0.8654
D 00 UTC + awan     0.1405                0.3199    0.4294  0.8476
A 00+06 UTC         0.1413                0.3164    0.4328  0.8451
C 00 UTC saja       0.1610                0.2210    0.4870  0.7932
klimatologi         0.2066                0.0000    0.6036  0.5702
him_f0          0.0283
tcwv_06         0.0070
cape_06         0.0067
him_suhu_p10    0.0024
t2m_06          0.0024
him_f3          0.0015
u_850_06        0.0010
u_700_06        0.0010
w_500_06        0.0010
him_suhu_min    0.0009
u_850_00        0.0008
r_700_06        0.0008
t2m_00          0.0007
r_500_06        0.0006
him_f2          0.0006
dtype: float64
count    36328.00
mean         3.44
std          4.74
min          0.00
25%   

In [ ]:
import numpy as np
import pandas as pd
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.metrics import brier_score_loss, log_loss, roc_auc_score

PROV = ['Riau', 'Jambi', 'Sumatera Selatan']
TAU = 0.05

tabel = pd.read_csv(HASIL / 'tabel_gabungan_v1.csv', parse_dates=['tanggal'])

wilayah = gadm2[gadm2['NAME_1'].isin(PROV)]
pusat = wilayah.to_crs('EPSG:6933').geometry.centroid.to_crs('EPSG:4326')
koor = pd.DataFrame({'GID_2': wilayah['GID_2'].values, 'lat': pusat.y.values, 'lon': pusat.x.values})
nama = wilayah[['GID_2', 'NAME_2']]
tabel = tabel.drop(columns=['NAME_2'], errors='ignore').merge(nama, on='GID_2', how='left')
tabel = tabel.merge(koor, on='GID_2', how='left')

hari = tabel['tanggal'].dt.dayofyear
tabel['doy_sin'] = np.sin(2 * np.pi * hari / 365.25)
tabel['doy_cos'] = np.cos(2 * np.pi * hari / 365.25)

curah = tabel['hujan_rata'] * 6
for tau in [0.02, 0.05, 0.1, 0.2]:
    k = np.where(curah >= 1, 2, np.where(tabel['frac_hujan'] >= tau, 1, 0))
    print('tau', tau, pd.Series(k).value_counts(normalize=True).sort_index().round(3).to_dict())

tabel['kelas'] = np.where(curah >= 1, 2, np.where(tabel['frac_hujan'] >= TAU, 1, 0))
print(pd.crosstab(tabel['tahun'], tabel['kelas'], normalize='index').round(3))
print(pd.crosstab(tabel['bulan'], tabel['kelas'], normalize='index').round(3))

fitur_dasar = ['lat', 'lon', 'doy_sin', 'doy_cos']
fitur_era = [c for c in tabel.columns if c.endswith(('_00', '_06'))]
fitur_era_00 = [c for c in fitur_era if c.endswith('_00')]
fitur_him = [c for c in tabel.columns
             if c.startswith('him_') and c not in ('him_n', 'him_f5')]
set_fitur = {
    'C prakiraan (00 UTC)': fitur_dasar + fitur_era_00,
    'D pemantauan (00 UTC + awan)': fitur_dasar + fitur_era_00 + fitur_him,
}

def model_gb():
    return HistGradientBoostingClassifier(max_depth=4, learning_rate=0.05, max_iter=250,
                                          l2_regularization=1.0, early_stopping=False,
                                          random_state=1)

def klim_kelas(latih, uji):
    peta = (pd.get_dummies(latih['kelas']).astype(float)
            .assign(GID_2=latih['GID_2'].values, bulan=latih['bulan'].values)
            .groupby(['GID_2', 'bulan']).mean().reset_index())
    P = uji[['GID_2', 'bulan']].merge(peta, on=['GID_2', 'bulan'], how='left')[[0, 1, 2]]
    rata = latih['kelas'].value_counts(normalize=True).sort_index()
    return P.fillna(pd.Series(rata.to_numpy(), index=[0, 1, 2])).to_numpy()

n = len(tabel)
kelas = tabel['kelas'].to_numpy()
Y = np.eye(3)[kelas]
tahun = tabel['tahun'].to_numpy()

prob = {'klimatologi': np.zeros((n, 3))}
for th in np.unique(tahun):
    prob['klimatologi'][tahun == th] = klim_kelas(tabel[tahun != th], tabel[tahun == th])

for nama_set, kolom in set_fitur.items():
    prob[nama_set] = np.zeros((n, 3))
    for th in np.unique(tahun):
        m = model_gb()
        m.fit(tabel.loc[tahun != th, kolom], kelas[tahun != th])
        prob[nama_set][tahun == th] = m.predict_proba(tabel.loc[tahun == th, kolom])
    print('selesai', nama_set)

def skor(P):
    P = np.clip(P, 1e-6, 1)
    P = P / P.sum(axis=1, keepdims=True)
    return {'brier_3kelas': ((P - Y) ** 2).sum(axis=1).mean(),
            'log_loss': log_loss(kelas, P, labels=[0, 1, 2]),
            'brier_K1': brier_score_loss(kelas == 1, P[:, 1]),
            'auc_K1': roc_auc_score(kelas == 1, P[:, 1])}

ringkas = pd.DataFrame({k: skor(P) for k, P in prob.items()}).T
ringkas['skill_3kelas'] = 1 - ringkas['brier_3kelas'] / ringkas.loc['klimatologi', 'brier_3kelas']
print(ringkas.round(4).to_string())

for nama_set in set_fitur:
    p1 = prob[nama_set][:, 1]
    grup = pd.qcut(p1, 10, duplicates='drop')
    print(nama_set)
    print(pd.DataFrame({'g': grup, 'prediksi': p1, 'amatan': (kelas == 1).astype(int)})
          .groupby('g', observed=True)
          .agg(jumlah=('amatan', 'size'), prediksi=('prediksi', 'mean'), amatan=('amatan', 'mean'))
          .round(3))

keluar = tabel[['tanggal', 'GID_2', 'NAME_2', 'kelas']].copy()
for kode, nama_set in [('C', 'C prakiraan (00 UTC)'), ('D', 'D pemantauan (00 UTC + awan)')]:
    for k in range(3):
        keluar[f'p{k}_{kode}'] = prob[nama_set][:, k]
keluar.to_csv(HASIL / 'prediksi_kelas_v1.csv', index=False)
print(keluar.groupby('NAME_2')[['p1_C', 'p1_D']].mean().sort_values('p1_D').tail(8).round(3))

tau 0.02 {0: 0.3, 1: 0.406, 2: 0.295}
tau 0.05 {0: 0.413, 1: 0.292, 2: 0.295}
tau 0.1 {0: 0.54, 1: 0.165, 2: 0.295}
tau 0.2 {0: 0.663, 1: 0.042, 2: 0.295}
kelas      0      1      2
tahun                     
2020   0.352  0.307  0.341
2021   0.370  0.307  0.323
2022   0.261  0.349  0.390
2023   0.586  0.233  0.181
2024   0.463  0.283  0.254
2025   0.429  0.273  0.298
2026   0.535  0.256  0.209
kelas      0      1      2
bulan                     
6      0.364  0.320  0.315
7      0.525  0.262  0.213
8      0.478  0.255  0.267
9      0.383  0.290  0.327
10     0.309  0.339  0.352
selesai C prakiraan (00 UTC)
selesai D pemantauan (00 UTC + awan)
                              brier_3kelas  log_loss  brier_K1  auc_K1  skill_3kelas
klimatologi                         0.6508    1.0765    0.2051  0.5665        0.0000
C prakiraan (00 UTC)                0.5388    0.9079    0.1991  0.6297        0.1721
D pemantauan (00 UTC + awan)        0.4876    0.8190    0.1930  0.6698        0.2508
C praki

In [ ]:
import pandas as pd

HASIL = ROOT / 'KTI_KARHUTLA' / 'hasil'
im = pd.read_csv(HASIL / 'imerg_kab_harian.csv', parse_dates=['tanggal'])
era = pd.read_csv(HASIL / 'era5_kab_harian.csv', usecols=['tanggal'], parse_dates=['tanggal'])
him = pd.read_csv(HASIL / 'himawari_prov_harian.csv', parse_dates=['tanggal'])

def rentang_hilang(ada, tahun):
    semua = pd.date_range(f'{tahun}-06-01', f'{tahun}-10-31')
    hilang = semua[~semua.isin(ada)]
    if len(hilang) == 0:
        return []
    s = hilang.to_series()
    grup = (s.diff().dt.days != 1).cumsum().values
    return [(g.iloc[0].date(), g.iloc[-1].date(), len(g)) for _, g in s.groupby(grup)]

for nama, d in [('imerg', im), ('era5', era), ('himawari', him)]:
    ada = set(d['tanggal'].unique())
    print(nama)
    for th in range(2020, 2027):
        r = rentang_hilang(ada, th)
        print(' ', th, r if r else 'lengkap')

okt = im[im['tanggal'].dt.month == 10]
print(okt.groupby([okt['tanggal'].dt.year, 'run'])['hujan_rata'].agg(['mean', 'count']).round(3))

imerg
  2020 lengkap
  2021 lengkap
  2022 lengkap
  2023 lengkap
  2024 lengkap
  2025 lengkap
  2026 [(datetime.date(2026, 10, 15), datetime.date(2026, 10, 31), 17)]
era5
  2020 lengkap
  2021 lengkap
  2022 lengkap
  2023 lengkap
  2024 lengkap
  2025 lengkap
  2026 [(datetime.date(2026, 10, 6), datetime.date(2026, 10, 31), 26)]
himawari
  2020 lengkap
  2021 lengkap
  2022 [(datetime.date(2022, 9, 10), datetime.date(2022, 9, 10), 1)]
  2023 [(datetime.date(2023, 6, 1), datetime.date(2023, 8, 16), 77)]
  2024 [(datetime.date(2024, 10, 12), datetime.date(2024, 10, 12), 1)]
  2025 [(datetime.date(2025, 10, 12), datetime.date(2025, 10, 31), 20)]
  2026 [(datetime.date(2026, 10, 15), datetime.date(2026, 10, 31), 17)]
              mean  count
tanggal run              
2020    F    0.281   1178
2021    F    0.285   1178
2022    F    0.293   1178
2023    F    0.137   1178
2024    F    0.250   1178
2025    L    0.259   1178
2026    L    0.111    532


In [ ]:
folder_him = ROOT / 'Himawari'
log = pd.concat([pd.read_csv(p) for p in sorted(folder_him.glob('log_*.csv'))], ignore_index=True)
print(log.columns.tolist())
print(log['status'].value_counts())

kol = [c for c in log.columns if any(k in c.lower() for k in ('tanggal', 'tgl', 'waktu', 'date'))]
print(kol)
if kol:
    t = pd.to_datetime(log[kol[0]], errors='coerce')
    gap = log[(t >= '2023-06-01') & (t <= '2023-08-16')]
    print(len(gap))
    print(gap['status'].value_counts())

for f in ['imerg', 'Himawari', 'MODIS']:
    print(f, [p.name for p in (ROOT / f).iterdir() if p.is_file()][:20])

['satelit', 'tanggal', 'waktu', 'status', 'detik']
status
ok                6646
hari tidak ada     140
slot tidak ada      60
Name: count, dtype: int64
['tanggal', 'waktu']
7
status
slot tidak ada    7
Name: count, dtype: int64
imerg ['katalog_imerg.csv']
Himawari ['prov_mask.npz', 'ekstraksi_himawari.csv', 'log_2020_06.csv', 'himawari_2020_06.csv', 'log_2020_07.csv', 'himawari_2020_07.csv', 'log_2020_08.csv', 'himawari_2020_08.csv', 'log_2020_09.csv', 'himawari_2020_09.csv', 'log_2020_10.csv', 'himawari_2020_10.csv', 'log_2021_06.csv', 'himawari_2021_06.csv', 'log_2021_07.csv', 'himawari_2021_07.csv', 'log_2021_08.csv', 'himawari_2021_08.csv', 'log_2021_09.csv', 'himawari_2021_09.csv']
MODIS ['granul_katalog_2019_2026.csv', 'granul_katalog_2020_2024.csv']


In [ ]:
print(log.groupby('status')['detik'].describe().round(1))

print([p.name for p in sorted(folder_him.glob('log_2023_*.csv'))])
print([p.name for p in sorted(folder_him.glob('himawari_2023_*.csv'))])

log['tanggal'] = pd.to_datetime(log['tanggal'])
ct = pd.crosstab([log['satelit'], log['tanggal'].dt.to_period('M')], log['status'])
print(ct)

print(pd.read_csv(folder_him / 'ekstraksi_himawari.csv').head(10))

                 count  mean  std  min   25%   50%   75%   max
status                                                        
hari tidak ada   140.0   0.0  0.0  0.0   0.0   0.0   0.0   0.0
ok              6646.0  16.1  6.6  5.1  11.5  15.2  19.3  99.5
slot tidak ada    60.0   0.0  0.0  0.0   0.0   0.0   0.0   0.0
['log_2023_08.csv', 'log_2023_09.csv', 'log_2023_10.csv']
['himawari_2023_08.csv', 'himawari_2023_09.csv', 'himawari_2023_10.csv']
status             hari tidak ada   ok  slot tidak ada
satelit   tanggal                                     
himawari8 2020-06               0  210               0
          2020-07               0  217               0
          2020-08               0  216               1
          2020-09               0  203               7
          2020-10               0  212               5
          2021-06               0  210               0
          2021-07               0  216               1
          2021-08               0  217               0
    

In [ ]:
!pip -q install s3fs
import s3fs
fs = s3fs.S3FileSystem(anon=True)

for bucket in ['noaa-himawari8', 'noaa-himawari9']:
    produk = [p.split('/')[-1] for p in fs.ls(bucket)]
    print(bucket, produk)
    for pr in produk:
        for tgl in ['2022/12/20', '2023/06/01', '2023/07/15', '2023/08/10', '2025/10/15']:
            try:
                n = len(fs.ls(f'{bucket}/{pr}/{tgl}'))
            except Exception as e:
                n = 'ERR ' + type(e).__name__
            print(' ', pr, tgl, n)

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 102.6/102.6 kB 6.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 221.7/221.7 kB 13.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 16.0/16.0 MB 42.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 336.2/336.2 kB 22.0 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
gcsfs 2025.12.0 requires fsspec==2025.12.0, but you have fsspec 2026.9.0 which is incompatible.
datasets 4.8.5 requires fsspec[http]<=2026.2.0,>=2023.1.0, but you have fsspec 2026.9.0 which is incompatible.
noaa-himawari8 ['AHI-L1b-FLDK', 'AHI-L1b-Japan', 'AHI-L1b-Target', 'AHI-L2-FLDK-Clouds', 'AHI-L2-FLDK-ISatSS', 'AHI-L2-FLDK-RainfallRate', 'AHI-L2-FLDK-SST', 'AHI-L2-FLDK-Winds', 'README.txt', 'index.html']
  AHI-L1b-FLDK 2022/12/20 ERR FileNotFoundError
  AHI-L1b-FLDK 2023/06/01 ERR FileNotFound

In [ ]:
def ada(bucket, produk, tgl):
    try:
        return len(fs.ls(f'{bucket}/{produk}/{tgl:%Y/%m/%d}'))
    except FileNotFoundError:
        return 0

for tgl in pd.date_range('2023-08-12', '2023-08-18'):
    print(tgl.date(), 'h9 Clouds', ada('noaa-himawari9', 'AHI-L2-FLDK-Clouds', tgl))

for tgl in pd.date_range('2025-10-09', '2025-11-03'):
    print(tgl.date(), 'h8', ada('noaa-himawari8', 'AHI-L2-FLDK-Clouds', tgl),
          'h9', ada('noaa-himawari9', 'AHI-L2-FLDK-Clouds', tgl))

2023-08-12 h9 Clouds 0
2023-08-13 h9 Clouds 0
2023-08-14 h9 Clouds 0
2023-08-15 h9 Clouds 0
2023-08-16 h9 Clouds 56
2023-08-17 h9 Clouds 140
2023-08-18 h9 Clouds 142
2025-10-09 h8 0 h9 141
2025-10-10 h8 0 h9 141
2025-10-11 h8 0 h9 91
2025-10-12 h8 41 h9 0
2025-10-13 h8 141 h9 0
2025-10-14 h8 142 h9 0
2025-10-15 h8 142 h9 0
2025-10-16 h8 140 h9 0
2025-10-17 h8 142 h9 0
2025-10-18 h8 142 h9 0
2025-10-19 h8 142 h9 0
2025-10-20 h8 142 h9 0
2025-10-21 h8 142 h9 0
2025-10-22 h8 142 h9 0
2025-10-23 h8 142 h9 0
2025-10-24 h8 142 h9 0
2025-10-25 h8 142 h9 0
2025-10-26 h8 142 h9 0
2025-10-27 h8 141 h9 0
2025-10-28 h8 142 h9 0
2025-10-29 h8 142 h9 0
2025-10-30 h8 141 h9 0
2025-10-31 h8 141 h9 0
2025-11-01 h8 142 h9 0
2025-11-02 h8 142 h9 0
2025-11-03 h8 142 h9 0


In [ ]:
for b, tgl in [('noaa-himawari9', '2023-08-16'),
               ('noaa-himawari9', '2025-10-11'),
               ('noaa-himawari8', '2025-10-12')]:
    d = pd.Timestamp(tgl)
    nama = sorted(p.rstrip('/').split('/')[-1]
                  for p in fs.ls(f'{b}/AHI-L2-FLDK-Clouds/{d:%Y/%m/%d}'))
    print(b, tgl, len(nama), nama[:2], nama[-2:])

print(sorted(pd.read_csv(folder_him / 'himawari_2025_10.csv')['waktu'].unique()))

noaa-himawari9 2023-08-16 56 ['1430', '1450'] ['2340', '2350']
noaa-himawari9 2025-10-11 91 ['0000', '0010'] ['1510', '1520']
noaa-himawari8 2025-10-12 41 ['1710', '1720'] ['2340', '2350']
[np.int64(300), np.int64(400), np.int64(500), np.int64(600), np.int64(700), np.int64(800), np.int64(900)]


In [ ]:
kat = pd.read_csv(ROOT / 'imerg' / 'katalog_imerg.csv')
print(kat.shape, kat.columns.tolist())
print(kat.head(3))
print(kat.tail(3))

(12648, 5) ['run', 'tanggal', 'slot', 'file', 'url']
  run     tanggal  slot                                               file  \
0   F  2020-06-01   300  3B-HHR.MS.MRG.3IMERG.20200601-S030000-E032959....   
1   F  2020-06-01   330  3B-HHR.MS.MRG.3IMERG.20200601-S033000-E035959....   
2   F  2020-06-01   400  3B-HHR.MS.MRG.3IMERG.20200601-S040000-E042959....   

                                                 url  
0  https://data.gesdisc.earthdata.nasa.gov/data/G...  
1  https://data.gesdisc.earthdata.nasa.gov/data/G...  
2  https://data.gesdisc.earthdata.nasa.gov/data/G...  
      run     tanggal  slot  \
12645   F  2023-07-31   730   
12646   F  2023-07-31   800   
12647   F  2023-07-31   830   

                                                    file  \
12645  3B-HHR.MS.MRG.3IMERG.20230731-S073000-E075959....   
12646  3B-HHR.MS.MRG.3IMERG.20230731-S080000-E082959....   
12647  3B-HHR.MS.MRG.3IMERG.20230731-S083000-E085959....   

                                                

In [ ]:
kat = pd.read_csv(ROOT / 'imerg' / 'katalog_imerg.csv', parse_dates=['tanggal'])
per_hari = kat.groupby(['tanggal', 'run']).size().rename('slot').reset_index()
print(per_hari['slot'].value_counts())
print(per_hari.groupby('run')['tanggal'].agg(['min', 'max', 'count']))

ada_npz = {pd.Timestamp(p.stem[2:]) for p in (ROOT / 'imerg' / 'potongan').glob('*.npz')}
hari_kat = set(per_hari['tanggal'])
batas = per_hari['tanggal'].max()
semua = {d for th in range(2020, 2027) for d in pd.date_range(f'{th}-06-01', f'{th}-10-31') if d <= batas}

def rentang(tgls):
    s = pd.Series(sorted(tgls))
    if s.empty:
        return []
    g = (s.diff().dt.days != 1).cumsum()
    return [(a.iloc[0].date(), a.iloc[-1].date(), len(a)) for _, a in s.groupby(g)]

print('ada di katalog, belum jadi npz:', rentang(hari_kat - ada_npz))
print('tidak ada di katalog:', rentang(semua - hari_kat))

slot
12    1054
Name: count, dtype: int64
           min        max  count
run                             
F   2020-06-01 2025-09-30    887
L   2025-10-01 2026-10-14    167
ada di katalog, belum jadi npz: []
tidak ada di katalog: []


In [ ]:
t = pd.read_csv(HASIL / 'tabel_gabungan_v1.csv', parse_dates=['tanggal'])
print(t.shape)
print(t.columns.tolist())
print(t.isna().mean().round(3)[lambda s: s > 0].to_string())
print(t.iloc[0].to_string())

(36328, 77)
['tanggal', 'run', 'GID_2', 'hujan_rata', 'hujan_maks', 'frac_hujan', 'NAME_1', 'NAME_2', 't_850_00', 't_850_06', 't_700_00', 't_700_06', 't_500_00', 't_500_06', 'r_850_00', 'r_850_06', 'r_700_00', 'r_700_06', 'r_500_00', 'r_500_06', 'u_850_00', 'u_850_06', 'u_700_00', 'u_700_06', 'u_500_00', 'u_500_06', 'v_850_00', 'v_850_06', 'v_700_00', 'v_700_06', 'v_500_00', 'v_500_06', 'cape_00', 'cape_06', 't2m_00', 't2m_06', 'd2m_00', 'd2m_06', 'w_850_00', 'w_850_06', 'w_700_00', 'w_700_06', 'w_500_00', 'w_500_06', 'q_850_00', 'q_850_06', 'q_700_00', 'q_700_06', 'q_500_00', 'q_500_06', 'tcwv_00', 'tcwv_06', 'tclw_00', 'tclw_06', 'tcc_00', 'tcc_06', 'sp_00', 'sp_06', 'satelit', 'him_suhu_mean', 'him_suhu_p10', 'him_suhu_min', 'him_lt273', 'him_lt253', 'him_lt233', 'him_n', 'him_f0', 'him_f1', 'him_f2', 'him_f3', 'him_f4', 'him_f5', 'titik_panas', 'frp_total', 'frp_maks', 'tahun', 'bulan']
t_850_00         0.008
t_850_06         0.008
t_700_00         0.008
t_700_06         0.008
t_50

In [ ]:
v1 = pd.read_csv(HASIL / 'tabel_gabungan_v1.csv', parse_dates=['tanggal'])
urutan = v1.columns.tolist()
kolom_era = [c for c in urutan if c.endswith(('_00', '_06'))]

imerg = pd.read_csv(HASIL / 'imerg_kab_harian.csv', parse_dates=['tanggal'])
imerg = imerg.sort_values('run').drop_duplicates(['tanggal', 'GID_2'], keep='first')
era = pd.read_csv(HASIL / 'era5_kab_harian.csv', parse_dates=['tanggal'])
him = pd.read_csv(HASIL / 'himawari_prov_harian.csv', parse_dates=['tanggal'])
fir = pd.read_csv(HASIL / 'firms_kab_harian.csv', parse_dates=['tanggal'])

nama = gadm2[['GID_2', 'NAME_1', 'NAME_2']]
t = (imerg.merge(nama, on='GID_2', how='left')
          .merge(era[['tanggal', 'GID_2'] + kolom_era], on=['tanggal', 'GID_2'], how='left')
          .merge(him.rename(columns={'provinsi': 'NAME_1'}), on=['tanggal', 'NAME_1'], how='left')
          .merge(fir, on=['tanggal', 'GID_2'], how='left'))

batas = fir['tanggal'].max()
for c in ['titik_panas', 'frp_total', 'frp_maks']:
    t.loc[t['tanggal'] <= batas, c] = t.loc[t['tanggal'] <= batas, c].fillna(0)

t['tahun'] = t['tanggal'].dt.year
t['bulan'] = t['tanggal'].dt.month
t = t[urutan]

a = v1.set_index(['tanggal', 'GID_2']).sort_index()
b = t.set_index(['tanggal', 'GID_2']).sort_index()
ada = a.index.intersection(b.index)
print('baris v1:', len(a), '| baris rekonstruksi:', len(b), '| sama kuncinya:', len(ada))

beda = []
for c in a.columns:
    x, y = a.loc[ada, c], b.loc[ada, c]
    if x.dtype.kind in 'fi':
        ok = np.allclose(x.to_numpy(float), y.to_numpy(float), equal_nan=True, rtol=1e-6, atol=1e-9)
    else:
        ok = (x.fillna('~').astype(str) == y.fillna('~').astype(str)).all()
    if not ok:
        beda.append(c)
print('kolom yang beda:', beda)

baris v1: 36328 | baris rekonstruksi: 40052 | sama kuncinya: 36328
kolom yang beda: []


In [ ]:
from google.colab import drive
drive.mount('/content/drive')
!pip -q install regionmask
from pathlib import Path
import numpy as np, pandas as pd, geopandas as gpd, regionmask

ROOT = Path('/content/drive/MyDrive/Logika Ui')
HASIL = ROOT / 'KTI_KARHUTLA' / 'hasil'
gadm2 = gpd.read_file(ROOT / 'GADM 2' / 'gadm41_IDN_2.shp')

PROV = ['Riau', 'Jambi', 'Sumatera Selatan']
studi = gadm2[gadm2['NAME_1'].isin(PROV)].reset_index(drop=True)
regions = regionmask.from_geopandas(studi, names='NAME_2')

berkas = sorted((ROOT / 'imerg' / 'potongan').glob('*.npz'))
print(len(berkas), 'file npz (harapan 1054)')

z0 = np.load(berkas[0])
peta = regions.mask(z0['lon'], z0['lat']).values
kode = np.unique(peta[~np.isnan(peta)]).astype(int)

baris = []
for p in berkas:
    z = np.load(p)
    run, tgl = p.stem.split('_')
    h = z['hujan']
    for k in kode:
        v = h[:, peta == k]
        v = v[~np.isnan(v)]
        if v.size == 0:
            continue
        baris.append({'tanggal': tgl, 'run': run, 'GID_2': studi.loc[k, 'GID_2'],
                      'hujan_rata': v.mean(), 'hujan_maks': v.max(),
                      'frac_hujan': (v >= 0.1).mean()})

imerg = pd.DataFrame(baris)
imerg['tanggal'] = pd.to_datetime(imerg['tanggal'])
imerg = (imerg.sort_values('run')
              .drop_duplicates(['tanggal', 'GID_2'], keep='first')
              .reset_index(drop=True))
imerg.to_csv(HASIL / 'imerg_kab_harian_v2.csv', index=False)

v1 = pd.read_csv(HASIL / 'tabel_gabungan_v1.csv', parse_dates=['tanggal'])
urutan = v1.columns.tolist()
kolom_era = [c for c in urutan if c.endswith(('_00', '_06'))]

era = pd.read_csv(HASIL / 'era5_kab_harian.csv', parse_dates=['tanggal'])
him = pd.read_csv(HASIL / 'himawari_prov_harian.csv', parse_dates=['tanggal'])
fir = pd.read_csv(HASIL / 'firms_kab_harian.csv', parse_dates=['tanggal'])

nama = gadm2[['GID_2', 'NAME_1', 'NAME_2']]
t = (imerg.merge(nama, on='GID_2', how='left')
          .merge(era[['tanggal', 'GID_2'] + kolom_era], on=['tanggal', 'GID_2'], how='left')
          .merge(him.rename(columns={'provinsi': 'NAME_1'}), on=['tanggal', 'NAME_1'], how='left')
          .merge(fir, on=['tanggal', 'GID_2'], how='left'))

batas = fir['tanggal'].max()
for c in ['titik_panas', 'frp_total', 'frp_maks']:
    t.loc[t['tanggal'] <= batas, c] = t.loc[t['tanggal'] <= batas, c].fillna(0)
t['tahun'] = t['tanggal'].dt.year
t['bulan'] = t['tanggal'].dt.month
t = t[urutan]
t.to_csv(HASIL / 'tabel_gabungan_v2.csv', index=False)

print('baris v2:', len(t), '| kolom:', t.shape[1], '(harapan 40052 | 77)')
print(t.groupby('tahun')['tanggal'].nunique().to_string())

a = v1.set_index(['tanggal', 'GID_2']).sort_index()
b = t.set_index(['tanggal', 'GID_2']).sort_index()
ada = a.index.intersection(b.index)
beda = []
for c in a.columns:
    x, y = a.loc[ada, c], b.loc[ada, c]
    if x.dtype.kind in 'fi':
        ok = np.allclose(x.to_numpy(float), y.to_numpy(float), equal_nan=True, rtol=1e-6, atol=1e-9)
    else:
        ok = (x.fillna('~').astype(str) == y.fillna('~').astype(str)).all()
    if not ok:
        beda.append(c)
print('kolom yang beda dari v1 di baris yang sama:', beda, '(harapan [])')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
1054 file npz (harapan 1054)
baris v2: 40052 | kolom: 77 (harapan 40052 | 77)
tahun
2020    153
2021    153
2022    153
2023    153
2024    153
2025    153
2026    136
kolom yang beda dari v1 di baris yang sama: [] (harapan [])


In [ ]:
from pathlib import Path
import pandas as pd

ROOT = Path('/content/drive/MyDrive/Logika Ui')
HASIL = ROOT / 'KTI_KARHUTLA' / 'hasil'

v2 = pd.read_csv(HASIL / 'tabel_gabungan_v2.csv')
kolom_him = [k for k in v2.columns if k.startswith('him_')]
kosong = v2[v2[kolom_him].isna().all(axis=1)]
hari = pd.to_datetime(kosong['tanggal']).dt.date.unique()
print(len(kosong), len(hari))
print(pd.Series(hari).groupby(pd.to_datetime(pd.Series(hari)).dt.to_period('M')).size())

3762 99
2022-09     1
2023-06    30
2023-07    31
2023-08    16
2024-10     1
2025-10    20
Freq: M, dtype: int64


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

from pathlib import Path
HASIL = Path('/content/drive/MyDrive/Logika Ui/imerg')
print(HASIL.exists())
print([p.name for p in HASIL.glob('*v2*')])

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
True
[]


In [ ]:
from pathlib import Path
import geopandas as gpd
ROOT = Path('/content/drive/MyDrive/Logika Ui')
HASIL = ROOT / 'KTI_KARHUTLA' / 'hasil'
gadm2 = gpd.read_file(ROOT / 'GADM 2' / 'gadm41_IDN_2.shp')

import numpy as np
import pandas as pd
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.metrics import brier_score_loss, log_loss, roc_auc_score

PROV = ['Riau', 'Jambi', 'Sumatera Selatan']
TAU = 0.05

tabel = pd.read_csv(HASIL / 'tabel_gabungan_v2.csv', parse_dates=['tanggal'])

wilayah = gadm2[gadm2['NAME_1'].isin(PROV)]
pusat = wilayah.to_crs('EPSG:6933').geometry.centroid.to_crs('EPSG:4326')
koor = pd.DataFrame({'GID_2': wilayah['GID_2'].values, 'lat': pusat.y.values, 'lon': pusat.x.values})
nama = wilayah[['GID_2', 'NAME_2']]
tabel = tabel.drop(columns=['NAME_2'], errors='ignore').merge(nama, on='GID_2', how='left')
tabel = tabel.merge(koor, on='GID_2', how='left')

hari = tabel['tanggal'].dt.dayofyear
tabel['doy_sin'] = np.sin(2 * np.pi * hari / 365.25)
tabel['doy_cos'] = np.cos(2 * np.pi * hari / 365.25)

curah = tabel['hujan_rata'] * 6
for tau in [0.02, 0.05, 0.1, 0.2]:
    k = np.where(curah >= 1, 2, np.where(tabel['frac_hujan'] >= tau, 1, 0))
    print('tau', tau, pd.Series(k).value_counts(normalize=True).sort_index().round(3).to_dict())

tabel['kelas'] = np.where(curah >= 1, 2, np.where(tabel['frac_hujan'] >= TAU, 1, 0))
print(pd.crosstab(tabel['tahun'], tabel['kelas'], normalize='index').round(3))
print(pd.crosstab(tabel['bulan'], tabel['kelas'], normalize='index').round(3))

fitur_dasar = ['lat', 'lon', 'doy_sin', 'doy_cos']
fitur_era = [c for c in tabel.columns if c.endswith(('_00', '_06'))]
fitur_era_00 = [c for c in fitur_era if c.endswith('_00')]
fitur_him = [c for c in tabel.columns
             if c.startswith('him_') and c not in ('him_n', 'him_f5')]
set_fitur = {
    'C prakiraan (00 UTC)': fitur_dasar + fitur_era_00,
    'D pemantauan (00 UTC + awan)': fitur_dasar + fitur_era_00 + fitur_him,
}

def model_gb():
    return HistGradientBoostingClassifier(max_depth=4, learning_rate=0.05, max_iter=250,
                                          l2_regularization=1.0, early_stopping=False,
                                          random_state=1)

def klim_kelas(latih, uji):
    peta = (pd.get_dummies(latih['kelas']).astype(float)
            .assign(GID_2=latih['GID_2'].values, bulan=latih['bulan'].values)
            .groupby(['GID_2', 'bulan']).mean().reset_index())
    P = uji[['GID_2', 'bulan']].merge(peta, on=['GID_2', 'bulan'], how='left')[[0, 1, 2]]
    rata = latih['kelas'].value_counts(normalize=True).sort_index()
    return P.fillna(pd.Series(rata.to_numpy(), index=[0, 1, 2])).to_numpy()

n = len(tabel)
kelas = tabel['kelas'].to_numpy()
Y = np.eye(3)[kelas]
tahun = tabel['tahun'].to_numpy()

prob = {'klimatologi': np.zeros((n, 3))}
for th in np.unique(tahun):
    prob['klimatologi'][tahun == th] = klim_kelas(tabel[tahun != th], tabel[tahun == th])

for nama_set, kolom in set_fitur.items():
    prob[nama_set] = np.zeros((n, 3))
    for th in np.unique(tahun):
        m = model_gb()
        m.fit(tabel.loc[tahun != th, kolom], kelas[tahun != th])
        prob[nama_set][tahun == th] = m.predict_proba(tabel.loc[tahun == th, kolom])
    print('selesai', nama_set)

def skor(P):
    P = np.clip(P, 1e-6, 1)
    P = P / P.sum(axis=1, keepdims=True)
    return {'brier_3kelas': ((P - Y) ** 2).sum(axis=1).mean(),
            'log_loss': log_loss(kelas, P, labels=[0, 1, 2]),
            'brier_K1': brier_score_loss(kelas == 1, P[:, 1]),
            'auc_K1': roc_auc_score(kelas == 1, P[:, 1])}

ringkas = pd.DataFrame({k: skor(P) for k, P in prob.items()}).T
ringkas['skill_3kelas'] = 1 - ringkas['brier_3kelas'] / ringkas.loc['klimatologi', 'brier_3kelas']
print(ringkas.round(4).to_string())

for nama_set in set_fitur:
    p1 = prob[nama_set][:, 1]
    grup = pd.qcut(p1, 10, duplicates='drop')
    print(nama_set)
    print(pd.DataFrame({'g': grup, 'prediksi': p1, 'amatan': (kelas == 1).astype(int)})
          .groupby('g', observed=True)
          .agg(jumlah=('amatan', 'size'), prediksi=('prediksi', 'mean'), amatan=('amatan', 'mean'))
          .round(3))

keluar = tabel[['tanggal', 'GID_2', 'NAME_2', 'kelas']].copy()
for kode, nama_set in [('C', 'C prakiraan (00 UTC)'), ('D', 'D pemantauan (00 UTC + awan)')]:
    for k in range(3):
        keluar[f'p{k}_{kode}'] = prob[nama_set][:, k]
keluar.to_csv(HASIL / 'prediksi_kelas_v2.csv', index=False)
print(keluar.groupby('NAME_2')[['p1_C', 'p1_D']].mean().sort_values('p1_D').tail(8).round(3))


tau 0.02 {0: 0.302, 1: 0.407, 2: 0.29}
tau 0.05 {0: 0.417, 1: 0.292, 2: 0.29}
tau 0.1 {0: 0.545, 1: 0.164, 2: 0.29}
tau 0.2 {0: 0.668, 1: 0.042, 2: 0.29}
kelas      0      1      2
tahun                     
2020   0.352  0.307  0.341
2021   0.370  0.307  0.323
2022   0.261  0.349  0.390
2023   0.541  0.257  0.202
2024   0.463  0.283  0.254
2025   0.411  0.283  0.307
2026   0.538  0.254  0.207
kelas      0      1      2
bulan                     
6      0.380  0.314  0.306
7      0.510  0.274  0.216
8      0.494  0.248  0.258
9      0.383  0.290  0.327
10     0.310  0.338  0.351
selesai C prakiraan (00 UTC)
selesai D pemantauan (00 UTC + awan)
                              brier_3kelas  log_loss  brier_K1  auc_K1  skill_3kelas
klimatologi                         0.6484    1.0730    0.2050  0.5673        0.0000
C prakiraan (00 UTC)                0.5391    0.9078    0.1988  0.6314        0.1685
D pemantauan (00 UTC + awan)        0.4984    0.8376    0.1934  0.6667        0.2314
C prakir

In [ ]:
def brier3(P, m):
    P = np.clip(P[m], 1e-6, 1)
    P = P / P.sum(axis=1, keepdims=True)
    return ((P - Y[m]) ** 2).sum(axis=1).mean()

ada_awan = tabel['him_n'].notna().to_numpy()
for nama_sub, m in [('semua baris', np.ones(len(tabel), bool)),
                    ('baris ADA awan Himawari', ada_awan),
                    ('baris TANPA awan Himawari', ~ada_awan)]:
    b0 = brier3(prob['klimatologi'], m)
    print(nama_sub, '| baris:', int(m.sum()),
          {k[:1]: round(1 - brier3(prob[k], m) / b0, 4) for k in set_fitur})

semua baris | baris: 40052 {'C': np.float64(0.1685), 'D': np.float64(0.2314)}
baris ADA awan Himawari | baris: 36290 {'C': np.float64(0.1711), 'D': np.float64(0.2512)}
baris TANPA awan Himawari | baris: 3762 {'C': np.float64(0.1426), 'D': np.float64(0.0359)}


In [ ]:
ada_awan = tabel['him_n'].notna().to_numpy()
pD = prob['D pemantauan (00 UTC + awan)'].copy()
pD[~ada_awan] = prob['C prakiraan (00 UTC)'][~ada_awan]

semua = np.ones(len(tabel), bool)
b0 = brier3(prob['klimatologi'], semua)
print('skill D dengan cadangan C saat awan kosong:', round(1 - brier3(pD, semua) / b0, 4))

for k in range(3):
    keluar[f'p{k}_D'] = pD[:, k]
keluar['ada_awan'] = ada_awan.astype(int)
keluar.to_csv(HASIL / 'prediksi_kelas_v2.csv', index=False)
print(keluar['ada_awan'].value_counts().to_dict())

skill D dengan cadangan C saat awan kosong: 0.2412
{1: 36290, 0: 3762}


In [ ]:
import numpy as np
import pandas as pd

akar = ROOT / 'KTI_KARHUTLA'
firms = pd.concat(
    [pd.read_csv(p, usecols=['acq_date', 'confidence', 'type', 'GID_2'])
     for p in sorted((akar / 'firms_gadm').glob('viirs_*.csv'))],
    ignore_index=True,
)
firms = firms[(firms['type'] == 0) & (firms['confidence'].astype(str).str.lower() != 'l')]
firms['tahun'] = pd.to_datetime(firms['acq_date']).dt.year
firms = firms.merge(gadm2[['GID_2', 'NAME_1']], on='GID_2', how='left')

n = firms[firms['tahun'].between(2015, 2025)].groupby('NAME_1').size().sort_values(ascending=False)
persen = (n / n.sum() * 100).round(1)
print(pd.DataFrame({'titik_panas': n, 'persen': persen, 'kumulatif': persen.cumsum().round(1)}).head(12))

berkas = sorted((ROOT / 'imerg' / 'potongan').glob('*.npz'))
z = np.load(berkas[0])
print(z['lat'].min(), z['lat'].max(), z['lon'].min(), z['lon'].max())

                     titik_panas  persen  kumulatif
NAME_1                                             
Kalimantan Tengah         351959    16.7       16.7
Sumatera Selatan          263943    12.5       29.2
Kalimantan Barat          242375    11.5       40.7
Papua                     155669     7.4       48.1
Nusa Tenggara Timur       147292     7.0       55.1
Kalimantan Timur          124800     5.9       61.0
Riau                      117214     5.6       66.6
Jambi                      87279     4.1       70.7
Kalimantan Selatan         85567     4.1       74.8
Nusa Tenggara Barat        62560     3.0       77.8
Jawa Timur                 47978     2.3       80.1
Sulawesi Tengah            44598     2.1       82.2
-4.95 2.95 100.049995 106.95


In [ ]:
import re
import pandas as pd
import s3fs
import xarray as xr
from pathlib import Path

fs = s3fs.S3FileSystem(anon=True)
PRODUK = 'AHI-L2-FLDK-Clouds'
TUJUAN = Path('/content/him8_okt2025')
TUJUAN.mkdir(exist_ok=True)
JAM_AWAL, JAM_AKHIR = '0300', '0830'
pola = re.compile(r'_s(\d{8})(\d{4})')

def daftar_slot(bucket, tgl):
    try:
        isi = fs.ls(f'{bucket}/{PRODUK}/{tgl:%Y/%m/%d}')
    except FileNotFoundError:
        return []
    keluar = []
    for p in isi:
        m = pola.search(p.split('/')[-1])
        if m and JAM_AWAL <= m.group(2) <= JAM_AKHIR:
            keluar.append(p)
    return sorted(keluar)

rekap = {}
for tgl in pd.date_range('2025-10-12', '2025-10-31'):
    slot = daftar_slot('noaa-himawari8', tgl)
    rekap[tgl.date()] = len(slot)
    for p in slot:
        tujuan = TUJUAN / p.split('/')[-1]
        if not tujuan.exists():
            fs.get(p, str(tujuan))
print(rekap)

contoh8 = sorted(TUJUAN.glob('*.nc'))[0]
slot9 = daftar_slot('noaa-himawari9', pd.Timestamp('2025-10-11'))
contoh9 = TUJUAN / slot9[0].split('/')[-1]
fs.get(slot9[0], str(contoh9))

for nama, f in [('H8', contoh8), ('H9', contoh9)]:
    d = xr.open_dataset(f)
    print(nama, f.name)
    print(list(d.data_vars), dict(d.sizes))

{datetime.date(2025, 10, 12): 0, datetime.date(2025, 10, 13): 0, datetime.date(2025, 10, 14): 0, datetime.date(2025, 10, 15): 0, datetime.date(2025, 10, 16): 0, datetime.date(2025, 10, 17): 0, datetime.date(2025, 10, 18): 0, datetime.date(2025, 10, 19): 0, datetime.date(2025, 10, 20): 0, datetime.date(2025, 10, 21): 0, datetime.date(2025, 10, 22): 0, datetime.date(2025, 10, 23): 0, datetime.date(2025, 10, 24): 0, datetime.date(2025, 10, 25): 0, datetime.date(2025, 10, 26): 0, datetime.date(2025, 10, 27): 0, datetime.date(2025, 10, 28): 0, datetime.date(2025, 10, 29): 0, datetime.date(2025, 10, 30): 0, datetime.date(2025, 10, 31): 0}


IndexError: list index out of range

In [ ]:
!pip -q install earthaccess regionmask h5py
import re
import h5py
import numpy as np
import pandas as pd
import regionmask
import earthaccess
from concurrent.futures import ThreadPoolExecutor

earthaccess.login()

OUT = HASIL / 'imerg_nasional_harian.csv'
W, E, S, N = 94.5, 141.5, -11.5, 6.5

lon_all = np.arange(3600) * 0.1 - 179.95
lat_all = np.arange(1800) * 0.1 - 89.95
i0, i1 = np.searchsorted(lon_all, [W, E])
j0, j1 = np.searchsorted(lat_all, [S, N])
lon, lat = lon_all[i0:i1], lat_all[j0:j1]

gadm = gadm2[['GID_2', 'geometry']].reset_index(drop=True)
regions = regionmask.from_geopandas(gadm, names='GID_2')
peta = regions.mask(lon, lat).values
valid = ~np.isnan(peta)
kode = peta[valid].astype(int)
gid = gadm['GID_2'].to_numpy()
nreg = len(gadm)

pola = re.compile(r'3IMERG\.(\d{8})-S(\d{4})')

def jenis(tgl):
    return ('F', 'GPM_3IMERGHH') if tgl < pd.Timestamp('2025-10-01') else ('L', 'GPM_3IMERGHHL')

def baca(f):
    with h5py.File(f, 'r') as h:
        a = h['Grid/precipitation'][0, i0:i1, j0:j1]
    a = a.T.astype('float32')
    a[a < -9000] = np.nan
    return a

def agregat(tumpuk, tgl, run):
    nilai = tumpuk[:, valid]
    fin = np.isfinite(nilai)
    v = np.where(fin, nilai, 0.0)
    jml = np.zeros(nreg)
    n = np.zeros(nreg)
    hjn = np.zeros(nreg)
    maks = np.full(nreg, -np.inf)
    for k in range(nilai.shape[0]):
        jml += np.bincount(kode, weights=v[k].astype(float), minlength=nreg)
        n += np.bincount(kode, weights=fin[k].astype(float), minlength=nreg)
        hjn += np.bincount(kode, weights=(v[k] >= 0.1).astype(float), minlength=nreg)
        np.maximum.at(maks, kode, np.where(fin[k], v[k], -np.inf))
    ada = n > 0
    return pd.DataFrame({'tanggal': tgl, 'run': run, 'GID_2': gid[ada],
                         'hujan_rata': jml[ada] / n[ada], 'hujan_maks': maks[ada],
                         'frac_hujan': hjn[ada] / n[ada]})

def proses(tgl, daftar, run):
    daftar = sorted(daftar, key=lambda x: x[0])
    if len(daftar) != 12:
        return None
    berkas = earthaccess.open([g for _, g in daftar])
    tumpuk = np.stack([baca(f) for f in berkas])
    return agregat(tumpuk, pd.Timestamp(tgl), run)

def cari(awal, akhir):
    run, sn = jenis(awal)
    granul = earthaccess.search_data(short_name=sn, version='07',
                                     temporal=(f'{awal:%Y-%m-%d} 00:00:00', f'{akhir:%Y-%m-%d} 23:59:59'),
                                     count=-1)
    d = {}
    for g in granul:
        m = pola.search(g.data_links()[0].split('/')[-1])
        if m and '0300' <= m.group(2) <= '0830':
            d.setdefault(m.group(1), []).append((m.group(2), g))
    return run, d

run, d = cari(pd.Timestamp('2023-09-01'), pd.Timestamp('2023-09-01'))
tes = proses('2023-09-01', d['20230901'], run)
lama = pd.read_csv(HASIL / 'imerg_kab_harian_v2.csv', parse_dates=['tanggal'])
lama = lama[lama['tanggal'] == '2023-09-01']
gab = lama.merge(tes, on=['tanggal', 'GID_2'], suffixes=('_lama', '_baru'))
print(len(lama), len(tes), len(gab))
for c in ['hujan_rata', 'hujan_maks', 'frac_hujan']:
    print(c, np.allclose(gab[c + '_lama'], gab[c + '_baru'], atol=1e-4))

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 78.7/78.7 kB 4.0 MB/s eta 0:00:00
Enter your Earthdata Login username: andarasalbriya
Enter your Earthdata password: ··········


QUEUEING TASKS | :   0%|          | 0/12 [00:00<?, ?it/s]

PROCESSING TASKS | :   0%|          | 0/12 [00:00<?, ?it/s]

COLLECTING RESULTS | :   0%|          | 0/12 [00:00<?, ?it/s]

38 479 38
hujan_rata True
hujan_maks True
frac_hujan True


In [ ]:
sudah = set()
if OUT.exists():
    sudah = set(pd.read_csv(OUT, usecols=['tanggal'])['tanggal'].unique())

potongan = []
for th in range(2020, 2027):
    akhir_th = pd.Timestamp('2026-10-14') if th == 2026 else pd.Timestamp(f'{th}-12-31')
    if th == 2025:
        potongan += [(pd.Timestamp('2025-06-01'), pd.Timestamp('2025-09-30')),
                     (pd.Timestamp('2025-10-01'), akhir_th)]
    else:
        potongan.append((pd.Timestamp(f'{th}-01-01'), akhir_th))

for awal, akhir in potongan:
    run, d = cari(awal, akhir)
    hari = [t for t in sorted(d) if str(pd.Timestamp(t).date()) not in sudah]
    for b in range(0, len(hari), 12):
        grup = hari[b:b + 12]
        if not grup:
            continue
        with ThreadPoolExecutor(max_workers=6) as ex:
            hasil = list(ex.map(lambda t: proses(t, d[t], run), grup))
        hasil = [h for h in hasil if h is not None]
        if hasil:
            pd.concat(hasil).to_csv(OUT, mode='a', header=not OUT.exists(), index=False)
            print(run, grup[0], 'sampai', grup[-1], '| hari tersimpan:', len(hasil), 'dari', len(grup))

In [ ]:
im = pd.read_csv(HASIL / 'imerg_nasional_harian.csv', parse_dates=['tanggal'])
im = im.sort_values('run').drop_duplicates(['tanggal', 'GID_2'], keep='first')
lama = pd.read_csv(HASIL / 'imerg_kab_harian_v2.csv', parse_dates=['tanggal'])

print(im['tanggal'].nunique(), 'hari |', im['GID_2'].nunique(), 'wilayah |', len(im), 'baris')
print(im['tanggal'].min().date(), im['tanggal'].max().date())

gab = lama.merge(im, on=['tanggal', 'GID_2'], suffixes=('_lama', '_baru'), how='left')
print(len(lama), 'baris lama |', gab['hujan_rata_baru'].notna().sum(), 'ketemu di yang baru')
for c in ['hujan_rata', 'hujan_maks', 'frac_hujan']:
    sel = gab[c + '_baru'].notna()
    print(c, np.allclose(gab.loc[sel, c + '_lama'], gab.loc[sel, c + '_baru'], atol=1e-4))

1054 hari | 479 wilayah | 504866 baris
2020-06-01 2026-10-14
40052 baris lama | 40052 ketemu di yang baru
hujan_rata True
hujan_maks True
frac_hujan True


In [ ]:
akar = ROOT / 'KTI_KARHUTLA'
firms = pd.concat(
    [pd.read_csv(p, usecols=['acq_date', 'confidence', 'type', 'GID_2', 'frp'])
     for p in sorted((akar / 'firms_gadm').glob('viirs_*.csv'))],
    ignore_index=True,
)
firms['tanggal'] = pd.to_datetime(firms['acq_date'])
rendah = firms['confidence'].astype(str).str.lower() == 'l'

syarat = {
    'semua': firms,
    'type0': firms[firms['type'] == 0],
    'bukan_low': firms[~rendah],
    'type0_bukan_low': firms[(firms['type'] == 0) & ~rendah],
}

def agregat_fire(d):
    return (d.groupby(['tanggal', 'GID_2'])
              .agg(titik_panas=('frp', 'size'), frp_total=('frp', 'sum'), frp_maks=('frp', 'max'))
              .reset_index())

lama = pd.read_csv(HASIL / 'firms_kab_harian.csv', parse_dates=['tanggal'])
batas = lama['tanggal'].max()
gid_s = gadm2.loc[gadm2['NAME_1'].isin(['Riau', 'Jambi', 'Sumatera Selatan']), 'GID_2']

cocok = None
for nama_s, d in syarat.items():
    a = agregat_fire(d)
    a = a[a['GID_2'].isin(gid_s) & (a['tanggal'] <= batas) & (a['tanggal'] >= lama['tanggal'].min())]
    g = lama[['tanggal', 'GID_2', 'titik_panas', 'frp_total']].merge(
        a[['tanggal', 'GID_2', 'titik_panas', 'frp_total']],
        on=['tanggal', 'GID_2'], how='outer', suffixes=('_lama', '_baru')).fillna(0)
    ok = (np.allclose(g['titik_panas_lama'], g['titik_panas_baru'])
          and np.allclose(g['frp_total_lama'], g['frp_total_baru'], atol=1e-3))
    print(nama_s, ok)
    if ok and cocok is None:
        cocok = nama_s

if cocok:
    agregat_fire(syarat[cocok]).to_csv(HASIL / 'firms_nasional_harian.csv', index=False)
    print('tersimpan dengan aturan:', cocok)

semua False
type0 False
bukan_low False
type0_bukan_low True
tersimpan dengan aturan: type0_bukan_low


In [ ]:
from pathlib import Path
import pandas as pd

ROOT = Path('/content/drive/MyDrive/Logika Ui')
folder_him = ROOT / 'Himawari'

him = pd.concat([pd.read_csv(p) for p in sorted(folder_him.glob('himawari_*.csv'))])
him['tanggal'] = pd.to_datetime(him['tanggal'])
ada = him['tanggal'].dt.to_period('M').value_counts().sort_index()
print(ada.loc['2023-05':'2023-09'])
print(ada.loc['2025-09':'2025-10'])

tanggal
2023-08    312
2023-09    627
Freq: M, Name: count, dtype: int64
tanggal
2025-09    630
2025-10    231
Freq: M, Name: count, dtype: int64


In [ ]:
from pathlib import Path
import pandas as pd

ROOT = Path('/content/drive/MyDrive/Logika Ui')
folder_him = ROOT / 'Himawari'

log = pd.concat([pd.read_csv(p) for p in sorted(folder_him.glob('log_*.csv'))], ignore_index=True)
print(log.columns.tolist())
print(log['status'].value_counts())
print(log.tail(10))

['satelit', 'tanggal', 'waktu', 'status', 'detik']
status
ok                6646
hari tidak ada     140
slot tidak ada      60
Name: count, dtype: int64
        satelit     tanggal  waktu          status  detik
6836  himawari9  2026-10-13    700              ok   17.0
6837  himawari9  2026-10-13    800              ok   20.0
6838  himawari9  2026-10-13    900              ok   21.8
6839  himawari9  2026-10-14    300              ok   18.9
6840  himawari9  2026-10-14    400              ok   21.8
6841  himawari9  2026-10-14    500              ok   16.6
6842  himawari9  2026-10-14    600  slot tidak ada    0.0
6843  himawari9  2026-10-14    700  slot tidak ada    0.0
6844  himawari9  2026-10-14    800  slot tidak ada    0.0
6845  himawari9  2026-10-14    900  slot tidak ada    0.0


In [ ]:
log['tanggal'] = pd.to_datetime(log['tanggal'])

p1 = log[(log['tanggal'] >= '2023-06-01') & (log['tanggal'] <= '2023-08-16')]
p2 = log[(log['tanggal'] >= '2025-10-12') & (log['tanggal'] <= '2025-10-31')]

print('Jun-Agu 2023:', len(p1)); print(p1['status'].value_counts())
print('Okt 2025:', len(p2)); print(p2['status'].value_counts())

Jun-Agu 2023: 7
status
slot tidak ada    7
Name: count, dtype: int64
Okt 2025: 140
status
hari tidak ada    140
Name: count, dtype: int64


In [ ]:
from pathlib import Path
import pandas as pd, xarray as xr

ROOT = Path('/content/drive/MyDrive/Logika Ui')
akar = ROOT / 'KTI_KARHUTLA'
HASIL = akar / 'hasil'

era = pd.read_csv(HASIL / 'era5_kab_harian.csv')
print(era.shape)
print(era.columns.tolist())
print(era.head(3))

for nama in ['pl_2020_06.nc', 'sl_2020_06.nc', 'plx_2020_06.nc', 'slx_2020_06.nc']:
    p = next(akar.rglob(nama), None)
    if p is None:
        print(nama, 'tidak ketemu')
        continue
    d = xr.open_dataset(p)
    print(nama, list(d.data_vars), dict(d.sizes))

(68780, 52)
['tanggal', 't_850_00', 't_850_06', 't_700_00', 't_700_06', 't_500_00', 't_500_06', 'r_850_00', 'r_850_06', 'r_700_00', 'r_700_06', 'r_500_00', 'r_500_06', 'u_850_00', 'u_850_06', 'u_700_00', 'u_700_06', 'u_500_00', 'u_500_06', 'v_850_00', 'v_850_06', 'v_700_00', 'v_700_06', 'v_500_00', 'v_500_06', 'cape_00', 'cape_06', 't2m_00', 't2m_06', 'd2m_00', 'd2m_06', 'w_850_00', 'w_850_06', 'w_700_00', 'w_700_06', 'w_500_00', 'w_500_06', 'q_850_00', 'q_850_06', 'q_700_00', 'q_700_06', 'q_500_00', 'q_500_06', 'tcwv_00', 'tcwv_06', 'tclw_00', 'tclw_06', 'tcc_00', 'tcc_06', 'sp_00', 'sp_06', 'GID_2']
      tanggal    t_850_00    t_850_06    t_700_00    t_700_06    t_500_00  \
0  2015-06-01  291.164377  292.026557  283.243434  283.197377  267.842998   
1  2015-06-01  291.513672  292.879359  283.579617  283.569438  267.859759   
2  2015-06-01  291.153593  292.063650  283.084090  283.130611  267.777722   

     t_500_06   r_850_00   r_850_06   r_700_00  ...  q_500_06    tcwv_00  \
0  268

In [ ]:
!pip -q install regionmask
from pathlib import Path
import numpy as np, pandas as pd, xarray as xr, geopandas as gpd, regionmask

ROOT = Path('/content/drive/MyDrive/Logika Ui')
akar = ROOT / 'KTI_KARHUTLA'
HASIL = akar / 'hasil'

gadm2 = gpd.read_file(ROOT / 'GADM 2' / 'gadm41_IDN_2.shp')
studi = gadm2[gadm2['NAME_1'].isin(['Riau', 'Jambi', 'Sumatera Selatan'])].reset_index(drop=True)
regions = regionmask.from_geopandas(studi, names='NAME_2')

def satu_berkas(path, jam=(0, 6)):
    d = xr.open_dataset(path)
    d = d.sel(valid_time=d.valid_time.dt.hour.isin(jam))
    peta = regions.mask(d.longitude.values, d.latitude.values).values
    waktu = d.valid_time.values
    kolom = {}
    for v in d.data_vars:
        a = d[v]
        if 'pressure_level' in a.dims:
            for lv in a.pressure_level.values:
                kolom[f'{v}_{int(lv)}'] = a.sel(pressure_level=lv).values
        else:
            kolom[v] = a.values
    hasil = []
    for k in range(len(studi)):
        m = peta == k
        isi = {n: (x[:, m].mean(axis=1) if m.any() else np.full(len(waktu), np.nan))
               for n, x in kolom.items()}
        idx = pd.MultiIndex.from_product([waktu, [k]], names=['valid_time', 'kab'])
        hasil.append(pd.DataFrame(isi, index=idx))
    return pd.concat(hasil)

# tes Juni 2020
potong = [satu_berkas(next(akar.rglob(f'{j}_2020_06.nc'))) for j in ['pl', 'sl', 'plx', 'slx']]
baru = pd.concat(potong, axis=1).reset_index()
baru['tanggal'] = baru['valid_time'].dt.normalize()
baru['jam'] = baru['valid_time'].dt.hour
nilai = [c for c in baru.columns if c not in ('valid_time', 'kab', 'tanggal', 'jam')]
baru = baru.set_index(['tanggal', 'kab', 'jam'])[nilai].unstack('jam')
baru.columns = [f'{v}_{j:02d}' for v, j in baru.columns]
baru = baru.reset_index()
baru['GID_2'] = studi['GID_2'].values[baru['kab'].to_numpy()]
baru = baru.drop(columns='kab')

lama = pd.read_csv(HASIL / 'era5_kab_harian.csv', parse_dates=['tanggal'])
lama = lama[(lama['tanggal'] >= '2020-06-01') & (lama['tanggal'] <= '2020-06-30')]
g = baru.merge(lama, on=['tanggal', 'GID_2'], suffixes=('_b', '_l'))
print('baris baru, lama, cocok:', len(baru), len(lama), len(g))
kol = [c for c in lama.columns if c not in ('tanggal', 'GID_2')]
selisih = pd.Series({c: (g[c + '_b'] - g[c + '_l']).abs().max() for c in kol})
print(selisih.describe())
print(selisih.sort_values(ascending=False).head(5))
print('NaN tertinggi di baru:', baru.isna().mean().max())

/tmp/ipykernel_4180/3845466115.py:33: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  return pd.concat(hasil)
/tmp/ipykernel_4180/3845466115.py:33: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  return pd.concat(hasil)
/tmp/ipykernel_4180/3845466115.py:33: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude th

baris baru, lama, cocok: 1140 1140 1140
count      50.000000
mean      131.090790
std       490.949302
min         0.000527
25%         0.418493
50%         1.680954
75%         3.922871
max      2364.594729
dtype: float64
sp_00       2364.594729
sp_06       2363.106429
cape_06      996.085092
cape_00      707.831190
r_700_00      15.380253
dtype: float64
NaN tertinggi di baru: 0.15789473684210525


In [ ]:
pusat = studi.to_crs('EPSG:6933').geometry.centroid.to_crs('EPSG:4326')
lat_k = xr.DataArray(pusat.y.values, dims='kab')
lon_k = xr.DataArray(pusat.x.values, dims='kab')

def satu_berkas(path, jam=(0, 6)):
    d = xr.open_dataset(path).sortby('latitude')
    d = d.sel(valid_time=d.valid_time.dt.hour.isin(jam))
    di = d.interp(latitude=lat_k, longitude=lon_k, method='linear')
    kolom = {}
    for v in di.data_vars:
        a = di[v]
        if 'pressure_level' in a.dims:
            for lv in a.pressure_level.values:
                kolom[f'{v}_{int(lv)}'] = a.sel(pressure_level=lv).transpose('valid_time', 'kab').values
        else:
            kolom[v] = a.transpose('valid_time', 'kab').values
    idx = pd.MultiIndex.from_product([di.valid_time.values, range(len(studi))], names=['valid_time', 'kab'])
    return pd.DataFrame({n: x.reshape(-1) for n, x in kolom.items()}, index=idx)

potong = [satu_berkas(next(akar.rglob(f'{j}_2020_06.nc'))) for j in ['pl', 'sl', 'plx', 'slx']]
baru = pd.concat(potong, axis=1).reset_index()
baru['tanggal'] = baru['valid_time'].dt.normalize()
baru['jam'] = baru['valid_time'].dt.hour
nilai = [c for c in baru.columns if c not in ('valid_time', 'kab', 'tanggal', 'jam')]
baru = baru.set_index(['tanggal', 'kab', 'jam'])[nilai].unstack('jam')
baru.columns = [f'{v}_{j:02d}' for v, j in baru.columns]
baru = baru.reset_index()
baru['GID_2'] = studi['GID_2'].values[baru['kab'].to_numpy()]
baru = baru.drop(columns='kab')

g = baru.merge(lama, on=['tanggal', 'GID_2'], suffixes=('_b', '_l'))
print('baris baru, lama, cocok:', len(baru), len(lama), len(g))
selisih = pd.Series({c: (g[c + '_b'] - g[c + '_l']).abs().max() for c in kol})
print(selisih.describe())
print(selisih.sort_values(ascending=False).head(5))
print('NaN tertinggi di baru:', baru.isna().mean().max())

baris baru, lama, cocok: 1140 1140 1140
count    5.000000e+01
mean     6.139867e-13
std      2.875352e-12
min      9.714451e-17
25%      1.110223e-16
50%      1.776357e-15
75%      5.684342e-14
max      1.455192e-11
dtype: float64
sp_06       1.455192e-11
sp_00       1.455192e-11
cape_06     4.547474e-13
cape_00     4.547474e-13
t_500_00    5.684342e-14
dtype: float64
NaN tertinggi di baru: 0.0


In [ ]:
import shutil, time, re

gadm2 = gpd.read_file(ROOT / 'GADM 2' / 'gadm41_IDN_2.shp')
studi = gadm2.reset_index(drop=True)
print('jumlah wilayah:', len(studi))

pusat = studi.to_crs('EPSG:6933').geometry.centroid.to_crs('EPSG:4326')
lat_k = xr.DataArray(pusat.y.values, dims='kab')
lon_k = xr.DataArray(pusat.x.values, dims='kab')

berkas_era = {}
for p in akar.rglob('*.nc'):
    m = re.fullmatch(r'(pl|sl|plx|slx)_(\d{4})_(\d{2})\.nc', p.name)
    if m:
        berkas_era[(m[1], int(m[2]), int(m[3]))] = p
print('berkas ketemu:', len(berkas_era))

LOKAL = Path('/content/era5_lokal'); LOKAL.mkdir(exist_ok=True)
potongan, gagal = [], []
for tahun in range(2015, 2027):
    for bulan in range(6, 11):
        kunci = [(j, tahun, bulan) for j in ['pl', 'sl', 'plx', 'slx']]
        if not all(k in berkas_era for k in kunci):
            gagal.append((tahun, bulan, 'tidak ada di Drive')); continue
        try:
            lok = []
            for k in kunci:
                tujuan = LOKAL / berkas_era[k].name
                if not (tujuan.exists() and tujuan.stat().st_size > 0):
                    shutil.copy(berkas_era[k], tujuan)
                lok.append(tujuan)
            potongan.append(pd.concat([satu_berkas(p) for p in lok], axis=1))
            for p in lok: p.unlink(missing_ok=True)
        except Exception as e:
            gagal.append((tahun, bulan, str(e)[:70]))
print('bulan berhasil:', len(potongan), '| bermasalah:', gagal)

era = pd.concat(potongan).reset_index()
era['tanggal'] = era['valid_time'].dt.normalize()
era['jam'] = era['valid_time'].dt.hour
nilai = [c for c in era.columns if c not in ('valid_time', 'kab', 'tanggal', 'jam')]
era = era.set_index(['tanggal', 'kab', 'jam'])[nilai].unstack('jam')
era.columns = [f'{v}_{j:02d}' for v, j in era.columns]
era = era.reset_index()
era['GID_2'] = studi['GID_2'].values[era['kab'].to_numpy()]
era = era.drop(columns='kab')
era.to_csv(HASIL / 'era5_nasional_harian.csv', index=False)

print(era.shape, '| NaN tertinggi:', era.isna().mean().max())
lama_all = pd.read_csv(HASIL / 'era5_kab_harian.csv', parse_dates=['tanggal'])
g = era.merge(lama_all, on=['tanggal', 'GID_2'], suffixes=('_b', '_l'))
kol_all = [c for c in lama_all.columns if c not in ('tanggal', 'GID_2')]
print('baris cocok dengan file lama:', len(g), 'dari', len(lama_all))
print('selisih maks:', max((g[c + '_b'] - g[c + '_l']).abs().max() for c in kol_all))

jumlah wilayah: 502
berkas ketemu: 568
bulan berhasil: 59 | bermasalah: [(2015, 7, "[Errno -101] NetCDF: HDF error: '/content/era5_lokal/pl_2015_07.nc'")]
(893058, 52) | NaN tertinggi: 0.0
baris cocok dengan file lama: 67602 dari 68780
selisih maks: 1.4551915228366852e-11


In [ ]:
def pivot(df):
    d = df.reset_index()
    d['tanggal'] = d['valid_time'].dt.normalize()
    d['jam'] = d['valid_time'].dt.hour
    n = [c for c in d.columns if c not in ('valid_time', 'kab', 'tanggal', 'jam')]
    d = d.set_index(['tanggal', 'kab', 'jam'])[n].unstack('jam')
    d.columns = [f'{v}_{j:02d}' for v, j in d.columns]
    d = d.reset_index()
    d['GID_2'] = studi['GID_2'].values[d['kab'].to_numpy()]
    return d.drop(columns='kab')

jenis = ['pl', 'sl', 'plx', 'slx']
for j in jenis:
    s = berkas_era[(j, 2015, 7)]
    t = LOKAL / s.name
    t.unlink(missing_ok=True)
    shutil.copy(s, t)
    print(j, 'drive:', s.stat().st_size, 'lokal:', t.stat().st_size)

try:
    baru = pivot(pd.concat([satu_berkas(LOKAL / berkas_era[(j, 2015, 7)].name) for j in jenis], axis=1))
    era2 = pd.concat([era, baru]).sort_values('tanggal', kind='stable').reset_index(drop=True)
    era2.to_csv(HASIL / 'era5_nasional_harian.csv', index=False)
    print('berhasil:', era2.shape, '| NaN tertinggi:', era2.isna().mean().max())
    g = era2.merge(lama_all, on=['tanggal', 'GID_2'], suffixes=('_b', '_l'))
    print('cocok dengan file lama:', len(g), 'dari', len(lama_all))
except Exception as e:
    print('masih gagal:', str(e)[:150])

pl drive: 11372083 lokal: 11372083
sl drive: 2308866 lokal: 2308866
plx drive: 6253505 lokal: 6253505
slx drive: 3308595 lokal: 3308595
berhasil: (908620, 52) | NaN tertinggi: 0.0
cocok dengan file lama: 68780 dari 68780


In [ ]:
!pip -q install cdsapi
import getpass
from pathlib import Path
import numpy as np, pandas as pd, xarray as xr, geopandas as gpd
import cdsapi

ROOT = Path('/content/drive/MyDrive/Logika Ui')
akar = ROOT / 'KTI_KARHUTLA'
HASIL = akar / 'hasil'

if not (Path.home() / '.cdsapirc').exists():
    token = getpass.getpass('CDS token: ')
    (Path.home() / '.cdsapirc').write_text(f'url: https://cds.climate.copernicus.eu/api\nkey: {token}\n')

# area dan level diambil dari berkas lama supaya sama persis
ref = xr.open_dataset(next(akar.rglob('pl_2020_06.nc')))
area = [float(ref.latitude.max()), float(ref.longitude.min()),
        float(ref.latitude.min()), float(ref.longitude.max())]
level = [str(int(x)) for x in ref.pressure_level.values]
print('area N,W,S,E:', area, '| level:', level)

dasar = {'product_type': ['reanalysis'], 'year': ['2026'], 'month': ['10'],
         'day': [f'{d:02d}' for d in range(1, 15)], 'time': ['00:00', '06:00'],
         'data_format': 'netcdf', 'download_format': 'unarchived', 'area': area}
paket = {
    'pl':  ('reanalysis-era5-pressure-levels',
            ['temperature', 'relative_humidity', 'u_component_of_wind', 'v_component_of_wind'], True),
    'plx': ('reanalysis-era5-pressure-levels', ['vertical_velocity', 'specific_humidity'], True),
    'sl':  ('reanalysis-era5-single-levels',
            ['convective_available_potential_energy', '2m_temperature', '2m_dewpoint_temperature'], False),
    'slx': ('reanalysis-era5-single-levels',
            ['total_column_water_vapour', 'total_column_cloud_liquid_water',
             'total_cloud_cover', 'surface_pressure'], False),
}
BARU = Path('/content/era5t_okt2026'); BARU.mkdir(exist_ok=True)
cl = cdsapi.Client()
for j, (dataset, var, pakai_level) in paket.items():
    tujuan = BARU / f'{j}_2026_10.nc'
    if tujuan.exists() and tujuan.stat().st_size > 0:
        continue
    req = dict(dasar, variable=var)
    if pakai_level:
        req['pressure_level'] = level
    cl.retrieve(dataset, req, str(tujuan))
    print(j, 'selesai', tujuan.stat().st_size)

# agregasi ke 502 wilayah (interpolasi ke titik pusat, sama seperti yang sudah tervalidasi)
gadm2 = gpd.read_file(ROOT / 'GADM 2' / 'gadm41_IDN_2.shp')
studi = gadm2.reset_index(drop=True)
pusat = studi.to_crs('EPSG:6933').geometry.centroid.to_crs('EPSG:4326')
lat_k = xr.DataArray(pusat.y.values, dims='kab')
lon_k = xr.DataArray(pusat.x.values, dims='kab')

def satu_berkas(path, jam=(0, 6)):
    d = xr.open_dataset(path)
    if 'expver' in d.dims:
        d = d.sel(expver=1).combine_first(d.sel(expver=5))
    d = d.sortby('latitude')
    d = d.sel(valid_time=d.valid_time.dt.hour.isin(jam))
    di = d.interp(latitude=lat_k, longitude=lon_k, method='linear')
    kolom = {}
    for v in di.data_vars:
        a = di[v]
        if 'pressure_level' in a.dims:
            for lv in a.pressure_level.values:
                kolom[f'{v}_{int(lv)}'] = a.sel(pressure_level=lv).transpose('valid_time', 'kab').values
        else:
            kolom[v] = a.transpose('valid_time', 'kab').values
    idx = pd.MultiIndex.from_product([di.valid_time.values, range(len(studi))], names=['valid_time', 'kab'])
    return pd.DataFrame({n: x.reshape(-1) for n, x in kolom.items()}, index=idx)

def pivot(df):
    d = df.reset_index()
    d['tanggal'] = d['valid_time'].dt.normalize()
    d['jam'] = d['valid_time'].dt.hour
    n = [c for c in d.columns if c not in ('valid_time', 'kab', 'tanggal', 'jam')]
    d = d.set_index(['tanggal', 'kab', 'jam'])[n].unstack('jam')
    d.columns = [f'{v}_{j:02d}' for v, j in d.columns]
    d = d.reset_index()
    d['GID_2'] = studi['GID_2'].values[d['kab'].to_numpy()]
    return d.drop(columns='kab')

baru = pivot(pd.concat([satu_berkas(BARU / f'{j}_2026_10.nc') for j in paket], axis=1))
lama = pd.read_csv(HASIL / 'era5_nasional_harian.csv', parse_dates=['tanggal'])
baru['tanggal'] = pd.to_datetime(baru['tanggal'])

ov = lama[lama['tanggal'] >= '2026-10-01'].merge(baru, on=['tanggal', 'GID_2'], suffixes=('_l', '_b'))
kol = [c for c in lama.columns if c not in ('tanggal', 'GID_2')]
print('tumpang tindih 1-5 Okt:', len(ov), '| selisih maks:', max((ov[c + '_l'] - ov[c + '_b']).abs().max() for c in kol))

era2 = pd.concat([lama[lama['tanggal'] < '2026-10-01'], baru[lama.columns]])
era2 = era2.sort_values('tanggal', kind='stable').reset_index(drop=True)
era2.to_csv(HASIL / 'era5_nasional_harian.csv', index=False)
print(era2.shape, '| tanggal terakhir:', era2['tanggal'].max().date(), '| NaN tertinggi:', era2.isna().mean().max())

CDS token: ··········
area N,W,S,E: [7.0, 95.0, -12.0, 142.0] | level: ['850', '700', '500']


2026-10-22 14:39:02,841 INFO [2026-10-06T00:00:00Z] The [CDS Terms of Use](https://cds.climate.copernicus.eu/disclaimer-privacy) will be updated on 28 October 2026. Continued use of Copernicus Products and Services through the CDS after this date constitutes acceptance of the updated terms. If you do not accept the change, please [contact ECMWF Support](https://jira.ecmwf.int/plugins/servlet/desk/portal/1/create/202) before 27 October 2026.
 **[More information about the upcoming change.](https://forum.ecmwf.int/t/upcoming-cds-terms-of-use-change/15374)**
INFO:ecmwf.datastores.legacy_client:[2026-10-06T00:00:00Z] The [CDS Terms of Use](https://cds.climate.copernicus.eu/disclaimer-privacy) will be updated on 28 October 2026. Continued use of Copernicus Products and Services through the CDS after this date constitutes acceptance of the updated terms. If you do not accept the change, please [contact ECMWF Support](https://jira.ecmwf.int/plugins/servlet/desk/portal/1/create/202) before 27 

fa67a8119666fb2bc3505c293af385e4.nc:   0%|          | 0.00/8.75M [00:00<?, ?B/s]

pl selesai 9174357


2026-10-22 14:39:06,124 INFO Request ID is 83443bb2-9ca4-4a1b-9724-4446caea2e70
INFO:ecmwf.datastores.legacy_client:Request ID is 83443bb2-9ca4-4a1b-9724-4446caea2e70
2026-10-22 14:39:06,269 INFO status has been updated to successful
INFO:ecmwf.datastores.legacy_client:status has been updated to successful


...

1c14d88ed6bbd0497bd131b02538f358.nc:   0%|          | 0.00/1.85M [00:00<?, ?B/s]

sl selesai 1939401


2026-10-22 14:39:11,355 INFO Request ID is 857316fd-3647-4d2c-a03f-8ef50b7829b9
INFO:ecmwf.datastores.legacy_client:Request ID is 857316fd-3647-4d2c-a03f-8ef50b7829b9
2026-10-22 14:39:11,561 INFO status has been updated to successful
INFO:ecmwf.datastores.legacy_client:status has been updated to successful


b4873fb0acbe8e02ee2a4bc36e061cd2.nc:   0%|          | 0.00/2.69M [00:00<?, ?B/s]

slx selesai 2819762
tumpang tindih 1-5 Okt: 2510 | selisih maks: 4171.78365496613
(913138, 52) | tanggal terakhir: 2026-10-14 | NaN tertinggi: 0.0


In [ ]:
from pathlib import Path
import numpy as np, pandas as pd, geopandas as gpd

ROOT = Path('/content/drive/MyDrive/Logika Ui')
HASIL = ROOT / 'KTI_KARHUTLA' / 'hasil'
gadm2 = gpd.read_file(ROOT / 'GADM 2' / 'gadm41_IDN_2.shp')

urutan = pd.read_csv(HASIL / 'tabel_gabungan_v1.csv', nrows=0).columns.tolist()
kolom_era = [c for c in urutan if c.endswith(('_00', '_06'))]

im = pd.read_csv(HASIL / 'imerg_nasional_harian.csv', parse_dates=['tanggal'])
im = im.sort_values('run').drop_duplicates(['tanggal', 'GID_2'], keep='first').reset_index(drop=True)
era = pd.read_csv(HASIL / 'era5_nasional_harian.csv', parse_dates=['tanggal'])
him = pd.read_csv(HASIL / 'himawari_prov_harian.csv', parse_dates=['tanggal'])
fir = pd.read_csv(HASIL / 'firms_nasional_harian.csv', parse_dates=['tanggal'])
print('provinsi di Himawari:', sorted(him['provinsi'].unique()))

nama = gadm2[['GID_2', 'NAME_1', 'NAME_2']]
t = (im.merge(nama, on='GID_2', how='left')
       .merge(era[['tanggal', 'GID_2'] + kolom_era], on=['tanggal', 'GID_2'], how='left')
       .merge(him.rename(columns={'provinsi': 'NAME_1'}), on=['tanggal', 'NAME_1'], how='left')
       .merge(fir, on=['tanggal', 'GID_2'], how='left'))
print('baris IMERG vs tabel (harus sama):', len(im), len(t))

batas = fir['tanggal'].max()
for c in ['titik_panas', 'frp_total', 'frp_maks']:
    t.loc[t['tanggal'] <= batas, c] = t.loc[t['tanggal'] <= batas, c].fillna(0)
t['tahun'] = t['tanggal'].dt.year
t['bulan'] = t['tanggal'].dt.month
t = t[urutan]
t.to_csv(HASIL / 'tabel_gabungan_nasional.csv', index=False)

kol_him = [c for c in t.columns if c.startswith('him_')]
print(t.shape, '|', t['GID_2'].nunique(), 'wilayah |', t['tanggal'].min().date(), 'sampai', t['tanggal'].max().date())
print('baris tanpa fitur Himawari:', round(t[kol_him].isna().all(axis=1).mean(), 3))
hilang = gadm2[~gadm2['GID_2'].isin(t['GID_2'])]
print('wilayah GADM tidak masuk tabel:', len(hilang), hilang['NAME_2'].tolist()[:25])

# validasi: 38 kab Sumatera harus sama dengan v2 (sebelum 1 Okt 2026, karena ERA5T bisa direvisi)
v2 = pd.read_csv(HASIL / 'tabel_gabungan_v2.csv', parse_dates=['tanggal'])
a = v2[v2['tanggal'] < '2026-10-01'].set_index(['tanggal', 'GID_2']).sort_index()
b = t.set_index(['tanggal', 'GID_2']).sort_index()
ada = a.index.intersection(b.index)
print(len(a), 'baris v2 |', len(ada), 'ada di nasional')
beda = []
for c in a.columns:
    x, y = a.loc[ada, c], b.loc[ada, c]
    if x.dtype.kind in 'fi':
        ok = np.allclose(x.to_numpy(float), y.to_numpy(float), equal_nan=True, rtol=1e-6, atol=1e-9)
    else:
        ok = (x.fillna('~').astype(str) == y.fillna('~').astype(str)).all()
    if not ok:
        beda.append(c)
print('kolom yang beda dari v2:', beda, '(harapan [])')

provinsi di Himawari: ['Jambi', 'Riau', 'Sumatera Selatan']
baris IMERG vs tabel (harus sama): 504866 504866
(504866, 77) | 479 wilayah | 2020-06-01 sampai 2026-10-14
baris tanpa fitur Himawari: 0.928
wilayah GADM tidak masuk tabel: 23 ['Danau Limboto', 'Jakarta Pusat', 'Jakarta Timur', 'Kepulauan Seribu', 'Waduk Cirata', 'Kota Magelang', 'Kota Pekalongan', 'Salatiga', 'Waduk Kedungombo', 'Kota Blitar', 'Kota Madiun', 'Kota Pasuruan', 'Kota Probolinggo', 'Banjarmasin', 'Metro', 'Mataram', 'Bukittinggi', 'Padang Panjang', 'Payakumbuh', 'Kota Tanjungbalai', 'Sibolga', 'Tebingtinggi', 'Kota Yogyakarta']
39520 baris v2 | 39520 ada di nasional
kolom yang beda dari v2: [] (harapan [])


In [ ]:
!pip -q install s3fs h5netcdf
import time
import numpy as np, xarray as xr, s3fs

fs = s3fs.S3FileSystem(anon=True)
prefix = 'noaa-himawari9/AHI-L2-FLDK-Clouds/2024/09/15'
files = fs.find(prefix)
tinggi = [p for p in files if p.split('/')[-2] == '0500' and 'CHGT' in p.split('/')[-1]][0]
print(tinggi)

with fs.open(tinggi, 'rb') as f:
    d = xr.open_dataset(f, engine='h5netcdf')
    for k, v in d.variables.items():
        print(k, v.dims, v.shape, v.dtype)
    for k, v in d.variables.items():
        if 'grid_mapping_name' in v.attrs:
            print('PROYEKSI', k, v.attrs)
    print({k: d.attrs[k] for k in d.attrs if any(s in k.lower() for s in ('proj', 'lon', 'lat', 'extent', 'resolution'))})

def waktu(sl, nama):
    with fs.open(tinggi, 'rb') as f:   # berkas dibuka baru tiap kali supaya cache tidak membantu
        with xr.open_dataset(f, engine='h5netcdf') as d:
            t0 = time.time()
            a = d['CldTopTemp'].isel(**sl).values
            t = time.time() - t0
    print(f'{nama}: {t:.1f} dtk, bentuk {a.shape}')
    return t

t_sum = waktu(dict(Rows=slice(2590, 3017), Columns=slice(765, 1045)), 'jendela Sumatera')
t_nas = waktu(dict(Rows=slice(2380, 3400), Columns=slice(580, 2810)), 'jendela nasional (perkiraan kasar)')
print('rasio nasional / Sumatera:', round(t_nas / t_sum, 1))

noaa-himawari9/AHI-L2-FLDK-Clouds/2024/09/15/0500/AHI-CHGT_v1r1_h09_s202409150500208_e202409150509402_c202409150517593.nc
StartRow () () int32
StartColumn () () int32
AchaMode () () float64
NumOfQualityFlag () () float32
Shadow_Mask ('Rows', 'Columns') (5500, 5500) float32
Latitude_Pc ('Rows', 'Columns') (5500, 5500) float32
Longitude_Pc ('Rows', 'Columns') (5500, 5500) float32
CldTopTemp ('Rows', 'Columns') (5500, 5500) float32
CldTopTempLow ('Rows', 'Columns') (5500, 5500) float32
CldTopPres ('Rows', 'Columns') (5500, 5500) float32
CldTopPresLow ('Rows', 'Columns') (5500, 5500) float32
CldTopHght ('Rows', 'Columns') (5500, 5500) float32
CldTopEmss ('Rows', 'Columns') (5500, 5500) float32
CldOptDpth ('Rows', 'Columns') (5500, 5500) float32
CloudHgtQF ('Rows', 'Columns') (5500, 5500) float32
CldHgtFlag ('Rows', 'Columns', 'CldHgtFlgCnst') (5500, 5500, 5) float32
ProcOrder ('Rows', 'Columns') (5500, 5500) float32
Cost ('Rows', 'Columns') (5500, 5500) float32
InverFlag ('Rows', 'Columns'

In [ ]:
!pip -q install s3fs h5netcdf regionmask
import time
from pathlib import Path
import numpy as np, xarray as xr, s3fs, geopandas as gpd, regionmask

ROOT = Path('/content/drive/MyDrive/Logika Ui')
fs = s3fs.S3FileSystem(anon=True)
prefix = 'noaa-himawari9/AHI-L2-FLDK-Clouds/2024/09/15'
tinggi = [p for p in fs.find(prefix) if p.split('/')[-2] == '0500' and 'CHGT' in p.split('/')[-1]][0]

R0, R1, C0, C1 = 2380, 3400, 580, 2810
t0 = time.time()
with fs.open(tinggi, 'rb') as f:
    with xr.open_dataset(f, engine='h5netcdf') as d:
        sl = dict(Rows=slice(R0, R1), Columns=slice(C0, C1))
        lat = d['Latitude'].isel(**sl).values
        lon = d['Longitude'].isel(**sl).values
print('baca lat/lon:', round(time.time() - t0), 'dtk')
print('lat', np.nanmin(lat), np.nanmax(lat), '| lon', np.nanmin(lon), np.nanmax(lon))

gadm2 = gpd.read_file(ROOT / 'GADM 2' / 'gadm41_IDN_2.shp')
prov = gadm2[['NAME_1', 'geometry']].dissolve(by='NAME_1').reset_index()
regions = regionmask.from_geopandas(prov, names='NAME_1')
nama = prov['NAME_1'].tolist()

t0 = time.time()
m = regions.mask(lon, lat)
m = np.asarray(m)
print('mask selesai:', round(time.time() - t0), 'dtk')

print('provinsi:', len(nama), '| piksel berprovinsi:', int(np.isfinite(m).sum()))
for n in ['Riau', 'Jambi', 'Sumatera Selatan']:
    print(n, int((m == nama.index(n)).sum()))
print('mask lama: Riau 14539, Jambi 8078, Sumatera Selatan 14805')

rr, cc = np.where(np.isfinite(m))
ra, ca = rr + R0, cc + C0
blok = set(zip((ra // 200).tolist(), (ca // 200).tolist()))
print('baris', ra.min(), 'sampai', ra.max() + 1, '| kolom', ca.min(), 'sampai', ca.max() + 1)
print('potongan 200x200 yang berisi daratan:', len(blok), '(jendela Sumatera lama: 12)')

mk = np.where(np.isfinite(m), m, -1).astype('int16')
np.savez_compressed(ROOT / 'prov_mask_nasional.npz', mask=mk, R0=R0, C0=C0, nama=np.array(nama))
print('tersimpan: prov_mask_nasional.npz')

baca lat/lon: 60 dtk
lat -12.675493 7.135721 | lon 92.6202 141.7965
mask selesai: 5 dtk
provinsi: 34 | piksel berprovinsi: 388636
Riau 14539
Jambi 8078
Sumatera Selatan 14805
mask lama: Riau 14539, Jambi 8078, Sumatera Selatan 14805
baris 2443 sampai 3347 | kolom 607 sampai 2768
potongan 200x200 yang berisi daratan: 45 (jendela Sumatera lama: 12)
tersimpan: prov_mask_nasional.npz


In [ ]:
%%writefile himawari_nasional.py
import time
import numpy as np
import xarray as xr
import s3fs

R0, R1, C0, C1 = 2400, 3400, 600, 2800
SL = dict(Rows=slice(R0, R1), Columns=slice(C0, C1))
KUNCI_FASE = ("CPHS", "CLOUD_PHASE")
KUNCI_TINGGI = ("CHGT", "CLOUD_HEIGHT")
AMBANG = (273.15, 253.15, 233.15)

_fs = None
_idx = None
_nama = None


def init(path_mask):
    global _fs, _idx, _nama
    _fs = s3fs.S3FileSystem(anon=True, skip_instance_cache=True)
    z = np.load(path_mask)
    r0, c0 = int(z["R0"]), int(z["C0"])
    m = z["mask"][R0 - r0:R1 - r0, C0 - c0:C1 - c0]
    _nama = [str(n) for n in z["nama"]]
    flat = m.ravel()
    _idx = [np.flatnonzero(flat == k) for k in range(len(_nama))]


def _baca(path, var, coba=3):
    for i in range(coba):
        try:
            with _fs.open(path, "rb") as f:
                with xr.open_dataset(f, engine="h5netcdf") as d:
                    return d[var].isel(**SL).values
        except FileNotFoundError:
            raise
        except Exception:
            if i == coba - 1:
                raise
            time.sleep(2 * (i + 1))


def _peta_hari(sat, tgl):
    y, m, d = tgl.split("-")
    prefix = f"noaa-{sat}/AHI-L2-FLDK-Clouds/{y}/{m}/{d}"
    peta = {}
    for p in _fs.find(prefix):
        bagian = p.split("/")
        slot, nama = bagian[-2], bagian[-1]
        if any(k in nama for k in KUNCI_FASE):
            peta.setdefault(slot, {})["fase"] = p
        elif any(k in nama for k in KUNCI_TINGGI):
            peta.setdefault(slot, {})["tinggi"] = p
    return peta


def _agregat(phase, tt):
    pf, tf = phase.ravel(), tt.ravel()
    baris = []
    for k, nama in enumerate(_nama):
        ix = _idx[k]
        if ix.size == 0:
            continue
        p, t = pf[ix], tf[ix]
        tv = t[np.isfinite(t)]
        r = {"provinsi": nama, "n_piksel": int(ix.size),
             "n_suhu_valid": int(tv.size), "fase_nan": int(np.isnan(p).sum())}
        for j in range(6):
            r[f"fase{j}"] = int((p == j).sum())
        if tv.size:
            r.update(suhu_mean=float(tv.mean()), suhu_p10=float(np.percentile(tv, 10)),
                     suhu_p50=float(np.percentile(tv, 50)), suhu_min=float(tv.min()))
        else:
            r.update(suhu_mean=np.nan, suhu_p10=np.nan, suhu_p50=np.nan, suhu_min=np.nan)
        for a in AMBANG:
            r[f"n_suhu_lt_{int(a)}"] = int((tv < a).sum())
        baris.append(r)
    return baris


def proses_hari(args):
    sat, tgl, slots = args
    rows, log = [], []
    try:
        peta = _peta_hari(sat, tgl)
    except Exception as e:
        err = f"GALAT daftar hari: {type(e).__name__}: {e}"[:150]
        return [], [(sat, tgl, s, err, 0.0) for s in slots]
    if not peta:
        return [], [(sat, tgl, s, "hari tidak ada", 0.0) for s in slots]
    for s in slots:
        t1 = time.time()
        info = peta.get(s)
        if info is None:
            log.append((sat, tgl, s, "slot tidak ada", 0.0))
            continue
        if "fase" not in info or "tinggi" not in info:
            log.append((sat, tgl, s, "file fase atau tinggi tidak lengkap", 0.0))
            continue
        try:
            phase = _baca(info["fase"], "CloudPhase")
            tt = _baca(info["tinggi"], "CldTopTemp")
            for r in _agregat(phase, tt):
                r.update(tanggal=tgl, waktu=s, satelit=sat,
                         file_fase=info["fase"].split("/")[-1],
                         file_tinggi=info["tinggi"].split("/")[-1])
                rows.append(r)
            log.append((sat, tgl, s, "ok", round(time.time() - t1, 1)))
        except Exception as e:
            log.append((sat, tgl, s, f"GALAT: {type(e).__name__}: {e}"[:150],
                        round(time.time() - t1, 1)))
    return rows, log

Overwriting himawari_nasional.py


In [ ]:
import sys, time, importlib
from pathlib import Path
import numpy as np, pandas as pd
sys.path.insert(0, '/content')
import himawari_nasional as hn
importlib.reload(hn)

ROOT = Path('/content/drive/MyDrive/Logika Ui')
hn.init(str(ROOT / 'prov_mask_nasional.npz'))
print('provinsi:', len(hn._nama), '| piksel berprovinsi:', sum(i.size for i in hn._idx))

t0 = time.time()
rows, log = hn.proses_hari(('himawari9', '2024-09-15', ('0500',)))
print(log, '|', round(time.time() - t0), 'dtk')
baru = pd.DataFrame(rows)
print(baru.shape)

folder = ROOT / 'Himawari_v2'
if not folder.exists():
    folder = ROOT / 'Himawari'
lama = pd.read_csv(folder / 'himawari_2024_09.csv')
lama = lama[(lama['tanggal'] == '2024-09-15') & (lama['waktu'].astype(int) == 500)]
kol = [c for c in lama.columns if c in baru.columns
       and c not in ('tanggal', 'waktu', 'satelit', 'provinsi', 'file_fase', 'file_tinggi')]
g = baru.merge(lama, on='provinsi', suffixes=('_b', '_l'))
print('provinsi cocok:', len(g), 'dari 3')
selisih = pd.Series({c: np.nanmax(np.abs(g[c + '_b'].astype(float) - g[c + '_l'].astype(float))) for c in kol})
print(selisih.sort_values(ascending=False).head(6))

provinsi: 34 | piksel berprovinsi: 388636
[('himawari9', '2024-09-15', '0500', 'ok', 28.2)] | 30 dtk
(34, 22)
provinsi cocok: 3 dari 3
suhu_p10        5.684342e-14
suhu_p50        2.842171e-14
suhu_min        2.842171e-14
n_piksel        0.000000e+00
fase1           0.000000e+00
n_suhu_valid    0.000000e+00
dtype: float64


In [ ]:
!pip -q install s3fs h5netcdf
from google.colab import drive
drive.mount('/content/drive')

import os, sys, time, shutil, multiprocessing as mp
from pathlib import Path
import numpy as np, pandas as pd
from concurrent.futures import ProcessPoolExecutor

ROOT = Path('/content/drive/MyDrive/Logika Ui')
MASK = str(ROOT / 'prov_mask_nasional.npz')
OUTDIR = ROOT / 'Himawari_nasional'
OUTDIR.mkdir(exist_ok=True)

# modul disimpan di Drive supaya tahan restart
lokal = Path('/content/himawari_nasional.py')
cadangan = ROOT / 'himawari_nasional.py'
if lokal.exists():
    shutil.copy(lokal, cadangan)
else:
    shutil.copy(cadangan, lokal)
sys.path.insert(0, '/content')
import himawari_nasional as hn

SLOTS = ("0300", "0400", "0500", "0600", "0700", "0800", "0900")
N_PROSES = 16          # kalau RAM Colab habis atau sesi crash, turunkan ke 12
HARI_PER_BATCH = 16
BULAN = range(1, 13)

H8 = (pd.Timestamp('2019-12-10'), pd.Timestamp('2022-12-13'))
H9 = (pd.Timestamp('2023-08-16'), pd.Timestamp('2026-10-14'))

def sat_untuk(d):
    if H8[0] <= d <= H8[1]: return 'himawari8'
    if H9[0] <= d <= H9[1]: return 'himawari9'
    return None

PERIODE = sorted({(d.year, d.month) for d in pd.date_range(H8[0], H9[1])
                  if sat_untuk(d) and d.month in BULAN})

def jalankan_bulan(th, bln):
    awal = pd.Timestamp(th, bln, 1)
    hari = [d for d in pd.date_range(awal, awal + pd.offsets.MonthEnd(0)) if sat_untuk(d)]
    for bi, b in enumerate(range(0, len(hari), HARI_PER_BATCH)):
        grup = hari[b:b + HARI_PER_BATCH]
        out = OUTDIR / f'himawari_{th}_{bln:02d}_{bi:02d}.csv'
        logf = OUTDIR / f'log_{th}_{bln:02d}_{bi:02d}.csv'
        if logf.exists():
            continue
        tugas = [(sat_untuk(d), f'{d:%Y-%m-%d}', SLOTS) for d in grup]
        t0 = time.time()
        with ProcessPoolExecutor(N_PROSES, mp_context=mp.get_context('spawn'),
                                 initializer=hn.init, initargs=(MASK,)) as ex:
            hasil = list(ex.map(hn.proses_hari, tugas, chunksize=1))
        rows = [r for rs, _ in hasil for r in rs]
        log = [l for _, ls in hasil for l in ls]
        lg = pd.DataFrame(log, columns=['satelit', 'tanggal', 'waktu', 'status', 'detik'])
        if rows:
            pd.DataFrame(rows).to_csv(out, index=False)
        lg.to_csv(logf, index=False)   # ditulis terakhir = tanda batch selesai
        galat = int(lg.status.str.startswith('GALAT').sum())
        ok = int((lg.status == 'ok').sum())
        print(f'{th}-{bln:02d} batch {bi} selesai {(time.time() - t0) / 60:.1f} menit | ok {ok}/{len(lg)} | galat {galat}', flush=True)

print(len(PERIODE), 'bulan:', PERIODE[0], 'sampai', PERIODE[-1])
t_mulai = time.time()
for th, bln in PERIODE:
    jalankan_bulan(th, bln)
print(f'SELESAI. Total {(time.time() - t_mulai) / 3600:.1f} jam')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
76 bulan: (2019, 12) sampai (2026, 10)
2026-08 batch 0 selesai 15.5 menit | ok 112/112 | galat 0
2026-08 batch 1 selesai 14.0 menit | ok 105/105 | galat 0
2026-09 batch 0 selesai 14.1 menit | ok 112/112 | galat 0
2026-09 batch 1 selesai 15.6 menit | ok 98/98 | galat 0
2026-10 batch 0 selesai 12.6 menit | ok 98/98 | galat 0
SELESAI. Total 1.2 jam


In [ ]:
from pathlib import Path
import pandas as pd

ROOT = Path('/content/drive/MyDrive/Logika Ui')
log = pd.concat([pd.read_csv(p) for p in sorted((ROOT / 'Himawari_nasional').glob('log_*.csv'))], ignore_index=True)
log['tanggal'] = pd.to_datetime(log['tanggal'])
ok_per_hari = log.assign(ok=log['status'].eq('ok')).groupby('tanggal')['ok'].sum()

kosong = ok_per_hari[ok_per_hari == 0]
grup = kosong.index.to_series().diff().dt.days.ne(1).cumsum()
print('hari tercatat:', len(ok_per_hari), '| hari kosong total:', len(kosong))
for _, g in kosong.groupby(grup):
    print(g.index.min().date(), 'sampai', g.index.max().date(), '|', len(g), 'hari')

parsial = ok_per_hari[(ok_per_hari > 0) & (ok_per_hari < 7)]
print('hari dengan sebagian slot saja:', len(parsial))

hari tercatat: 2256 | hari kosong total: 49
2019-12-10 sampai 2019-12-10 | 1 hari
2022-09-10 sampai 2022-09-10 | 1 hari
2023-08-16 sampai 2023-08-16 | 1 hari
2024-10-12 sampai 2024-10-12 | 1 hari
2025-10-12 sampai 2025-11-25 | 45 hari
hari dengan sebagian slot saja: 52


In [ ]:
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

OUT = HASIL / 'era5_semua_bulan'
OUT.mkdir(exist_ok=True)

def salin(src):
    tujuan = LOKAL / src.name
    galat = None
    for _ in range(3):
        try:
            tujuan.unlink(missing_ok=True)
            shutil.copy(src, tujuan)
            if tujuan.stat().st_size == src.stat().st_size:
                return tujuan
        except Exception as e:
            galat = e
    raise RuntimeError(f'salin gagal {src.name}: {galat}')

jenis = ['pl', 'sl', 'plx', 'slx']
gagal = []
for tahun in range(2015, 2027):
    f = OUT / f'era5_nasional_{tahun}.csv'
    if f.exists():
        continue
    bagian = []
    for bulan in range(1, 13):
        kunci = [(j, tahun, bulan) for j in jenis]
        if not all(k in berkas_era for k in kunci):
            if not (tahun == 2026 and bulan > 10):
                gagal.append((tahun, bulan, 'berkas tidak ada'))
            continue
        lok = [salin(berkas_era[k]) for k in kunci]   # kalau Drive putus lagi, langsung berhenti dengan pesan asli
        bagian.append(pivot(pd.concat([satu_berkas(p) for p in lok], axis=1)))
        for p in lok:
            p.unlink(missing_ok=True)
    pd.concat(bagian).to_csv(f, index=False)
    print(tahun, 'tersimpan', flush=True)
print('bermasalah:', gagal)

era = pd.concat([pd.read_csv(p, parse_dates=['tanggal']) for p in sorted(OUT.glob('era5_nasional_*.csv'))],
                ignore_index=True)
era = era.sort_values('tanggal', kind='stable').reset_index(drop=True)
print(era.shape, '| tanggal:', era['tanggal'].min().date(), 'sampai', era['tanggal'].max().date(),
      '| NaN tertinggi:', era.isna().mean().max())

lama = pd.read_csv(HASIL / 'era5_nasional_harian.csv', parse_dates=['tanggal'])
g = era.merge(lama, on=['tanggal', 'GID_2'], suffixes=('_b', '_l'))
kol = [c for c in lama.columns if c not in ('tanggal', 'GID_2')]
print('baris cocok dengan file lama:', len(g), 'dari', len(lama),
      '| selisih maks:', max((g[c + '_b'] - g[c + '_l']).abs().max() for c in kol))
era.to_csv(HASIL / 'era5_nasional_harian_semua_bulan.csv', index=False)
print('tersimpan: era5_nasional_harian_semua_bulan.csv')

Mounted at /content/drive
2015 tersimpan
2016 tersimpan
2017 tersimpan
2018 tersimpan
2019 tersimpan
2020 tersimpan
2021 tersimpan
2022 tersimpan
2023 tersimpan
2024 tersimpan
2025 tersimpan
2026 tersimpan
bermasalah: []
(2156592, 52) | tanggal: 2015-01-01 sampai 2026-10-05 | NaN tertinggi: 0.0
baris cocok dengan file lama: 908620 dari 913138 | selisih maks: 4171.783654966115
tersimpan: era5_nasional_harian_semua_bulan.csv


In [ ]:
g['bulan'] = g['tanggal'].dt.to_period('M')
sel = pd.DataFrame({c: (g[c + '_b'] - g[c + '_l']).abs() for c in kol})
sel['bulan'] = g['bulan'].values
per_bulan = sel.groupby('bulan').max().max(axis=1)
print('bulan dengan selisih > 1e-6:')
print(per_bulan[per_bulan > 1e-6])
print('kolom terburuk:')
print(sel.drop(columns='bulan').max().sort_values(ascending=False).head(5))
print('bulan yang cocok bersih:', int((per_bulan <= 1e-6).sum()), 'dari', len(per_bulan))

bulan dengan selisih > 1e-6:
bulan
2026-10    4171.783655
Freq: M, dtype: float64
kolom terburuk:
sp_06       4171.783655
sp_00       4146.576489
cape_06     1354.732709
cape_00      957.190209
r_700_06      27.778980
dtype: float64
bulan yang cocok bersih: 59 dari 60


In [ ]:
from google.colab import drive
drive.mount('/content/drive')
!pip -q install cdsapi
import getpass, shutil
from pathlib import Path
import numpy as np, pandas as pd, xarray as xr, geopandas as gpd
import cdsapi

ROOT = Path('/content/drive/MyDrive/Logika Ui')
akar = ROOT / 'KTI_KARHUTLA'
HASIL = akar / 'hasil'

if not (Path.home() / '.cdsapirc').exists():
    token = getpass.getpass('CDS token: ')
    (Path.home() / '.cdsapirc').write_text(f'url: https://cds.climate.copernicus.eu/api\nkey: {token}\n')

# agregasi ke 502 wilayah (interpolasi ke titik pusat, cara yang sudah tervalidasi)
gadm2 = gpd.read_file(ROOT / 'GADM 2' / 'gadm41_IDN_2.shp')
studi = gadm2.reset_index(drop=True)
pusat = studi.to_crs('EPSG:6933').geometry.centroid.to_crs('EPSG:4326')
lat_k = xr.DataArray(pusat.y.values, dims='kab')
lon_k = xr.DataArray(pusat.x.values, dims='kab')

def satu_berkas(path, jam=(0, 6)):
    d = xr.open_dataset(path)
    if 'expver' in d.dims:
        d = d.sel(expver=1).combine_first(d.sel(expver=5))
    d = d.sortby('latitude')
    d = d.sel(valid_time=d.valid_time.dt.hour.isin(jam))
    di = d.interp(latitude=lat_k, longitude=lon_k, method='linear')
    kolom = {}
    for v in di.data_vars:
        a = di[v]
        if 'pressure_level' in a.dims:
            for lv in a.pressure_level.values:
                kolom[f'{v}_{int(lv)}'] = a.sel(pressure_level=lv).transpose('valid_time', 'kab').values
        else:
            kolom[v] = a.transpose('valid_time', 'kab').values
    idx = pd.MultiIndex.from_product([di.valid_time.values, range(len(studi))], names=['valid_time', 'kab'])
    return pd.DataFrame({n: x.reshape(-1) for n, x in kolom.items()}, index=idx)

def pivot(df):
    d = df.reset_index()
    d['tanggal'] = d['valid_time'].dt.normalize()
    d['jam'] = d['valid_time'].dt.hour
    n = [c for c in d.columns if c not in ('valid_time', 'kab', 'tanggal', 'jam')]
    d = d.set_index(['tanggal', 'kab', 'jam'])[n].unstack('jam')
    d.columns = [f'{v}_{j:02d}' for v, j in d.columns]
    d = d.reset_index()
    d['GID_2'] = studi['GID_2'].values[d['kab'].to_numpy()]
    return d.drop(columns='kab')

era = pd.read_csv(HASIL / 'era5_nasional_harian_semua_bulan.csv', parse_dates=['tanggal'])
print('file ERA5 semua bulan:', era.shape, '| tanggal terakhir:', era['tanggal'].max().date())

# unduh ulang Okt 2026 dengan grid yang sama seperti berkas lama
ref = xr.open_dataset(next(akar.rglob('pl_2020_06.nc')))
area = [float(ref.latitude.max()), float(ref.longitude.min()),
        float(ref.latitude.min()), float(ref.longitude.max())]
level = [str(int(x)) for x in ref.pressure_level.values]
res = round(float(np.abs(np.diff(ref.latitude.values)).mean()), 3)
print('area:', area, '| level:', level, '| grid berkas lama:', res, 'derajat')

dasar = {'product_type': ['reanalysis'], 'year': ['2026'], 'month': ['10'],
         'day': [f'{d:02d}' for d in range(1, 15)], 'time': ['00:00', '06:00'],
         'data_format': 'netcdf', 'download_format': 'unarchived',
         'area': area, 'grid': [str(res), str(res)]}
paket = {
    'pl':  ('reanalysis-era5-pressure-levels',
            ['temperature', 'relative_humidity', 'u_component_of_wind', 'v_component_of_wind'], True),
    'plx': ('reanalysis-era5-pressure-levels', ['vertical_velocity', 'specific_humidity'], True),
    'sl':  ('reanalysis-era5-single-levels',
            ['convective_available_potential_energy', '2m_temperature', '2m_dewpoint_temperature'], False),
    'slx': ('reanalysis-era5-single-levels',
            ['total_column_water_vapour', 'total_column_cloud_liquid_water',
             'total_cloud_cover', 'surface_pressure'], False),
}
BARU = Path('/content/era5t_okt2026_grid')
shutil.rmtree(BARU, ignore_errors=True)
BARU.mkdir()
cl = cdsapi.Client()
for j, (dataset, var, pakai_level) in paket.items():
    req = dict(dasar, variable=var)
    if pakai_level:
        req['pressure_level'] = level
    cl.retrieve(dataset, req, str(BARU / f'{j}_2026_10.nc'))
    print(j, 'selesai', flush=True)

d0 = xr.open_dataset(BARU / 'pl_2026_10.nc')
print('titik grid baru:', d0.latitude.size, d0.longitude.size, '| berkas lama:', ref.latitude.size, ref.longitude.size)
assert (d0.latitude.size, d0.longitude.size) == (ref.latitude.size, ref.longitude.size), 'grid masih beda, jangan lanjut'

baru = pivot(pd.concat([satu_berkas(BARU / f'{j}_2026_10.nc') for j in paket], axis=1))
baru['tanggal'] = pd.to_datetime(baru['tanggal'])
kol = [c for c in era.columns if c not in ('tanggal', 'GID_2')]
raw = era[era['tanggal'] >= '2026-10-01']
ov = raw.merge(baru, on=['tanggal', 'GID_2'], suffixes=('_r', '_b'))
sel = pd.Series({c: (ov[c + '_r'] - ov[c + '_b']).abs().max() for c in kol})
print('tumpang tindih 1-5 Okt:', len(ov), '(harapan 2510)')
print(sel.sort_values(ascending=False).head(5))

if sel[['sp_00', 'sp_06']].max() < 100:
    era_final = pd.concat([era[era['tanggal'] < '2026-10-01'], baru[era.columns]])
    era_final = era_final.sort_values('tanggal', kind='stable').reset_index(drop=True)
    era_final.to_csv(HASIL / 'era5_nasional_harian_semua_bulan.csv', index=False)
    print('tersimpan:', era_final.shape, era_final['tanggal'].max().date(),
          '| NaN tertinggi:', era_final.isna().mean().max())
else:
    print('TIDAK disimpan: selisih sp masih besar, kirim hasil print di atas')

Mounted at /content/drive
file ERA5 semua bulan: (2156592, 52) | tanggal terakhir: 2026-10-05
area: [7.0, 95.0, -12.0, 142.0] | level: ['850', '700', '500'] | grid berkas lama: 0.5 derajat


2026-10-23 23:16:24,125 INFO [2026-10-06T00:00:00Z] The [CDS Terms of Use](https://cds.climate.copernicus.eu/disclaimer-privacy) will be updated on 28 October 2026. Continued use of Copernicus Products and Services through the CDS after this date constitutes acceptance of the updated terms. If you do not accept the change, please [contact ECMWF Support](https://jira.ecmwf.int/plugins/servlet/desk/portal/1/create/202) before 27 October 2026.
 **[More information about the upcoming change.](https://forum.ecmwf.int/t/upcoming-cds-terms-of-use-change/15374)**
INFO:ecmwf.datastores.legacy_client:[2026-10-06T00:00:00Z] The [CDS Terms of Use](https://cds.climate.copernicus.eu/disclaimer-privacy) will be updated on 28 October 2026. Continued use of Copernicus Products and Services through the CDS after this date constitutes acceptance of the updated terms. If you do not accept the change, please [contact ECMWF Support](https://jira.ecmwf.int/plugins/servlet/desk/portal/1/create/202) before 27 

721e21050b320eb8d2d25437d4ec316e.nc:   0%|          | 0.00/2.51M [00:00<?, ?B/s]

pl selesai


2026-10-23 23:18:22,186 INFO Request ID is 9c18fd19-0d11-4f5c-89e6-df435f63b767
INFO:ecmwf.datastores.legacy_client:Request ID is 9c18fd19-0d11-4f5c-89e6-df435f63b767
2026-10-23 23:18:22,338 INFO status has been updated to accepted
INFO:ecmwf.datastores.legacy_client:status has been updated to accepted
2026-10-23 23:19:38,661 INFO status has been updated to running
INFO:ecmwf.datastores.legacy_client:status has been updated to running
2026-10-23 23:20:17,270 INFO status has been updated to successful
INFO:ecmwf.datastores.legacy_client:status has been updated to successful


...

843e04b0b1b6204efd88f491dd3cd3fb.nc:   0%|          | 0.00/545k [00:00<?, ?B/s]

sl selesai


2026-10-23 23:21:40,929 INFO Request ID is a94c0190-18ad-4f14-867a-5309439159eb
INFO:ecmwf.datastores.legacy_client:Request ID is a94c0190-18ad-4f14-867a-5309439159eb
2026-10-23 23:21:41,082 INFO status has been updated to accepted
INFO:ecmwf.datastores.legacy_client:status has been updated to accepted
2026-10-23 23:22:14,475 INFO status has been updated to running
INFO:ecmwf.datastores.legacy_client:status has been updated to running
2026-10-23 23:22:57,499 INFO status has been updated to successful
INFO:ecmwf.datastores.legacy_client:status has been updated to successful


44c59883be52820f42429415d130910.nc:   0%|          | 0.00/783k [00:00<?, ?B/s]

slx selesai
titik grid baru: 39 95 | berkas lama: 39 95
tumpang tindih 1-5 Okt: 2510 (harapan 2510)
sp_06       1.455192e-11
sp_00       1.455192e-11
cape_06     6.821210e-13
cape_00     2.273737e-13
t_500_00    5.684342e-14
dtype: float64
tersimpan: (2161110, 52) 2026-10-14 | NaN tertinggi: 0.0


In [ ]:
!pip -q install s3fs
import s3fs
fs = s3fs.S3FileSystem(anon=True)
for tgl in ['2022-12-20', '2023-02-15', '2023-05-15', '2023-07-15', '2025-10-15', '2025-11-10']:
    y, m, d = tgl.split('-')
    for sat in ['himawari8', 'himawari9']:
        try:
            f = fs.find(f'noaa-{sat}/AHI-L2-FLDK-Clouds/{y}/{m}/{d}')
        except Exception:
            f = []
        print(tgl, sat, '| berkas:', len(f), '| CHGT:', sum('CHGT' in p for p in f), '| CPHS:', sum('CPHS' in p for p in f))

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
gcsfs 2025.12.0 requires fsspec==2025.12.0, but you have fsspec 2026.9.0 which is incompatible.
datasets 4.8.5 requires fsspec[http]<=2026.2.0,>=2023.1.0, but you have fsspec 2026.9.0 which is incompatible.
2022-12-20 himawari8 | berkas: 0 | CHGT: 0 | CPHS: 0
2022-12-20 himawari9 | berkas: 0 | CHGT: 0 | CPHS: 0
2023-02-15 himawari8 | berkas: 0 | CHGT: 0 | CPHS: 0
2023-02-15 himawari9 | berkas: 0 | CHGT: 0 | CPHS: 0
2023-05-15 himawari8 | berkas: 0 | CHGT: 0 | CPHS: 0
2023-05-15 himawari9 | berkas: 0 | CHGT: 0 | CPHS: 0
2023-07-15 himawari8 | berkas: 0 | CHGT: 0 | CPHS: 0
2023-07-15 himawari9 | berkas: 0 | CHGT: 0 | CPHS: 0
2025-10-15 himawari8 | berkas: 426 | CHGT: 142 | CPHS: 142
2025-10-15 himawari9 | berkas: 0 | CHGT: 0 | CPHS: 0
2025-11-10 himawari8 | berkas: 423 | CHGT: 141 | CPHS: 141
2025-11-10 himawari9 | 

In [ ]:
!pip -q install s3fs h5netcdf
from google.colab import drive
drive.mount('/content/drive')
import sys, time, shutil, multiprocessing as mp
from pathlib import Path
import pandas as pd
from concurrent.futures import ProcessPoolExecutor

ROOT = Path('/content/drive/MyDrive/Logika Ui')
MASK = str(ROOT / 'prov_mask_nasional.npz')
OUTDIR = ROOT / 'Himawari_nasional'

lokal = Path('/content/himawari_nasional.py')
cadangan = ROOT / 'himawari_nasional.py'
if not lokal.exists():
    shutil.copy(cadangan, lokal)
kode = lokal.read_text()
if 'def proses_hari_multi' not in kode:
    kode += '''

def proses_hari_multi(args):
    sats, tgl, slots = args
    rows, log = [], []
    sisa = list(slots)
    for sat in sats:
        r, l = proses_hari((sat, tgl, tuple(sisa)))
        rows += r
        ok = [x for x in l if x[3] == 'ok']
        log += ok
        sudah = {x[2] for x in ok}
        sisa = [s for s in sisa if s not in sudah]
        if not sisa:
            break
    log += [(sats[-1], tgl, s, 'tidak ada di kedua satelit', 0.0) for s in sisa]
    return rows, log
'''
    lokal.write_text(kode)
shutil.copy(lokal, cadangan)
sys.path.insert(0, '/content')
import himawari_nasional as hn

SLOTS = ("0300", "0400", "0500", "0600", "0700", "0800", "0900")
N_PROSES = 16
hari = list(pd.date_range('2025-10-12', '2025-11-30'))

for bi, b in enumerate(range(0, len(hari), 16)):
    grup = hari[b:b + 16]
    out = OUTDIR / f'himawari_patch_H8_{bi:02d}.csv'
    logf = OUTDIR / f'log_patch_H8_{bi:02d}.csv'
    if logf.exists():
        continue
    tugas = [(('himawari9', 'himawari8'), f'{d:%Y-%m-%d}', SLOTS) for d in grup]
    t0 = time.time()
    with ProcessPoolExecutor(N_PROSES, mp_context=mp.get_context('spawn'),
                             initializer=hn.init, initargs=(MASK,)) as ex:
        hasil = list(ex.map(hn.proses_hari_multi, tugas, chunksize=1))
    rows = [r for rs, _ in hasil for r in rs]
    log = [l for _, ls in hasil for l in ls]
    lg = pd.DataFrame(log, columns=['satelit', 'tanggal', 'waktu', 'status', 'detik'])
    if rows:
        pd.DataFrame(rows).to_csv(out, index=False)
    lg.to_csv(logf, index=False)
    ok = int((lg.status == 'ok').sum())
    print(f'patch batch {bi} ({grup[0]:%d %b} sampai {grup[-1]:%d %b}) selesai {(time.time() - t0) / 60:.1f} menit | '
          f'slot ok {ok}/{len(grup) * len(SLOTS)} | satelit: {lg[lg.status == "ok"].satelit.value_counts().to_dict()}', flush=True)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
patch batch 0 (12 Oct sampai 27 Oct) selesai 10.0 menit | slot ok 103/112 | satelit: {'himawari8': 103}
patch batch 1 (28 Oct sampai 12 Nov) selesai 10.8 menit | slot ok 112/112 | satelit: {'himawari8': 112}
patch batch 2 (13 Nov sampai 28 Nov) selesai 10.2 menit | slot ok 111/112 | satelit: {'himawari8': 92, 'himawari9': 19}
patch batch 3 (29 Nov sampai 30 Nov) selesai 3.9 menit | slot ok 14/14 | satelit: {'himawari9': 14}


In [ ]:
from google.colab import drive
drive.mount('/content/drive')
import os, glob, time, getpass, shutil
from pathlib import Path
import numpy as np, pandas as pd, geopandas as gpd
from shapely.geometry import box

ROOT = "/content/drive/MyDrive/Logika Ui"
BASE = f"{ROOT}/KTI_KARHUTLA"
FIRMS_KEY = getpass.getpass('FIRMS MAP_KEY: ')

adm = gpd.read_file(f"{ROOT}/GADM 2/gadm41_IDN_2.shp").to_crs("EPSG:4326")
w, s, e, n = adm.total_bounds
NX, NY = 3, 2   # harus sama dengan saat unduh FIRMS
xs = np.linspace(w, e, NX + 1)
ys = np.linspace(s, n, NY + 1)
land = adm.union_all()
tiles = [f"{xs[i]:.3f},{ys[j]:.3f},{xs[i+1]:.3f},{ys[j+1]:.3f}"
         for i in range(NX) for j in range(NY)
         if box(xs[i], ys[j], xs[i+1], ys[j+1]).intersects(land)]
print(len(tiles), 'tile')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
FIRMS MAP_KEY: ··········
6 tile


In [ ]:
import numpy as np, pandas as pd
BASE = "/content/drive/MyDrive/Logika Ui/KTI_KARHUTLA"

f = pd.read_csv(f"{BASE}/firms/viirs_2026.csv")
f['tgl'] = pd.to_datetime(f['acq_date'])
if 'type' not in f.columns:
    f['type'] = np.nan
print('kolom:', list(f.columns))
f['bln'] = f['tgl'].dt.to_period('M')
print(f.groupby('bln').agg(baris=('frp', 'size'), hari=('tgl', 'nunique'),
                           type_kosong=('type', lambda s: int(s.isna().sum())),
                           type0=('type', lambda s: int((s == 0).sum()))))
print(f.groupby(f['tgl'] >= '2026-07-01')['confidence'].value_counts())
ada = set(f['tgl'].dt.normalize())
hilang = [d.date() for d in pd.date_range('2026-01-01', f['tgl'].max()) if d not in ada]
print('hari tanpa satu pun titik:', len(hilang), hilang[:20])

try:
    import requests
    r = requests.get(f"https://firms.modaps.eosdis.nasa.gov/api/area/csv/{FIRMS_KEY}/VIIRS_SNPP_NRT/{tiles[0]}/1/2026-10-11", timeout=60)
    print('API NRT:', r.status_code, '|', r.text[:200].replace('\n', ' | '))
    r2 = requests.get(f"https://firms.modaps.eosdis.nasa.gov/mapserver/mapkey_status/?MAP_KEY={FIRMS_KEY}", timeout=60)
    print('status key:', r2.text[:200].replace('\n', ' | '))
except NameError:
    print('FIRMS_KEY/tiles belum ada, lewati tes API')

kolom: ['latitude', 'longitude', 'bright_ti4', 'scan', 'track', 'acq_date', 'acq_time', 'satellite', 'instrument', 'confidence', 'version', 'bright_ti5', 'frp', 'daynight', 'type', 'tgl']
          baris  hari  type_kosong  type0
bln                                      
2026-01    6672    31            0   5239
2026-02    4175    28            0   3178
2026-03    9455    28            0   8051
2026-04    6771    27            0   4519
2026-06    8526    28            0   6132
2026-07   19629    21        19629      0
2026-08  148173    30       148173      0
2026-09  218255    30       218255      0
2026-10   88704    10        88704      0
tgl    confidence
False  n              33585
       l               1386
       h                628
True   n             433111
       l              24808
       h              16842
Name: count, dtype: int64
hari tanpa satu pun titik: 50 [datetime.date(2026, 3, 10), datetime.date(2026, 3, 11), datetime.date(2026, 3, 12), datetime.date(2026, 4, 

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
import os, glob, time, getpass, shutil
from pathlib import Path
import numpy as np, pandas as pd, geopandas as gpd
from shapely.geometry import box

ROOT = "/content/drive/MyDrive/Logika Ui"
BASE = f"{ROOT}/KTI_KARHUTLA"
FIRMS_KEY = getpass.getpass('FIRMS MAP_KEY: ')

adm = gpd.read_file(f"{ROOT}/GADM 2/gadm41_IDN_2.shp").to_crs("EPSG:4326")
w, s, e, n = adm.total_bounds
NX, NY = 3, 2   # harus sama dengan saat unduh FIRMS
xs = np.linspace(w, e, NX + 1)
ys = np.linspace(s, n, NY + 1)
land = adm.union_all()
tiles = [f"{xs[i]:.3f},{ys[j]:.3f},{xs[i+1]:.3f},{ys[j+1]:.3f}"
         for i in range(NX) for j in range(NY)
         if box(xs[i], ys[j], xs[i+1], ys[j+1]).intersects(land)]
print(len(tiles), 'tile')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
FIRMS MAP_KEY: ··········
6 tile


In [ ]:
from google.colab import drive
drive.mount('/content/drive')
!pip -q install cdsapi
import getpass, shutil
from pathlib import Path
import numpy as np, pandas as pd, xarray as xr, geopandas as gpd
import cdsapi

ROOT = Path('/content/drive/MyDrive/Logika Ui')
akar = ROOT / 'KTI_KARHUTLA'
HASIL = akar / 'hasil'

if not (Path.home() / '.cdsapirc').exists():
    token = getpass.getpass('CDS token: ')
    (Path.home() / '.cdsapirc').write_text(f'url: https://cds.climate.copernicus.eu/api\nkey: {token}\n')

# agregasi ke 502 wilayah (interpolasi ke titik pusat, cara yang sudah tervalidasi)
gadm2 = gpd.read_file(ROOT / 'GADM 2' / 'gadm41_IDN_2.shp')
studi = gadm2.reset_index(drop=True)
pusat = studi.to_crs('EPSG:6933').geometry.centroid.to_crs('EPSG:4326')
lat_k = xr.DataArray(pusat.y.values, dims='kab')
lon_k = xr.DataArray(pusat.x.values, dims='kab')

def satu_berkas(path, jam=(0, 6)):
    d = xr.open_dataset(path)
    if 'expver' in d.dims:
        d = d.sel(expver=1).combine_first(d.sel(expver=5))
    d = d.sortby('latitude')
    d = d.sel(valid_time=d.valid_time.dt.hour.isin(jam))
    di = d.interp(latitude=lat_k, longitude=lon_k, method='linear')
    kolom = {}
    for v in di.data_vars:
        a = di[v]
        if 'pressure_level' in a.dims:
            for lv in a.pressure_level.values:
                kolom[f'{v}_{int(lv)}'] = a.sel(pressure_level=lv).transpose('valid_time', 'kab').values
        else:
            kolom[v] = a.transpose('valid_time', 'kab').values
    idx = pd.MultiIndex.from_product([di.valid_time.values, range(len(studi))], names=['valid_time', 'kab'])
    return pd.DataFrame({n: x.reshape(-1) for n, x in kolom.items()}, index=idx)

def pivot(df):
    d = df.reset_index()
    d['tanggal'] = d['valid_time'].dt.normalize()
    d['jam'] = d['valid_time'].dt.hour
    n = [c for c in d.columns if c not in ('valid_time', 'kab', 'tanggal', 'jam')]
    d = d.set_index(['tanggal', 'kab', 'jam'])[n].unstack('jam')
    d.columns = [f'{v}_{j:02d}' for v, j in d.columns]
    d = d.reset_index()
    d['GID_2'] = studi['GID_2'].values[d['kab'].to_numpy()]
    return d.drop(columns='kab')

era = pd.read_csv(HASIL / 'era5_nasional_harian_semua_bulan.csv', parse_dates=['tanggal'])
print('file ERA5 semua bulan:', era.shape, '| tanggal terakhir:', era['tanggal'].max().date())

# unduh ulang Okt 2026 dengan grid yang sama seperti berkas lama
ref = xr.open_dataset(next(akar.rglob('pl_2020_06.nc')))
area = [float(ref.latitude.max()), float(ref.longitude.min()),
        float(ref.latitude.min()), float(ref.longitude.max())]
level = [str(int(x)) for x in ref.pressure_level.values]
res = round(float(np.abs(np.diff(ref.latitude.values)).mean()), 3)
print('area:', area, '| level:', level, '| grid berkas lama:', res, 'derajat')

dasar = {'product_type': ['reanalysis'], 'year': ['2026'], 'month': ['10'],
         'day': [f'{d:02d}' for d in range(1, 15)], 'time': ['00:00', '06:00'],
         'data_format': 'netcdf', 'download_format': 'unarchived',
         'area': area, 'grid': [str(res), str(res)]}
paket = {
    'pl':  ('reanalysis-era5-pressure-levels',
            ['temperature', 'relative_humidity', 'u_component_of_wind', 'v_component_of_wind'], True),
    'plx': ('reanalysis-era5-pressure-levels', ['vertical_velocity', 'specific_humidity'], True),
    'sl':  ('reanalysis-era5-single-levels',
            ['convective_available_potential_energy', '2m_temperature', '2m_dewpoint_temperature'], False),
    'slx': ('reanalysis-era5-single-levels',
            ['total_column_water_vapour', 'total_column_cloud_liquid_water',
             'total_cloud_cover', 'surface_pressure'], False),
}
BARU = Path('/content/era5t_okt2026_grid')
shutil.rmtree(BARU, ignore_errors=True)
BARU.mkdir()
cl = cdsapi.Client()
for j, (dataset, var, pakai_level) in paket.items():
    req = dict(dasar, variable=var)
    if pakai_level:
        req['pressure_level'] = level
    cl.retrieve(dataset, req, str(BARU / f'{j}_2026_10.nc'))
    print(j, 'selesai', flush=True)

d0 = xr.open_dataset(BARU / 'pl_2026_10.nc')
print('titik grid baru:', d0.latitude.size, d0.longitude.size, '| berkas lama:', ref.latitude.size, ref.longitude.size)
assert (d0.latitude.size, d0.longitude.size) == (ref.latitude.size, ref.longitude.size), 'grid masih beda, jangan lanjut'

baru = pivot(pd.concat([satu_berkas(BARU / f'{j}_2026_10.nc') for j in paket], axis=1))
baru['tanggal'] = pd.to_datetime(baru['tanggal'])
kol = [c for c in era.columns if c not in ('tanggal', 'GID_2')]
raw = era[era['tanggal'] >= '2026-10-01']
ov = raw.merge(baru, on=['tanggal', 'GID_2'], suffixes=('_r', '_b'))
sel = pd.Series({c: (ov[c + '_r'] - ov[c + '_b']).abs().max() for c in kol})
print('tumpang tindih 1-5 Okt:', len(ov), '(harapan 2510)')
print(sel.sort_values(ascending=False).head(5))

if sel[['sp_00', 'sp_06']].max() < 100:
    era_final = pd.concat([era[era['tanggal'] < '2026-10-01'], baru[era.columns]])
    era_final = era_final.sort_values('tanggal', kind='stable').reset_index(drop=True)
    era_final.to_csv(HASIL / 'era5_nasional_harian_semua_bulan.csv', index=False)
    print('tersimpan:', era_final.shape, era_final['tanggal'].max().date(),
          '| NaN tertinggi:', era_final.isna().mean().max())
else:
    print('TIDAK disimpan: selisih sp masih besar, kirim hasil print di atas')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
file ERA5 semua bulan: (2161110, 52) | tanggal terakhir: 2026-10-14
area: [7.0, 95.0, -12.0, 142.0] | level: ['850', '700', '500'] | grid berkas lama: 0.5 derajat


2026-10-24 03:10:50,702 INFO [2026-10-06T00:00:00Z] The [CDS Terms of Use](https://cds.climate.copernicus.eu/disclaimer-privacy) will be updated on 28 October 2026. Continued use of Copernicus Products and Services through the CDS after this date constitutes acceptance of the updated terms. If you do not accept the change, please [contact ECMWF Support](https://jira.ecmwf.int/plugins/servlet/desk/portal/1/create/202) before 27 October 2026.
 **[More information about the upcoming change.](https://forum.ecmwf.int/t/upcoming-cds-terms-of-use-change/15374)**
INFO:ecmwf.datastores.legacy_client:[2026-10-06T00:00:00Z] The [CDS Terms of Use](https://cds.climate.copernicus.eu/disclaimer-privacy) will be updated on 28 October 2026. Continued use of Copernicus Products and Services through the CDS after this date constitutes acceptance of the updated terms. If you do not accept the change, please [contact ECMWF Support](https://jira.ecmwf.int/plugins/servlet/desk/portal/1/create/202) before 27 

721e21050b320eb8d2d25437d4ec316e.nc:   0%|          | 0.00/2.51M [00:00<?, ?B/s]

pl selesai


2026-10-24 03:10:53,995 INFO Request ID is 5c95157e-ae05-4ad3-b724-25b88fc1bde2
INFO:ecmwf.datastores.legacy_client:Request ID is 5c95157e-ae05-4ad3-b724-25b88fc1bde2
2026-10-24 03:10:54,145 INFO status has been updated to successful
INFO:ecmwf.datastores.legacy_client:status has been updated to successful


...

843e04b0b1b6204efd88f491dd3cd3fb.nc:   0%|          | 0.00/545k [00:00<?, ?B/s]

sl selesai


2026-10-24 03:10:59,470 INFO Request ID is 16aac879-161c-451a-8584-f062a294ce35
INFO:ecmwf.datastores.legacy_client:Request ID is 16aac879-161c-451a-8584-f062a294ce35
2026-10-24 03:10:59,629 INFO status has been updated to successful
INFO:ecmwf.datastores.legacy_client:status has been updated to successful


44c59883be52820f42429415d130910.nc:   0%|          | 0.00/783k [00:00<?, ?B/s]

slx selesai
titik grid baru: 39 95 | berkas lama: 39 95
tumpang tindih 1-5 Okt: 7028 (harapan 2510)
sp_06       1.455192e-11
sp_00       1.455192e-11
cape_06     9.094947e-13
cape_00     4.547474e-13
t_500_00    5.684342e-14
dtype: float64
tersimpan: (2161110, 52) 2026-10-14 | NaN tertinggi: 0.0


In [ ]:
from pathlib import Path
import numpy as np, pandas as pd, geopandas as gpd

ROOT = Path('/content/drive/MyDrive/Logika Ui')
HASIL = ROOT / 'KTI_KARHUTLA' / 'hasil'
gadm2 = gpd.read_file(ROOT / 'GADM 2' / 'gadm41_IDN_2.shp')

urutan = pd.read_csv(HASIL / 'tabel_gabungan_v1.csv', nrows=0).columns.tolist()
kolom_era = [c for c in urutan if c.endswith(('_00', '_06'))]

# IMERG
im = pd.read_csv(HASIL / 'imerg_nasional_harian.csv', parse_dates=['tanggal'])
im = im.sort_values('run').drop_duplicates(['tanggal', 'GID_2'], keep='first').reset_index(drop=True)
lengkap = pd.date_range(im['tanggal'].min(), im['tanggal'].max())
hilang = sorted(set(lengkap) - set(im['tanggal']))
print('IMERG:', im.shape, im['tanggal'].min().date(), 'sampai', im['tanggal'].max().date(),
      '| hari hilang di tengah:', len(hilang), [d.date() for d in hilang[:10]])

# ERA5
era = pd.read_csv(HASIL / 'era5_nasional_harian_semua_bulan.csv', parse_dates=['tanggal'])

# Himawari: harian per provinsi, logika sama dengan Fuwa sel 9
him = pd.concat([pd.read_csv(p) for p in sorted((ROOT / 'Himawari_nasional').glob('himawari_*.csv'))],
                ignore_index=True)
him['tanggal'] = pd.to_datetime(him['tanggal'])
him = him.drop_duplicates(['tanggal', 'waktu', 'provinsi'])
for k in range(6):
    him[f'f{k}'] = him[f'fase{k}'] / him['n_piksel']
valid = him['n_suhu_valid'].where(him['n_suhu_valid'] > 0)
for a in (273, 253, 233):
    him[f'lt{a}'] = him[f'n_suhu_lt_{a}'] / valid
agg = {
    'satelit': ('satelit', 'first'),
    'him_suhu_mean': ('suhu_mean', 'mean'),
    'him_suhu_p10': ('suhu_p10', 'mean'),
    'him_suhu_min': ('suhu_min', 'min'),
    'him_lt273': ('lt273', 'mean'),
    'him_lt253': ('lt253', 'mean'),
    'him_lt233': ('lt233', 'mean'),
    'him_n': ('suhu_mean', 'size'),
}
for k in range(6):
    agg[f'him_f{k}'] = (f'f{k}', 'mean')
him_harian = him.groupby(['tanggal', 'provinsi']).agg(**agg).reset_index()
him_harian.to_csv(HASIL / 'himawari_nasional_prov_harian.csv', index=False)
print('Himawari:', him_harian.shape, '|', him_harian['provinsi'].nunique(), 'provinsi |',
      him_harian['tanggal'].min().date(), 'sampai', him_harian['tanggal'].max().date())

# FIRMS
fir = pd.read_csv(HASIL / 'firms_nasional_harian.csv', parse_dates=['tanggal'])
print('FIRMS:', fir.shape, fir['tanggal'].min().date(), 'sampai', fir['tanggal'].max().date())

# gabung
nama = gadm2[['GID_2', 'NAME_1', 'NAME_2']]
t = (im.merge(nama, on='GID_2', how='left')
       .merge(era[['tanggal', 'GID_2'] + kolom_era], on=['tanggal', 'GID_2'], how='left')
       .merge(him_harian.rename(columns={'provinsi': 'NAME_1'}), on=['tanggal', 'NAME_1'], how='left')
       .merge(fir, on=['tanggal', 'GID_2'], how='left'))
print('baris IMERG vs tabel (harus sama):', len(im), len(t))

batas = fir['tanggal'].max()
for c in ['titik_panas', 'frp_total', 'frp_maks']:
    t.loc[t['tanggal'] <= batas, c] = t.loc[t['tanggal'] <= batas, c].fillna(0)
t['tahun'] = t['tanggal'].dt.year
t['bulan'] = t['tanggal'].dt.month
t = t[urutan]
t.to_csv(HASIL / 'tabel_gabungan_nasional_semua_bulan.csv', index=False)

kol_him = [c for c in t.columns if c.startswith('him_')]
print(t.shape, '|', t['GID_2'].nunique(), 'wilayah |', t['tanggal'].min().date(), 'sampai', t['tanggal'].max().date())
print('baris tanpa ERA5:', round(t[kolom_era].isna().all(axis=1).mean(), 4),
      '| tanpa Himawari:', round(t[kol_him].isna().all(axis=1).mean(), 4))
print(t.groupby('bulan').size())

# validasi: 38 kab Sumatera harus sama dengan v2 (sebelum 1 Okt 2026)
v2 = pd.read_csv(HASIL / 'tabel_gabungan_v2.csv', parse_dates=['tanggal'])
a = v2[v2['tanggal'] < '2026-10-01'].set_index(['tanggal', 'GID_2']).sort_index()
b = t.set_index(['tanggal', 'GID_2']).sort_index()
ada = a.index.intersection(b.index)
print(len(a), 'baris v2 |', len(ada), 'ada di tabel baru')
beda = []
for c in a.columns:
    x, y = a.loc[ada, c], b.loc[ada, c]
    if x.dtype.kind in 'fi':
        ok = np.allclose(x.to_numpy(float), y.to_numpy(float), equal_nan=True, rtol=1e-6, atol=1e-9)
    else:
        ok = (x.fillna('~').astype(str) == y.fillna('~').astype(str)).all()
    if not ok:
        beda.append(c)
print('kolom yang beda dari v2:', beda, '(harapan [])')

In [ ]:
import io, requests, time, shutil
import numpy as np, pandas as pd, geopandas as gpd
from pathlib import Path

def ketersediaan(src):
    r = requests.get(f"https://firms.modaps.eosdis.nasa.gov/api/data_availability/csv/{FIRMS_KEY}/{src}", timeout=60)
    t = pd.read_csv(io.StringIO(r.text))
    return pd.Timestamp(t['min_date'].iloc[0]), pd.Timestamp(t['max_date'].iloc[0])

sp_min, sp_max = ketersediaan('VIIRS_SNPP_SP')
nrt_min, nrt_max = ketersediaan('VIIRS_SNPP_NRT')
print('SP tersedia:', sp_min.date(), 'sampai', sp_max.date(), '| NRT tersedia:', nrt_min.date(), 'sampai', nrt_max.date())

AKHIR = pd.Timestamp('2026-10-14')
gagal, contoh = [], []

def chunk(src, bb, d, hari=5):
    url = f"https://firms.modaps.eosdis.nasa.gov/api/area/csv/{FIRMS_KEY}/{src}/{bb}/{hari}/{d:%Y-%m-%d}"
    for coba in range(3):
        try:
            r = requests.get(url, timeout=120)
            df = pd.read_csv(io.StringIO(r.text))
            if 'frp' in df.columns:
                return df
            contoh.append(r.text[:150])
        except Exception as e:
            contoh.append(str(e)[:150])
        time.sleep(5 * (coba + 1))
    gagal.append((src, bb, str(d.date())))
    return None

def ambil(src, d0, d1):
    d0, d1 = pd.Timestamp(d0), pd.Timestamp(d1)
    hasil = []
    for d in pd.date_range(d0, d1, freq='5D'):
        hari = int(min(5, (d1 - d).days + 1))
        for bb in tiles:
            df = chunk(src, bb, d, hari)
            if df is not None and not df.empty:
                hasil.append(df)
    return pd.concat(hasil) if hasil else pd.DataFrame()

# SP: tambal Mar dan seluruh Apr (akhir) sampai batas SP; NRT: sisanya sampai 14 Okt
sp = pd.concat([ambil('VIIRS_SNPP_SP', '2026-03-08', '2026-03-14'),
                ambil('VIIRS_SNPP_SP', '2026-04-25', min(sp_max, AKHIR))], ignore_index=True)
nrt = ambil('VIIRS_SNPP_NRT', max(sp_max + pd.Timedelta(days=1), nrt_min), AKHIR)
print('titik SP baru:', len(sp), '| NRT baru:', len(nrt), '| permintaan gagal:', len(gagal))
if gagal:
    print('contoh error server:', contoh[:3])

path = f"{BASE}/firms/viirs_2026.csv"
shutil.copy(path, path + '.bak2')
f0 = pd.read_csv(path)
f0['_nrt'] = f0['version'].astype(str).str.contains('NRT')
sp_tgl = set(sp['acq_date']) if len(sp) else set()
buang = f0['_nrt'] & f0['acq_date'].isin(sp_tgl)
print('baris NRT lama yang diganti SP:', int(buang.sum()))
key = ['latitude', 'longitude', 'acq_date', 'acq_time', 'satellite']
semua = pd.concat([f0.loc[~buang].drop(columns='_nrt'), sp, nrt], ignore_index=True)
semua = semua.drop_duplicates(subset=key, keep='first')
semua.to_csv(path, index=False)
print('file 2026:', len(semua), 'baris | tanggal terakhir:', semua['acq_date'].max())

# potong GADM ulang
df = semua.drop_duplicates()
pts = gpd.GeoDataFrame(df, geometry=gpd.points_from_xy(df.longitude, df.latitude), crs="EPSG:4326")
j = gpd.sjoin(pts, adm[["GID_1", "NAME_1", "GID_2", "NAME_2", "geometry"]], how="inner", predicate="within")
j = j.drop(columns=["geometry", "index_right"])
j.to_csv(f"{BASE}/firms_gadm/viirs_2026.csv", index=False)
print('firms_gadm 2026:', len(df), '->', len(j))

# agregasi nasional: type kosong (NRT) dianggap 0
HASIL = Path(BASE) / 'hasil'
fr = pd.concat([pd.read_csv(p, usecols=['acq_date', 'confidence', 'type', 'GID_2', 'frp', 'version'])
                for p in sorted(Path(BASE, 'firms_gadm').glob('viirs_*.csv'))], ignore_index=True)
fr['tanggal'] = pd.to_datetime(fr['acq_date'])
rendah = fr['confidence'].astype(str).str.lower() == 'l'
dd = fr[(fr['type'].fillna(0) == 0) & ~rendah]
nas = (dd.groupby(['tanggal', 'GID_2'])
         .agg(titik_panas=('frp', 'size'), frp_total=('frp', 'sum'), frp_maks=('frp', 'max'))
         .reset_index())

lama = pd.read_csv(HASIL / 'firms_nasional_harian.csv', parse_dates=['tanggal'])
x = lama[lama['tanggal'] < '2026-03-08'].merge(nas, on=['tanggal', 'GID_2'], how='outer', suffixes=('_l', '_b')).fillna(0)
x = x[x['tanggal'] < '2026-03-08']
print('baris lama vs baru (sebelum 8 Mar) cocok:',
      np.allclose(x['titik_panas_l'], x['titik_panas_b']) and np.allclose(x['frp_total_l'], x['frp_total_b'], atol=1e-3))
nas.to_csv(HASIL / 'firms_nasional_harian.csv', index=False)

fr26 = fr[fr['tanggal'] >= '2026-01-01'].copy()
fr26['bln'] = fr26['tanggal'].dt.to_period('M')
fr26['nrt'] = fr26['version'].astype(str).str.contains('NRT')
print(fr26.groupby('bln').agg(titik=('frp', 'size'), hari=('tanggal', 'nunique'),
                              persen_nrt=('nrt', 'mean'),
                              persen_type0_sp=('type', lambda s: float((s == 0).sum() / max(s.notna().sum(), 1)))).round(3))
ada = set(fr['tanggal'])
hilang = [d.date() for d in pd.date_range('2026-01-01', AKHIR) if d not in ada]
print('hari 2026 tanpa titik:', len(hilang), hilang[:15])

SP tersedia: 2012-01-20 sampai 2026-06-30 | NRT tersedia: 2026-07-01 sampai 2026-10-24
titik SP baru: 15552 | NRT baru: 513194 | permintaan gagal: 0
baris NRT lama yang diganti SP: 0
file 2026: 553905 baris | tanggal terakhir: 2026-10-14
firms_gadm 2026: 553905 -> 534598
baris lama vs baru (sebelum 8 Mar) cocok: True
          titik  hari  persen_nrt  persen_type0_sp
bln                                               
2026-01    5728    31         0.0            0.822
2026-02    3289    28         0.0            0.806
2026-03    7230    28         0.0            0.893
2026-04    4393    27         0.0            0.670
2026-05    4229    31         0.0            0.708
2026-06    7531    28         0.0            0.762
2026-07   21750    25         1.0            0.000
2026-08  145278    30         1.0            0.000
2026-09  214709    30         1.0            0.000
2026-10  120461    14         1.0            0.000
hari 2026 tanpa titik: 15 [datetime.date(2026, 3, 10), datetime.date(

In [ ]:
from pathlib import Path
import numpy as np, pandas as pd, geopandas as gpd

ROOT = Path('/content/drive/MyDrive/Logika Ui')
HASIL = ROOT / 'KTI_KARHUTLA' / 'hasil'
gadm2 = gpd.read_file(ROOT / 'GADM 2' / 'gadm41_IDN_2.shp')

urutan = pd.read_csv(HASIL / 'tabel_gabungan_v1.csv', nrows=0).columns.tolist()
kolom_era = [c for c in urutan if c.endswith(('_00', '_06'))]

# IMERG
im = pd.read_csv(HASIL / 'imerg_nasional_harian.csv', parse_dates=['tanggal'])
im = im.sort_values('run').drop_duplicates(['tanggal', 'GID_2'], keep='first').reset_index(drop=True)
lengkap = pd.date_range(im['tanggal'].min(), im['tanggal'].max())
hilang = sorted(set(lengkap) - set(im['tanggal']))
print('IMERG:', im['tanggal'].min().date(), 'sampai', im['tanggal'].max().date(),
      '| hari hilang:', len(hilang), [d.date() for d in hilang[:5]])

# ERA5
era = pd.read_csv(HASIL / 'era5_nasional_harian_semua_bulan.csv', parse_dates=['tanggal'])

# Himawari: batch utama plus tambalan Himawari-8, lalu harian per provinsi (logika sama dengan Fuwa sel 9)
him = pd.concat([pd.read_csv(p) for p in sorted((ROOT / 'Himawari_nasional').glob('himawari_*.csv'))],
                ignore_index=True)
him['tanggal'] = pd.to_datetime(him['tanggal'])
him = him.drop_duplicates(['tanggal', 'waktu', 'provinsi'], keep='first')
for k in range(6):
    him[f'f{k}'] = him[f'fase{k}'] / him['n_piksel']
valid = him['n_suhu_valid'].where(him['n_suhu_valid'] > 0)
for a_ in (273, 253, 233):
    him[f'lt{a_}'] = him[f'n_suhu_lt_{a_}'] / valid
agg = {
    'satelit': ('satelit', 'first'),
    'him_suhu_mean': ('suhu_mean', 'mean'),
    'him_suhu_p10': ('suhu_p10', 'mean'),
    'him_suhu_min': ('suhu_min', 'min'),
    'him_lt273': ('lt273', 'mean'),
    'him_lt253': ('lt253', 'mean'),
    'him_lt233': ('lt233', 'mean'),
    'him_n': ('suhu_mean', 'size'),
}
for k in range(6):
    agg[f'him_f{k}'] = (f'f{k}', 'mean')
him_harian = him.groupby(['tanggal', 'provinsi']).agg(**agg).reset_index()
him_harian.to_csv(HASIL / 'himawari_nasional_prov_harian.csv', index=False)

# FIRMS
fir = pd.read_csv(HASIL / 'firms_nasional_harian.csv', parse_dates=['tanggal'])

# gabung
nama = gadm2[['GID_2', 'NAME_1', 'NAME_2']]
t = (im.merge(nama, on='GID_2', how='left')
       .merge(era[['tanggal', 'GID_2'] + kolom_era], on=['tanggal', 'GID_2'], how='left')
       .merge(him_harian.rename(columns={'provinsi': 'NAME_1'}), on=['tanggal', 'NAME_1'], how='left')
       .merge(fir, on=['tanggal', 'GID_2'], how='left'))
assert len(t) == len(im), 'baris berubah setelah merge'

# FIRMS: hari tanpa satu pun titik di seluruh Indonesia dibiarkan NaN, bukan 0
batas = fir['tanggal'].max()
hari_ada = set(fir['tanggal'])
tanpa_data = ~t['tanggal'].isin(hari_ada)
for c in ['titik_panas', 'frp_total', 'frp_maks']:
    isi = (t['tanggal'] <= batas) & ~tanpa_data
    t.loc[isi, c] = t.loc[isi, c].fillna(0)
tgl_kosong = sorted(t.loc[tanpa_data, 'tanggal'].dt.date.unique())

t['tahun'] = t['tanggal'].dt.year
t['bulan'] = t['tanggal'].dt.month
t = t[urutan]
t.to_csv(HASIL / 'tabel_gabungan_nasional_semua_bulan.csv', index=False)

kol_him = [c for c in t.columns if c.startswith('him_')]
print(t.shape, '|', t['GID_2'].nunique(), 'wilayah |', t['tanggal'].min().date(), 'sampai', t['tanggal'].max().date())
print('baris tanpa ERA5:', round(t[kolom_era].isna().all(axis=1).mean(), 4),
      '| tanpa Himawari:', round(t[kol_him].isna().all(axis=1).mean(), 4))
print('hari FIRMS tanpa data (NaN):', len(tgl_kosong), tgl_kosong[:25])

# validasi: 38 kab Sumatera lawan v2 (sebelum 1 Okt 2026)
v2 = pd.read_csv(HASIL / 'tabel_gabungan_v2.csv', parse_dates=['tanggal'])
a = v2[v2['tanggal'] < '2026-10-01'].set_index(['tanggal', 'GID_2']).sort_index()
b = t.set_index(['tanggal', 'GID_2']).sort_index()
ada = a.index.intersection(b.index)
print(len(a), 'baris v2 |', len(ada), 'ada di tabel baru')
for c in a.columns:
    x, y = a.loc[ada, c], b.loc[ada, c]
    if x.dtype.kind in 'fi':
        sama = np.isclose(x.to_numpy(float), y.to_numpy(float), equal_nan=True, rtol=1e-6, atol=1e-9)
    else:
        sama = (x.fillna('~').astype(str) == y.fillna('~').astype(str)).to_numpy()
    if not sama.all():
        bln = pd.Series(ada.get_level_values('tanggal')[~sama]).dt.to_period('M')
        print('BEDA', c, '| baris:', int((~sama).sum()), '|', {str(k): v for k, v in bln.value_counts().sort_index().items()})

IMERG: 2020-01-01 sampai 2026-10-14 | hari hilang: 0 []
(1187441, 77) | 479 wilayah | 2020-01-01 sampai 2026-10-14
baris tanpa ERA5: 0.0 | tanpa Himawari: 0.1004
hari FIRMS tanpa data (NaN): 54 [datetime.date(2021, 2, 1), datetime.date(2022, 7, 27), datetime.date(2022, 7, 28), datetime.date(2022, 7, 29), datetime.date(2022, 7, 30), datetime.date(2022, 7, 31), datetime.date(2022, 8, 1), datetime.date(2022, 8, 2), datetime.date(2022, 8, 3), datetime.date(2022, 8, 4), datetime.date(2022, 8, 5), datetime.date(2022, 8, 6), datetime.date(2022, 8, 7), datetime.date(2022, 8, 8), datetime.date(2022, 8, 9), datetime.date(2022, 8, 10), datetime.date(2023, 7, 26), datetime.date(2023, 7, 27), datetime.date(2024, 5, 30), datetime.date(2024, 5, 31), datetime.date(2024, 6, 1), datetime.date(2024, 6, 2), datetime.date(2024, 6, 3), datetime.date(2024, 7, 10), datetime.date(2024, 7, 11)]
39520 baris v2 | 39520 ada di tabel baru
BEDA satelit | baris: 722 | {'2025-10': 722}
BEDA him_suhu_mean | baris: 722 

In [ ]:
import io, requests, time, shutil
import numpy as np, pandas as pd, geopandas as gpd
from pathlib import Path

HASIL = Path(BASE) / 'hasil'
SP_MAX = pd.Timestamp('2026-06-30')
AKHIR = pd.Timestamp('2026-10-14')
fir = pd.read_csv(HASIL / 'firms_nasional_harian.csv', parse_dates=['tanggal'])
ada = set(fir['tanggal'])
kosong = [d for d in pd.date_range('2020-01-01', AKHIR) if d not in ada]
print('hari kosong sebelum perbaikan:', len(kosong))

gagal, contoh = [], []
def chunk(src, bb, d, hari=5):
    url = f"https://firms.modaps.eosdis.nasa.gov/api/area/csv/{FIRMS_KEY}/{src}/{bb}/{hari}/{d:%Y-%m-%d}"
    for coba in range(3):
        try:
            r = requests.get(url, timeout=120)
            df = pd.read_csv(io.StringIO(r.text))
            if 'frp' in df.columns:
                return df
            contoh.append(r.text[:150])
        except Exception as e:
            contoh.append(str(e)[:150])
        time.sleep(5 * (coba + 1))
    gagal.append((src, bb, str(d.date())))
    return None

# kelompokkan hari kosong berurutan jadi rentang, lalu jendela maksimal 5 hari
runs, awal, prev = [], None, None
for d in kosong:
    if awal is None:
        awal = prev = d
    elif (d - prev).days == 1:
        prev = d
    else:
        runs.append((awal, prev))
        awal = prev = d
if awal is not None:
    runs.append((awal, prev))

baru = []
for r0, r1 in runs:
    for d in pd.date_range(r0, r1, freq='5D'):
        hari = int(min(5, (r1 - d).days + 1))
        akhir_w = d + pd.Timedelta(days=hari - 1)
        srcs = []
        if d <= SP_MAX:
            srcs.append('VIIRS_SNPP_SP')
        if akhir_w > SP_MAX:
            srcs.append('VIIRS_SNPP_NRT')
        for src in srcs:
            for bb in tiles:
                df = chunk(src, bb, d, hari)
                if df is not None and not df.empty:
                    baru.append(df)

new = pd.concat(baru, ignore_index=True) if baru else pd.DataFrame()
print('titik baru:', len(new), '| permintaan gagal:', len(gagal))
if gagal:
    print('contoh error server:', contoh[:3])

key = ['latitude', 'longitude', 'acq_date', 'acq_time', 'satellite']
if len(new):
    new['yr'] = pd.to_datetime(new['acq_date']).dt.year
    for yr, g in new.groupby('yr'):
        p = f"{BASE}/firms/viirs_{yr}.csv"
        shutil.copy(p, p + '.bak3')
        semua = pd.concat([pd.read_csv(p), g.drop(columns='yr')], ignore_index=True)
        semua = semua.drop_duplicates(subset=key, keep='first')
        semua.to_csv(p, index=False)
        df = semua.drop_duplicates()
        pts = gpd.GeoDataFrame(df, geometry=gpd.points_from_xy(df.longitude, df.latitude), crs="EPSG:4326")
        j = gpd.sjoin(pts, adm[["GID_1", "NAME_1", "GID_2", "NAME_2", "geometry"]], how="inner", predicate="within")
        j = j.drop(columns=["geometry", "index_right"])
        j.to_csv(f"{BASE}/firms_gadm/viirs_{yr}.csv", index=False)
        print(yr, 'baris', len(semua), '-> gadm', len(j))

# bangun ulang agregasi nasional (type kosong = NRT dianggap 0)
fr = pd.concat([pd.read_csv(p, usecols=['acq_date', 'confidence', 'type', 'GID_2', 'frp'])
                for p in sorted(Path(BASE, 'firms_gadm').glob('viirs_*.csv'))], ignore_index=True)
fr['tanggal'] = pd.to_datetime(fr['acq_date'])
rendah = fr['confidence'].astype(str).str.lower() == 'l'
dd = fr[(fr['type'].fillna(0) == 0) & ~rendah]
nas = (dd.groupby(['tanggal', 'GID_2'])
         .agg(titik_panas=('frp', 'size'), frp_total=('frp', 'sum'), frp_maks=('frp', 'max'))
         .reset_index())
nas.to_csv(HASIL / 'firms_nasional_harian.csv', index=False)

ada2 = set(nas['tanggal'])
masih = [d.date() for d in pd.date_range('2020-01-01', AKHIR) if d not in ada2]
print('hari kosong sesudah perbaikan:', len(masih), 'dari', len(kosong))
print(masih[:30])

hari kosong sebelum perbaikan: 54
titik baru: 23 | permintaan gagal: 0
2021 baris 80498 -> gadm 67916
2024 baris 119277 -> gadm 104011
hari kosong sesudah perbaikan: 54 dari 54
[datetime.date(2021, 2, 1), datetime.date(2022, 7, 27), datetime.date(2022, 7, 28), datetime.date(2022, 7, 29), datetime.date(2022, 7, 30), datetime.date(2022, 7, 31), datetime.date(2022, 8, 1), datetime.date(2022, 8, 2), datetime.date(2022, 8, 3), datetime.date(2022, 8, 4), datetime.date(2022, 8, 5), datetime.date(2022, 8, 6), datetime.date(2022, 8, 7), datetime.date(2022, 8, 8), datetime.date(2022, 8, 9), datetime.date(2022, 8, 10), datetime.date(2023, 7, 26), datetime.date(2023, 7, 27), datetime.date(2024, 5, 30), datetime.date(2024, 5, 31), datetime.date(2024, 6, 1), datetime.date(2024, 6, 2), datetime.date(2024, 6, 3), datetime.date(2024, 7, 10), datetime.date(2024, 7, 11), datetime.date(2024, 7, 12), datetime.date(2024, 7, 13), datetime.date(2024, 7, 14), datetime.date(2024, 7, 15), datetime.date(2024, 7, 

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
from pathlib import Path
import numpy as np, pandas as pd, geopandas as gpd

ROOT = Path('/content/drive/MyDrive/Logika Ui')
HASIL = ROOT / 'KTI_KARHUTLA' / 'hasil'
gadm2 = gpd.read_file(ROOT / 'GADM 2' / 'gadm41_IDN_2.shp')

urutan = pd.read_csv(HASIL / 'tabel_gabungan_v1.csv', nrows=0).columns.tolist()
kolom_era = [c for c in urutan if c.endswith(('_00', '_06'))]

# IMERG
im = pd.read_csv(HASIL / 'imerg_nasional_harian.csv', parse_dates=['tanggal'])
im = im.sort_values('run').drop_duplicates(['tanggal', 'GID_2'], keep='first').reset_index(drop=True)

# ERA5
era = pd.read_csv(HASIL / 'era5_nasional_harian_semua_bulan.csv', parse_dates=['tanggal'])

# Himawari: batch utama plus tambalan Himawari-8, harian per provinsi
him = pd.concat([pd.read_csv(p) for p in sorted((ROOT / 'Himawari_nasional').glob('himawari_*.csv'))],
                ignore_index=True)
him['tanggal'] = pd.to_datetime(him['tanggal'])
him = him.drop_duplicates(['tanggal', 'waktu', 'provinsi'], keep='first')
for k in range(6):
    him[f'f{k}'] = him[f'fase{k}'] / him['n_piksel']
valid = him['n_suhu_valid'].where(him['n_suhu_valid'] > 0)
for a_ in (273, 253, 233):
    him[f'lt{a_}'] = him[f'n_suhu_lt_{a_}'] / valid
agg = {
    'satelit': ('satelit', 'first'),
    'him_suhu_mean': ('suhu_mean', 'mean'),
    'him_suhu_p10': ('suhu_p10', 'mean'),
    'him_suhu_min': ('suhu_min', 'min'),
    'him_lt273': ('lt273', 'mean'),
    'him_lt253': ('lt253', 'mean'),
    'him_lt233': ('lt233', 'mean'),
    'him_n': ('suhu_mean', 'size'),
}
for k in range(6):
    agg[f'him_f{k}'] = (f'f{k}', 'mean')
him_harian = him.groupby(['tanggal', 'provinsi']).agg(**agg).reset_index()

# FIRMS
fir = pd.read_csv(HASIL / 'firms_nasional_harian.csv', parse_dates=['tanggal'])

# gabung
nama = gadm2[['GID_2', 'NAME_1', 'NAME_2']]
t = (im.merge(nama, on='GID_2', how='left')
       .merge(era[['tanggal', 'GID_2'] + kolom_era], on=['tanggal', 'GID_2'], how='left')
       .merge(him_harian.rename(columns={'provinsi': 'NAME_1'}), on=['tanggal', 'NAME_1'], how='left')
       .merge(fir, on=['tanggal', 'GID_2'], how='left'))
assert len(t) == len(im), 'baris berubah setelah merge'

# FIRMS: hari tanpa satu pun titik di seluruh Indonesia dibiarkan NaN, bukan 0
batas = fir['tanggal'].max()
tanpa_data = ~t['tanggal'].isin(set(fir['tanggal']))
for c in ['titik_panas', 'frp_total', 'frp_maks']:
    isi = (t['tanggal'] <= batas) & ~tanpa_data
    t.loc[isi, c] = t.loc[isi, c].fillna(0)

t['tahun'] = t['tanggal'].dt.year
t['bulan'] = t['tanggal'].dt.month
t = t[urutan]
t.to_csv(HASIL / 'tabel_gabungan_nasional_semua_bulan.csv', index=False)

kol_him = [c for c in t.columns if c.startswith('him_')]
print('tersimpan:', t.shape, '|', t['GID_2'].nunique(), 'wilayah |',
      t['tanggal'].min().date(), 'sampai', t['tanggal'].max().date())
print('baris tanpa Himawari:', round(t[kol_him].isna().all(axis=1).mean(), 4),
      '| hari FIRMS tanpa data:', int(t.loc[tanpa_data, 'tanggal'].nunique()))

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
tersimpan: (1187441, 77) | 479 wilayah | 2020-01-01 sampai 2026-10-14
baris tanpa Himawari: 0.1004 | hari FIRMS tanpa data: 54


In [ ]:
from pathlib import Path
import numpy as np, pandas as pd, geopandas as gpd
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.metrics import brier_score_loss, log_loss, roc_auc_score

ROOT = Path('/content/drive/MyDrive/Logika Ui')
HASIL = ROOT / 'KTI_KARHUTLA' / 'hasil'
gadm2 = gpd.read_file(ROOT / 'GADM 2' / 'gadm41_IDN_2.shp')
TAU = 0.05

tabel = pd.read_csv(HASIL / 'tabel_gabungan_nasional_semua_bulan.csv', parse_dates=['tanggal'])
pusat = gadm2.to_crs('EPSG:6933').geometry.centroid.to_crs('EPSG:4326')
koor = pd.DataFrame({'GID_2': gadm2['GID_2'].values, 'lat': pusat.y.values, 'lon': pusat.x.values})
tabel = tabel.merge(koor, on='GID_2', how='left')

hari = tabel['tanggal'].dt.dayofyear
tabel['doy_sin'] = np.sin(2 * np.pi * hari / 365.25)
tabel['doy_cos'] = np.cos(2 * np.pi * hari / 365.25)
curah = tabel['hujan_rata'] * 6
tabel['kelas'] = np.where(curah >= 1, 2, np.where(tabel['frac_hujan'] >= TAU, 1, 0))
print(tabel.shape, '| sebaran kelas:', tabel['kelas'].value_counts(normalize=True).sort_index().round(3).to_dict())

fitur_dasar = ['lat', 'lon', 'doy_sin', 'doy_cos']
fitur_era_00 = [c for c in tabel.columns if c.endswith('_00')]
fitur_him = [c for c in tabel.columns if c.startswith('him_') and c not in ('him_n', 'him_f5')]
set_fitur = {
    'C prakiraan (00 UTC)': fitur_dasar + fitur_era_00,
    'D pemantauan (00 UTC + awan)': fitur_dasar + fitur_era_00 + fitur_him,
}

def model_gb():
    return HistGradientBoostingClassifier(max_depth=4, learning_rate=0.05, max_iter=250,
                                          l2_regularization=1.0, early_stopping=False,
                                          random_state=1)

def klim_kelas(latih, uji):
    peta = (pd.get_dummies(latih['kelas']).astype(float)
            .assign(GID_2=latih['GID_2'].values, bulan=latih['bulan'].values)
            .groupby(['GID_2', 'bulan']).mean().reset_index())
    P = uji[['GID_2', 'bulan']].merge(peta, on=['GID_2', 'bulan'], how='left')[[0, 1, 2]]
    rata = latih['kelas'].value_counts(normalize=True).sort_index()
    return P.fillna(pd.Series(rata.to_numpy(), index=[0, 1, 2])).to_numpy()

n = len(tabel)
kelas = tabel['kelas'].to_numpy()
Y = np.eye(3)[kelas]
tahun = tabel['tahun'].to_numpy()

prob = {'klimatologi': np.zeros((n, 3))}
for th in np.unique(tahun):
    prob['klimatologi'][tahun == th] = klim_kelas(tabel[tahun != th], tabel[tahun == th])
for nama_set, kolom in set_fitur.items():
    prob[nama_set] = np.zeros((n, 3))
    for th in np.unique(tahun):
        m = model_gb()
        m.fit(tabel.loc[tahun != th, kolom], kelas[tahun != th])
        prob[nama_set][tahun == th] = m.predict_proba(tabel.loc[tahun == th, kolom])
        print('selesai', nama_set, 'uji tahun', th, flush=True)

def brier3(P, m):
    P = np.clip(P[m], 1e-6, 1)
    P = P / P.sum(axis=1, keepdims=True)
    return ((P - Y[m]) ** 2).sum(axis=1).mean()

def skill(P, m):
    return 1 - brier3(P, m) / brier3(prob['klimatologi'], m)

kC, kD = 'C prakiraan (00 UTC)', 'D pemantauan (00 UTC + awan)'
ada_awan = tabel['him_n'].notna().to_numpy()
pD = prob[kD].copy()
pD[~ada_awan] = prob[kC][~ada_awan]     # cadangan C kalau awan Himawari kosong
semua = np.ones(n, bool)

print('\nSKILL 3 KELAS terhadap klimatologi, seluruh Indonesia, semua bulan')
print('  C prakiraan:', round(skill(prob[kC], semua), 4), '| D pemantauan (cadangan C):', round(skill(pD, semua), 4))
print('  baris ADA awan:', int(ada_awan.sum()), '| D:', round(skill(prob[kD], ada_awan), 4),
      '| C:', round(skill(prob[kC], ada_awan), 4))

sumatera = tabel['NAME_1'].isin(['Riau', 'Jambi', 'Sumatera Selatan']).to_numpy()
jjo = tabel['bulan'].between(6, 10).to_numpy()
m1 = sumatera & jjo
print('\nSUMATERA 3 PROVINSI, Jun-Okt (pembanding v2 khusus Sumatera: C 0,1685 | D cadangan 0,2412)')
print('  model nasional -> C:', round(skill(prob[kC], m1), 4), '| D (cadangan C):', round(skill(pD, m1), 4))

print('\nSKILL PER BULAN (D dengan cadangan C)')
print({int(b): round(skill(pD, (tabel['bulan'] == b).to_numpy()), 3) for b in range(1, 13)})

print('\nSKILL PER PROVINSI (D dengan cadangan C)')
sp = {p: skill(pD, (tabel['NAME_1'] == p).to_numpy()) for p in sorted(tabel['NAME_1'].dropna().unique())}
print(pd.Series(sp).round(3).sort_values().to_string())

keluar = tabel[['tanggal', 'GID_2', 'NAME_1', 'kelas']].copy()
for k in range(3):
    keluar[f'p{k}_C'] = prob[kC][:, k]
    keluar[f'p{k}_D'] = pD[:, k]
keluar['ada_awan'] = ada_awan.astype(int)
keluar.to_csv(HASIL / 'prediksi_kelas_nasional.csv', index=False)
print('\ntersimpan: prediksi_kelas_nasional.csv', keluar.shape)

/tmp/ipykernel_2045/1010179882.py:11: DtypeWarning: Columns (58) have mixed types. Specify dtype option on import or set low_memory=False.
  tabel = pd.read_csv(HASIL / 'tabel_gabungan_nasional_semua_bulan.csv', parse_dates=['tanggal'])


(1187441, 82) | sebaran kelas: {0: 0.337, 1: 0.26, 2: 0.404}
selesai C prakiraan (00 UTC) uji tahun 2020
selesai C prakiraan (00 UTC) uji tahun 2021
selesai C prakiraan (00 UTC) uji tahun 2022
selesai C prakiraan (00 UTC) uji tahun 2023
selesai C prakiraan (00 UTC) uji tahun 2024
selesai C prakiraan (00 UTC) uji tahun 2025
selesai C prakiraan (00 UTC) uji tahun 2026
selesai D pemantauan (00 UTC + awan) uji tahun 2020
selesai D pemantauan (00 UTC + awan) uji tahun 2021
selesai D pemantauan (00 UTC + awan) uji tahun 2022
selesai D pemantauan (00 UTC + awan) uji tahun 2023
selesai D pemantauan (00 UTC + awan) uji tahun 2024
selesai D pemantauan (00 UTC + awan) uji tahun 2025


KeyboardInterrupt: 

In [ ]:
import time, warnings
warnings.filterwarnings('ignore')
from pathlib import Path
import numpy as np, pandas as pd, geopandas as gpd
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, HistGradientBoostingClassifier
from sklearn.metrics import brier_score_loss, log_loss, roc_auc_score

sebelumnya = globals().get('prob')     # gradient boosting data penuh dari sel model pertama, kalau masih ada

ROOT = Path('/content/drive/MyDrive/Logika Ui')
HASIL = ROOT / 'KTI_KARHUTLA' / 'hasil'
gadm2 = gpd.read_file(ROOT / 'GADM 2' / 'gadm41_IDN_2.shp')
TAU = 0.05
SUB = 300_000          # baris latih per lipatan, SAMA untuk semua model

tabel = pd.read_csv(HASIL / 'tabel_gabungan_nasional_semua_bulan.csv', parse_dates=['tanggal'])
pusat = gadm2.to_crs('EPSG:6933').geometry.centroid.to_crs('EPSG:4326')
koor = pd.DataFrame({'GID_2': gadm2['GID_2'].values, 'lat': pusat.y.values, 'lon': pusat.x.values})
tabel = tabel.merge(koor, on='GID_2', how='left')
hari = tabel['tanggal'].dt.dayofyear
tabel['doy_sin'] = np.sin(2 * np.pi * hari / 365.25)
tabel['doy_cos'] = np.cos(2 * np.pi * hari / 365.25)
curah = tabel['hujan_rata'] * 6
tabel['kelas'] = np.where(curah >= 1, 2, np.where(tabel['frac_hujan'] >= TAU, 1, 0))

fitur_dasar = ['lat', 'lon', 'doy_sin', 'doy_cos']
fitur_era_00 = [c for c in tabel.columns if c.endswith('_00')]
fitur_him = [c for c in tabel.columns if c.startswith('him_') and c not in ('him_n', 'him_f5')]
kC, kD = 'C prakiraan (00 UTC)', 'D pemantauan (00 UTC + awan)'
set_fitur = {kC: fitur_dasar + fitur_era_00, kD: fitur_dasar + fitur_era_00 + fitur_him}

n = len(tabel)
kelas = tabel['kelas'].to_numpy()
Y = np.eye(3)[kelas]
tahun = tabel['tahun'].to_numpy()
tahun_unik = np.unique(tahun)

# baris latih per lipatan: dipilih sekali, dipakai semua model dan kedua set fitur
rng = np.random.default_rng(1)
latih_per_th = {}
for th in tahun_unik:
    idx = np.flatnonzero(tahun != th)
    latih_per_th[th] = rng.choice(idx, SUB, replace=False) if len(idx) > SUB else idx
print(tabel.shape, '| baris latih per lipatan:', {int(k): len(v) for k, v in latih_per_th.items()})

def buat_model():
    return {
        'logistik': make_pipeline(SimpleImputer(strategy='median'), StandardScaler(),
                                  LogisticRegression(C=0.1, max_iter=300)),
        'random forest': make_pipeline(SimpleImputer(strategy='median'),
                                       RandomForestClassifier(n_estimators=100, min_samples_leaf=20,
                                                              max_features='sqrt', n_jobs=-1,
                                                              random_state=1)),
        'gradient boosting': HistGradientBoostingClassifier(max_depth=4, learning_rate=0.05,
                                                            max_iter=250, l2_regularization=1.0,
                                                            early_stopping=False, random_state=1),
    }

def klim_kelas(latih, uji):
    peta = (pd.get_dummies(latih['kelas']).astype(float)
            .assign(GID_2=latih['GID_2'].values, bulan=latih['bulan'].values)
            .groupby(['GID_2', 'bulan']).mean().reset_index())
    P = uji[['GID_2', 'bulan']].merge(peta, on=['GID_2', 'bulan'], how='left')[[0, 1, 2]]
    rata = latih['kelas'].value_counts(normalize=True).sort_index()
    return P.fillna(pd.Series(rata.to_numpy(), index=[0, 1, 2])).to_numpy()

prob = {'klimatologi': np.zeros((n, 3))}
for th in tahun_unik:
    prob['klimatologi'][tahun == th] = klim_kelas(tabel[tahun != th], tabel[tahun == th])   # klimatologi pakai semua baris

for nama_set, kolom in set_fitur.items():
    for nama, model in buat_model().items():
        kunci = f'{nama_set} | {nama}'
        prob[kunci] = np.zeros((n, 3))
        for th in tahun_unik:
            t0 = time.time()
            latih = latih_per_th[th]
            model.fit(tabel.iloc[latih][kolom], kelas[latih])
            uji = tahun == th
            prob[kunci][uji] = model.predict_proba(tabel.loc[uji, kolom])
            print(kunci, '| uji tahun', th, '|', round(time.time() - t0), 'dtk', flush=True)

def brier3(P, m):
    P = np.clip(P[m], 1e-6, 1)
    P = P / P.sum(axis=1, keepdims=True)
    return ((P - Y[m]) ** 2).sum(axis=1).mean()

def skill(P, m):
    return 1 - brier3(P, m) / brier3(prob['klimatologi'], m)

ada_awan = tabel['him_n'].notna().to_numpy()
semua = np.ones(n, bool)
sumatera = (tabel['NAME_1'].isin(['Riau', 'Jambi', 'Sumatera Selatan']) & tabel['bulan'].between(6, 10)).to_numpy()

daftar = []
for nama in ['logistik', 'random forest', 'gradient boosting']:
    PC = prob[f'{kC} | {nama}']
    PD = prob[f'{kD} | {nama}'].copy()
    PD[~ada_awan] = PC[~ada_awan]          # cadangan C kalau awan Himawari kosong
    daftar.append((nama, 'C prakiraan', PC))
    daftar.append((nama, 'D pemantauan (cadangan C)', PD))

if sebelumnya is not None and kC in sebelumnya and kD in sebelumnya and len(sebelumnya[kC]) == n:
    PCf = sebelumnya[kC]
    PDf = sebelumnya[kD].copy()
    PDf[~ada_awan] = PCf[~ada_awan]
    daftar.append(('gradient boosting (data penuh, pembanding)', 'C prakiraan', PCf))
    daftar.append(('gradient boosting (data penuh, pembanding)', 'D pemantauan (cadangan C)', PDf))

baris = []
for nama, lab, P in daftar:
    Pn = np.clip(P, 1e-6, 1)
    Pn = Pn / Pn.sum(axis=1, keepdims=True)
    baris.append({'model': nama, 'set fitur': lab,
                  'skill_semua': skill(P, semua),
                  'skill_Sumatera_JunOkt': skill(P, sumatera),
                  'brier_K1': brier_score_loss(kelas == 1, Pn[:, 1]),
                  'auc_K1': roc_auc_score(kelas == 1, Pn[:, 1]),
                  'log_loss': log_loss(kelas, Pn, labels=[0, 1, 2])})
ringkas = pd.DataFrame(baris).set_index(['model', 'set fitur']).round(4).sort_values('skill_semua', ascending=False)
print('\nPERBANDINGAN MODEL (skill 3 kelas terhadap klimatologi, validasi per tahun, baris latih sama)')
print(ringkas.to_string())
print('\npembanding v2 khusus Sumatera Jun-Okt: C 0,1685 | D cadangan 0,2412')

print('\nSKILL PER TAHUN UJI (konsistensi antar-lipatan)')
tab = {(nama, lab): {int(th): round(skill(P, tahun == th), 3) for th in tahun_unik} for nama, lab, P in daftar}
print(pd.DataFrame(tab).T.to_string())

nama_kunci = list(prob.keys())
np.savez_compressed(HASIL / 'prob_oof_nasional.npz', nama=np.array(nama_kunci),
                    **{f'm{i}': prob[k] for i, k in enumerate(nama_kunci)})
print('\ntersimpan: prob_oof_nasional.npz')

(1187441, 82) | baris latih per lipatan: {2020: 300000, 2021: 300000, 2022: 300000, 2023: 300000, 2024: 300000, 2025: 300000, 2026: 300000}
C prakiraan (00 UTC) | logistik | uji tahun 2020 | 9 dtk
C prakiraan (00 UTC) | logistik | uji tahun 2021 | 12 dtk
C prakiraan (00 UTC) | logistik | uji tahun 2022 | 9 dtk
C prakiraan (00 UTC) | logistik | uji tahun 2023 | 9 dtk
C prakiraan (00 UTC) | logistik | uji tahun 2024 | 11 dtk
C prakiraan (00 UTC) | logistik | uji tahun 2025 | 10 dtk
C prakiraan (00 UTC) | logistik | uji tahun 2026 | 8 dtk
C prakiraan (00 UTC) | random forest | uji tahun 2020 | 190 dtk
C prakiraan (00 UTC) | random forest | uji tahun 2021 | 192 dtk
C prakiraan (00 UTC) | random forest | uji tahun 2022 | 193 dtk
C prakiraan (00 UTC) | random forest | uji tahun 2023 | 189 dtk
C prakiraan (00 UTC) | random forest | uji tahun 2024 | 187 dtk
C prakiraan (00 UTC) | random forest | uji tahun 2025 | 190 dtk
C prakiraan (00 UTC) | random forest | uji tahun 2026 | 186 dtk
C prakiraa